In [6]:
import pandas as pd

DATA_PATH = "/Users/deepanjayapala/Desktop/SPAM_Detector/enron_data_fraud_labeled.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Number of columns:", len(df.columns))
print("\nExact column names:")
for i, col in enumerate(df.columns):
    print(i, repr(col))

Number of columns: 32

Exact column names:
0 'Folder-User'
1 'Folder-Name'
2 'Message-ID'
3 'Date'
4 'From'
5 'To'
6 'Subject'
7 'Mime-Version'
8 'Content-Type'
9 'Content-Transfer-Encoding'
10 'X-From'
11 'X-To'
12 'X-cc'
13 'X-bcc'
14 'X-Folder'
15 'X-Origin'
16 'X-FileName'
17 'Body'
18 'Cc'
19 'Bcc'
20 'Time'
21 'Attendees'
22 'Re'
23 'Source'
24 'Mail-ID'
25 'POI-Present'
26 'Suspicious-Folders'
27 'Sender-Type'
28 'Unique-Mails-From-Sender'
29 'Low-Comm'
30 'Contains-Reply-Forwards'
31 'Label'


In [7]:
# ============================================================
# 3. LABEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("LABEL DISTRIBUTION")
print("=" * 60)

label_counts = df["Label"].value_counts().sort_index()

print(label_counts)

print("\nPercentage:")
print(
    df["Label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(4)
)


LABEL DISTRIBUTION
Label
0    445090
1      2327
Name: count, dtype: int64

Percentage:
Label
0    99.4799
1     0.5201
Name: proportion, dtype: float64


In [8]:
# ============================================================
# 4. DUPLICATE AUDIT
# ============================================================

print("\n" + "=" * 60)
print("DUPLICATE AUDIT")
print("=" * 60)

print(f"Exact duplicate rows: {df.duplicated().sum():,}")

print(
    f"Duplicate Message-ID: "
    f"{df['Message-ID'].duplicated().sum():,}"
)

print(
    f"Unique bodies: "
    f"{df['Body'].nunique(dropna=False):,}"
)

print(
    f"Duplicate body rows: "
    f"{df['Body'].duplicated().sum():,}"
)

print(
    f"Unique subjects: "
    f"{df['Subject'].nunique(dropna=False):,}"
)


DUPLICATE AUDIT
Exact duplicate rows: 0
Duplicate Message-ID: 0
Unique bodies: 242,220
Duplicate body rows: 205,197
Unique subjects: 158,825


In [9]:
# ============================================================
# BODY / LABEL CONSISTENCY CHECK
# ============================================================

body_label_counts = (
    df.groupby("Body")["Label"]
      .nunique()
)

conflicting_bodies = (body_label_counts > 1).sum()

print("=" * 60)
print("BODY / LABEL CONSISTENCY")
print("=" * 60)

print(f"Bodies appearing with both labels: {conflicting_bodies:,}")

BODY / LABEL CONSISTENCY
Bodies appearing with both labels: 3


In [10]:
# ============================================================
# HOW MANY LABELLED EMAILS COME FROM REPEATED BODIES?
# ============================================================

body_frequency = df["Body"].value_counts()

repeated_body_mask = df["Body"].map(body_frequency) > 1

print("\n" + "=" * 60)
print("REPEATED BODY ANALYSIS")
print("=" * 60)

print(
    f"Emails with a repeated body: "
    f"{repeated_body_mask.sum():,}"
)

print(
    f"Emails with a unique body: "
    f"{(~repeated_body_mask).sum():,}"
)

print("\nLabel distribution among repeated bodies:")

print(
    df.loc[repeated_body_mask, "Label"]
      .value_counts()
      .sort_index()
)


REPEATED BODY ANALYSIS
Emails with a repeated body: 319,514
Emails with a unique body: 127,903

Label distribution among repeated bodies:
Label
0    318534
1       980
Name: count, dtype: int64


Subject and Body

In [11]:
MODEL_A_COLUMNS = [
    "Subject",
    "Body",
    "Label"
]

model_a = df[MODEL_A_COLUMNS].copy()

print("Model A shape:", model_a.shape)

print("\nColumns:")
print(model_a.columns.tolist())

print("\nLabel distribution:")
print(model_a["Label"].value_counts())

Model A shape: (447417, 3)

Columns:
['Subject', 'Body', 'Label']

Label distribution:
Label
0    445090
1      2327
Name: count, dtype: int64


In [12]:
# Handle missing text
model_a["Subject"] = model_a["Subject"].fillna("")
model_a["Body"] = model_a["Body"].fillna("")

# Combine Subject + Body
model_a["email_text"] = (
    "SUBJECT: " + model_a["Subject"].astype(str)
    + " BODY: " + model_a["Body"].astype(str)
)

print(model_a[["email_text", "Label"]].head())

                                          email_text  Label
0  SUBJECT: Status BODY: Status John: I'm not rea...      0
1  SUBJECT: re:summer inverses BODY: re:summer in...      0
2  SUBJECT: The WTI Bullet swap contracts BODY: T...      0
3  SUBJECT: Fwd: NYTimes.com Article: Suspended R...      0
4  SUBJECT: daily charts and matrices as hot link...      0


In [13]:
# ============================================================
# STEP 2 — INVESTIGATE CONFLICTING BODY LABELS
# ============================================================

body_label_counts = (
    model_a.groupby("Body")["Label"]
    .nunique()
)

conflicting_bodies = body_label_counts[
    body_label_counts > 1
].index

print("=" * 70)
print("CONFLICTING BODIES")
print("=" * 70)

print(f"Number of conflicting bodies: {len(conflicting_bodies)}")

for i, body in enumerate(conflicting_bodies, start=1):

    print("\n" + "=" * 70)
    print(f"CONFLICTING BODY {i}")
    print("=" * 70)

    subset = model_a[
        model_a["Body"] == body
    ][
        ["Subject", "Body", "Label"]
    ]

    print("\nLabels:")
    print(subset["Label"].value_counts().sort_index())

    print("\nSubjects:")
    for subject in subset["Subject"].unique():
        print(repr(subject))

    print("\nBody:")
    print(str(body)[:2000])

CONFLICTING BODIES
Number of conflicting bodies: 3

CONFLICTING BODY 1

Labels:
Label
0    4
1    1
Name: count, dtype: int64

Subjects:
'<<Concur Expense Document>> - Company Card Notification'

Body:
<<Concur Expense Document>> Company Card Notification Company Card transactions have arrived. To add these transactions to an expense report, click on the following link for Company Card. <URL>

CONFLICTING BODY 2

Labels:
Label
0    9
1    3
Name: count, dtype: int64

Subjects:
'Holiday Party'

Body:
Holiday Party Prebon Energy has sent you an invitation from Sendomatic.com! To view your invitation click the web address below or copy and paste it into your web browser: <URL> If you are having trouble viewing the above web address, visit <URL> and create an account to view the events you have received. Hope you have fun! P.S. If you'd like to send your own personalized invitations or announcements go to <URL> 

CONFLICTING BODY 3

Labels:
Label
0    1
1    3
Name: count, dtype: int64

Su

In [14]:
# ============================================================
# STEP 3 — CREATE MODEL A DATASET
# ============================================================

MODEL_A_COLUMNS = [
    "Subject",
    "Body",
    "Label"
]

model_a = df[MODEL_A_COLUMNS].copy()

# Handle missing text
model_a["Subject"] = model_a["Subject"].fillna("")
model_a["Body"] = model_a["Body"].fillna("")

# Convert to string
model_a["Subject"] = model_a["Subject"].astype(str)
model_a["Body"] = model_a["Body"].astype(str)

# Combine Subject + Body
model_a["email_text"] = (
    "SUBJECT: "
    + model_a["Subject"]
    + " BODY: "
    + model_a["Body"]
)

print("=" * 70)
print("MODEL A DATASET")
print("=" * 70)

print(f"Rows: {len(model_a):,}")
print(f"Columns: {len(model_a.columns)}")

print("\nColumns:")
print(model_a.columns.tolist())

print("\nLabel distribution:")
print(model_a["Label"].value_counts().sort_index())

MODEL A DATASET
Rows: 447,417
Columns: 4

Columns:
['Subject', 'Body', 'Label', 'email_text']

Label distribution:
Label
0    445090
1      2327
Name: count, dtype: int64


In [15]:
# ============================================================
# STEP 4 — INSPECT EMAIL TEXT
# ============================================================

print("=" * 70)
print("NON-FRAUD EXAMPLE")
print("=" * 70)

non_fraud = model_a[
    model_a["Label"] == 0
].sample(2, random_state=42)

for _, row in non_fraud.iterrows():
    print("\nSUBJECT:")
    print(row["Subject"])

    print("\nBODY:")
    print(row["Body"][:1000])

    print("\n" + "-" * 70)


print("\n" + "=" * 70)
print("FRAUD EXAMPLE")
print("=" * 70)

fraud = model_a[
    model_a["Label"] == 1
].sample(5, random_state=42)

for _, row in fraud.iterrows():
    print("\nSUBJECT:")
    print(row["Subject"])

    print("\nBODY:")
    print(row["Body"][:1000])

    print("\n" + "-" * 70)

NON-FRAUD EXAMPLE

SUBJECT:
Re: Mid Year PRC

BODY:
Re: Mid Year PRC Soma, Yes, no problem. Vince Soma Ghosh 05/26/2000 08:43 AM 

----------------------------------------------------------------------

SUBJECT:
RE:

BODY:
what the heck does that mean? don't call me retarded. i'm trying to make you happy. To: Nelson, Michelle 

----------------------------------------------------------------------

FRAUD EXAMPLE

SUBJECT:
Your Ofoto membership

BODY:
Your Ofoto membership Dear larry, Thanks for joining Ofoto! Your account has been credited with 25 FREE 4 x 6" prints. Whether you're into film or digital photography, Ofoto delivers the best prints you've ever seen right to your door. Here are a few of the most popular ways to use Ofoto: Get FREE film processing and high-resolution scanning on your first two rolls Store all of your digital photos on a secure Web site at no charge Invite family and friends to view your online photo albums Order beautiful prints on premium Kodak paper 24 ho

In [ ]:
# ============================================================
# STEP 5 — CREATE BODY-LEVEL GROUP TABLE
# ============================================================

body_groups = (
    model_a.groupby("Body")["Label"]
    .agg(
        label="first",
        n_emails="size"
    )
    .reset_index()
)

print("=" * 70)
print("BODY-LEVEL GROUP ANALYSIS")
print("=" * 70)

print(f"Total email rows       : {len(model_a):,}")
print(f"Unique body groups     : {len(body_groups):,}")

print("\nBody groups by label:")
print(body_groups["label"].value_counts().sort_index())

print("\nEmail rows represented by those groups:")
print(
    body_groups.groupby("label")["n_emails"]
    .sum()
    .sort_index()
)

In [17]:
# ============================================================
# STEP 5 — CREATE BODY-LEVEL GROUP TABLE
# ============================================================

body_groups = (
    model_a.groupby("Body")["Label"]
    .agg(
        label="first",
        n_emails="size"
    )
    .reset_index()
)

print("=" * 70)
print("BODY-LEVEL GROUP ANALYSIS")
print("=" * 70)

print(f"Total email rows       : {len(model_a):,}")
print(f"Unique body groups     : {len(body_groups):,}")

print("\nBody groups by label:")
print(body_groups["label"].value_counts().sort_index())

print("\nEmail rows represented by those groups:")
print(
    body_groups.groupby("label")["n_emails"]
    .sum()
    .sort_index()
)

BODY-LEVEL GROUP ANALYSIS
Total email rows       : 447,417
Unique body groups     : 242,220

Body groups by label:
label
0    240514
1      1706
Name: count, dtype: int64

Email rows represented by those groups:
label
0    445093
1      2324
Name: n_emails, dtype: int64


In [19]:
# ============================================================
# STEP 6 — IDENTIFY CONFLICTING BODY GROUPS
# ============================================================

body_label_counts = (
    model_a.groupby("Body")["Label"]
    .nunique()
)

conflicting_bodies = body_label_counts[
    body_label_counts > 1
].index

print("=" * 70)
print("CONFLICTING BODY GROUPS")
print("=" * 70)

print(f"Conflicting body groups: {len(conflicting_bodies)}")

conflicting_rows = model_a[
    model_a["Body"].isin(conflicting_bodies)
]

print(f"Rows belonging to these groups: {len(conflicting_rows):,}")

print("\nLabel distribution:")
print(conflicting_rows["Label"].value_counts().sort_index())

CONFLICTING BODY GROUPS
Conflicting body groups: 3
Rows belonging to these groups: 21

Label distribution:
Label
0    14
1     7
Name: count, dtype: int64


In [20]:
# ============================================================
# STEP 7 — PREPARE GROUPS FOR TRAIN/TEST SPLIT
# ============================================================

consistent_bodies = body_label_counts[
    body_label_counts == 1
].index

body_groups_clean = body_groups[
    body_groups["Body"].isin(consistent_bodies)
].copy()

print("=" * 70)
print("CLEAN BODY GROUPS")
print("=" * 70)

print(f"Consistent body groups: {len(body_groups_clean):,}")

print("\nGroups by label:")
print(
    body_groups_clean["label"]
    .value_counts()
    .sort_index()
)

CLEAN BODY GROUPS
Consistent body groups: 242,217

Groups by label:
label
0    240512
1      1705
Name: count, dtype: int64


In [30]:
%pip install -U scikit-learn

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 11.1 MB 5.2 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 7.3 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [31]:
# ============================================================
# STEP 8 — STRATIFIED BODY-GROUP TRAIN/TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

train_bodies, test_bodies = train_test_split(
    body_groups_clean,
    test_size=0.20,
    random_state=42,
    stratify=body_groups_clean["label"]
)

print("=" * 70)
print("TRAIN / TEST BODY-GROUP SPLIT")
print("=" * 70)

print(f"Training body groups: {len(train_bodies):,}")
print(f"Testing body groups : {len(test_bodies):,}")

print("\nTraining group labels:")
print(train_bodies["label"].value_counts().sort_index())

print("\nTesting group labels:")
print(test_bodies["label"].value_counts().sort_index())

TRAIN / TEST BODY-GROUP SPLIT
Training body groups: 193,773
Testing body groups : 48,444

Training group labels:
label
0    192409
1      1364
Name: count, dtype: int64

Testing group labels:
label
0    48103
1      341
Name: count, dtype: int64


In [32]:
# ============================================================
# STEP 9 — CREATE TRAIN / TEST EMAIL DATASETS
# ============================================================

train_body_values = set(train_bodies["Body"])
test_body_values = set(test_bodies["Body"])

# Consistent groups
train_df = model_a[
    model_a["Body"].isin(train_body_values)
].copy()

test_df = model_a[
    model_a["Body"].isin(test_body_values)
].copy()

# Put the 3 conflicting groups entirely into training
conflicting_train = model_a[
    model_a["Body"].isin(conflicting_bodies)
].copy()

train_df = pd.concat(
    [train_df, conflicting_train],
    ignore_index=True
)

print("=" * 70)
print("EMAIL-LEVEL TRAIN / TEST DATA")
print("=" * 70)

print(f"Training emails: {len(train_df):,}")
print(f"Testing emails : {len(test_df):,}")

print("\nTraining labels:")
print(train_df["Label"].value_counts().sort_index())

print("\nTesting labels:")
print(test_df["Label"].value_counts().sort_index())

EMAIL-LEVEL TRAIN / TEST DATA
Training emails: 358,558
Testing emails : 88,859

Training labels:
Label
0    356681
1      1877
Name: count, dtype: int64

Testing labels:
Label
0    88409
1      450
Name: count, dtype: int64


In [33]:
# ============================================================
# STEP 10 — VERIFY NO BODY LEAKAGE
# ============================================================

train_body_set = set(train_df["Body"])
test_body_set = set(test_df["Body"])

overlap = train_body_set.intersection(test_body_set)

print("=" * 70)
print("LEAKAGE CHECK")
print("=" * 70)

print(f"Unique train bodies: {len(train_body_set):,}")
print(f"Unique test bodies : {len(test_body_set):,}")
print(f"Overlapping bodies : {len(overlap):,}")

LEAKAGE CHECK
Unique train bodies: 193,776
Unique test bodies : 48,444
Overlapping bodies : 0


In [34]:
# ============================================================
# STEP 10 — VERIFY NO BODY LEAKAGE
# ============================================================

train_body_set = set(train_df["Body"])
test_body_set = set(test_df["Body"])

overlap = train_body_set.intersection(test_body_set)

print("=" * 70)
print("LEAKAGE CHECK")
print("=" * 70)

print(f"Unique train bodies: {len(train_body_set):,}")
print(f"Unique test bodies : {len(test_body_set):,}")
print(f"Overlapping bodies : {len(overlap):,}")

LEAKAGE CHECK
Unique train bodies: 193,776
Unique test bodies : 48,444
Overlapping bodies : 0


In [35]:
# ============================================================
# STEP 11 — PREPARE FEATURES AND TARGET
# ============================================================

X_train = (
    "SUBJECT: "
    + train_df["Subject"].fillna("").astype(str)
    + " BODY: "
    + train_df["Body"].fillna("").astype(str)
)

X_test = (
    "SUBJECT: "
    + test_df["Subject"].fillna("").astype(str)
    + " BODY: "
    + test_df["Body"].fillna("").astype(str)
)

y_train = train_df["Label"].astype(int)
y_test = test_df["Label"].astype(int)

print("=" * 70)
print("MODEL INPUT")
print("=" * 70)

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")

print(f"\ny_train:")
print(y_train.value_counts().sort_index())

print(f"\ny_test:")
print(y_test.value_counts().sort_index())

MODEL INPUT
X_train: (358558,)
X_test : (88859,)

y_train:
Label
0    356681
1      1877
Name: count, dtype: int64

y_test:
Label
0    88409
1      450
Name: count, dtype: int64


In [36]:
# ============================================================
# STEP 12 — TF-IDF VECTORIZATION
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    max_features=150_000,
    sublinear_tf=True,
    dtype="float32"
)

print("Fitting TF-IDF on TRAINING data only...")

X_train_tfidf = tfidf.fit_transform(X_train)

print("Transforming TEST data...")

X_test_tfidf = tfidf.transform(X_test)

print("\n" + "=" * 70)
print("TF-IDF COMPLETE")
print("=" * 70)

print(f"Training matrix: {X_train_tfidf.shape}")
print(f"Testing matrix : {X_test_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_):,}")

Fitting TF-IDF on TRAINING data only...


/Users/deepanjayapala/Library/Python/3.9/lib/python/site-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


Transforming TEST data...

TF-IDF COMPLETE
Training matrix: (358558, 150000)
Testing matrix : (88859, 150000)
Vocabulary size: 150,000


In [37]:
class_weight="balanced"

In [38]:
# ============================================================
# STEP 13 — LOGISTIC REGRESSION
# ============================================================

from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    solver="liblinear",
    random_state=42
)

print("Training Logistic Regression...")

log_reg.fit(
    X_train_tfidf,
    y_train
)

print("✓ Logistic Regression trained")

Training Logistic Regression...
✓ Logistic Regression trained


In [39]:
# ============================================================
# STEP 14 — LOGISTIC REGRESSION EVALUATION
# ============================================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

y_pred = log_reg.predict(X_test_tfidf)
y_prob = log_reg.predict_proba(X_test_tfidf)[:, 1]

print("=" * 70)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 70)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nKey Metrics:")

print(
    f"Precision : "
    f"{precision_score(y_test, y_pred):.4f}"
)

print(
    f"Recall    : "
    f"{recall_score(y_test, y_pred):.4f}"
)

print(
    f"F1        : "
    f"{f1_score(y_test, y_pred):.4f}"
)

print(
    f"ROC-AUC   : "
    f"{roc_auc_score(y_test, y_prob):.4f}"
)

print(
    f"PR-AUC    : "
    f"{average_precision_score(y_test, y_prob):.4f}"
)

LOGISTIC REGRESSION RESULTS

Classification Report:
              precision    recall  f1-score   support

           0     0.9991    0.9912    0.9951     88409
           1     0.3231    0.8222    0.4639       450

    accuracy                         0.9904     88859
   macro avg     0.6611    0.9067    0.7295     88859
weighted avg     0.9957    0.9904    0.9925     88859

Confusion Matrix:
[[87634   775]
 [   80   370]]

Key Metrics:
Precision : 0.3231
Recall    : 0.8222
F1        : 0.4639
ROC-AUC   : 0.9924
PR-AUC    : 0.6181


In [41]:
# ============================================================
# STEP 15 — LINEAR SVM
# ============================================================

from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    class_weight="balanced",
    C=1.0,
    max_iter=2000,
    random_state=42
)

print("Training Linear SVM...")

svm_model.fit(
    X_train_tfidf,
    y_train
)

print("✓ Linear SVM trained")

Training Linear SVM...
✓ Linear SVM trained


In [42]:
# ============================================================
# STEP 16 — LINEAR SVM EVALUATION
# ============================================================

y_svm_pred = svm_model.predict(X_test_tfidf)

y_svm_score = svm_model.decision_function(
    X_test_tfidf
)

print("=" * 70)
print("LINEAR SVM RESULTS")
print("=" * 70)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_svm_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_svm_pred
    )
)

print("\nKey Metrics:")

print(
    f"Precision : "
    f"{precision_score(y_test, y_svm_pred):.4f}"
)

print(
    f"Recall    : "
    f"{recall_score(y_test, y_svm_pred):.4f}"
)

print(
    f"F1        : "
    f"{f1_score(y_test, y_svm_pred):.4f}"
)

print(
    f"ROC-AUC   : "
    f"{roc_auc_score(y_test, y_svm_score):.4f}"
)

print(
    f"PR-AUC    : "
    f"{average_precision_score(y_test, y_svm_score):.4f}"
)

LINEAR SVM RESULTS

Classification Report:
              precision    recall  f1-score   support

           0     0.9980    0.9984    0.9982     88409
           1     0.6627    0.6156    0.6382       450

    accuracy                         0.9965     88859
   macro avg     0.8304    0.8070    0.8182     88859
weighted avg     0.9963    0.9965    0.9964     88859


Confusion Matrix:
[[88268   141]
 [  173   277]]

Key Metrics:
Precision : 0.6627
Recall    : 0.6156
F1        : 0.6382
ROC-AUC   : 0.9899
PR-AUC    : 0.6819


In [43]:
# ============================================================
# STEP 17 — MODEL COMPARISON
# ============================================================

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM"
    ],

    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_svm_pred)
    ],

    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_svm_pred)
    ],

    "F1": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_svm_pred)
    ],

    "ROC_AUC": [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test, y_svm_score)
    ],

    "PR_AUC": [
        average_precision_score(y_test, y_prob),
        average_precision_score(y_test, y_svm_score)
    ]
})

print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(
    results.round(4).to_string(index=False)
)

MODEL COMPARISON
              Model  Precision  Recall     F1  ROC_AUC  PR_AUC
Logistic Regression     0.3231  0.8222 0.4639   0.9924  0.6181
         Linear SVM     0.6627  0.6156 0.6382   0.9899  0.6819


In [44]:
# ============================================================
# STEP 18 — LOGISTIC REGRESSION FEATURE IMPORTANCE
# ============================================================

import numpy as np

feature_names = np.array(
    tfidf.get_feature_names_out()
)

coefficients = log_reg.coef_[0]

top_fraud_indices = np.argsort(
    coefficients
)[-30:][::-1]

top_nonfraud_indices = np.argsort(
    coefficients
)[:30]

print("=" * 70)
print("TOP FRAUD-ASSOCIATED FEATURES")
print("=" * 70)

for idx in top_fraud_indices:
    print(
        f"{feature_names[idx]:40s} "
        f"{coefficients[idx]:.4f}"
    )

print("\n" + "=" * 70)
print("TOP NON-FRAUD-ASSOCIATED FEATURES")
print("=" * 70)

for idx in top_nonfraud_indices:
    print(
        f"{feature_names[idx]:40s} "
        f"{coefficients[idx]:.4f}"
    )

TOP FRAUD-ASSOCIATED FEATURES
your                                     11.2538
com                                      10.1254
dear                                     8.8480
click                                    8.6653
free                                     8.4007
you                                      7.6875
remove                                   7.3020
money                                    6.7177
image                                    6.0869
our                                      5.9203
internet                                 5.7127
mailings                                 5.6321
online                                   5.3209
account                                  5.2068
offer                                    5.0256
card                                     4.8572
equilon                                  4.7530
800                                      4.6487
commerce solutions                       4.6478
en                                       4.6464
order   

In [ ]:
# ============================================================
# STEP 19 — SAVE MODEL
# ============================================================

import joblib
from pathlib import Path

MODEL_DIR = Path(
    "/Users/deepanjayapala/Desktop/SPAM_Detector/models"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    tfidf,
    MODEL_DIR / "tfidf_vectorizer.joblib"
)

joblib.dump(
    log_reg,
    MODEL_DIR / "logistic_regression.joblib"
)

joblib.dump(
    svm_model,
    MODEL_DIR / "linear_svm.joblib"
)

print("✓ Models saved to:")
print(MODEL_DIR)

In [45]:
# ============================================================
# STEP 18 — THRESHOLD ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(
    0.05,
    0.96,
    0.05
)

threshold_results = []

for threshold in thresholds:

    y_threshold = (
        y_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test,
            y_threshold,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            y_threshold,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            y_threshold,
            zero_division=0
        )
    })

threshold_results = pd.DataFrame(
    threshold_results
)

print(
    threshold_results.round(4).to_string(
        index=False
    )
)

 Threshold  Precision  Recall     F1
      0.05     0.0725  0.9800 0.1350
      0.10     0.1171  0.9444 0.2084
      0.15     0.1511  0.9222 0.2597
      0.20     0.1762  0.8956 0.2945
      0.25     0.2069  0.8844 0.3353
      0.30     0.2270  0.8711 0.3601
      0.35     0.2492  0.8533 0.3857
      0.40     0.2766  0.8444 0.4167
      0.45     0.2982  0.8311 0.4390
      0.50     0.3231  0.8222 0.4639
      0.55     0.3425  0.8067 0.4808
      0.60     0.3664  0.7889 0.5004
      0.65     0.3858  0.7511 0.5098
      0.70     0.4082  0.7311 0.5239
      0.75     0.4308  0.6844 0.5288
      0.80     0.4717  0.6667 0.5525
      0.85     0.5330  0.6467 0.5843
      0.90     0.6071  0.6044 0.6058
      0.95     0.6958  0.5133 0.5908


In [46]:
# ============================================================
# STEP 19 — BEST F1 THRESHOLD
# ============================================================

best_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

print("=" * 70)
print("BEST THRESHOLD BY F1")
print("=" * 70)

print(
    f"Threshold : {best_row['Threshold']:.2f}"
)

print(
    f"Precision : {best_row['Precision']:.4f}"
)

print(
    f"Recall    : {best_row['Recall']:.4f}"
)

print(
    f"F1        : {best_row['F1']:.4f}"
)

BEST THRESHOLD BY F1
Threshold : 0.90
Precision : 0.6071
Recall    : 0.6044
F1        : 0.6058


In [47]:
# ============================================================
# STEP 20 — PRECISION TARGET ANALYSIS
# ============================================================

target_precisions = [
    0.50,
    0.60,
    0.70,
    0.80
]

print("=" * 70)
print("PRECISION TARGET ANALYSIS")
print("=" * 70)

for target in target_precisions:

    candidates = threshold_results[
        threshold_results["Precision"] >= target
    ]

    if len(candidates) == 0:

        print(
            f"\nNo tested threshold achieved "
            f"{target:.0%} precision."
        )

    else:

        best = candidates.loc[
            candidates["F1"].idxmax()
        ]

        print(
            f"\nTarget precision: {target:.0%}"
        )

        print(
            f"Threshold: {best['Threshold']:.2f}"
        )

        print(
            f"Precision: {best['Precision']:.4f}"
        )

        print(
            f"Recall:    {best['Recall']:.4f}"
        )

        print(
            f"F1:        {best['F1']:.4f}"
        )

PRECISION TARGET ANALYSIS

Target precision: 50%
Threshold: 0.90
Precision: 0.6071
Recall:    0.6044
F1:        0.6058

Target precision: 60%
Threshold: 0.90
Precision: 0.6071
Recall:    0.6044
F1:        0.6058

No tested threshold achieved 70% precision.

No tested threshold achieved 80% precision.


In [48]:
# ============================================================
# STEP 21 — SVM THRESHOLD ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds_svm = np.arange(
    -2.0,
    2.01,
    0.1
)

svm_threshold_results = []

for threshold in thresholds_svm:

    y_threshold = (
        y_svm_score >= threshold
    ).astype(int)

    svm_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test,
            y_threshold,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            y_threshold,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            y_threshold,
            zero_division=0
        )
    })

svm_threshold_results = pd.DataFrame(
    svm_threshold_results
)

print(
    svm_threshold_results.round(4)
    .to_string(index=False)
)

 Threshold  Precision  Recall     F1
      -2.0     0.0074  1.0000 0.0147
      -1.9     0.0086  1.0000 0.0170
      -1.8     0.0102  1.0000 0.0203
      -1.7     0.0127  1.0000 0.0250
      -1.6     0.0160  0.9978 0.0316
      -1.5     0.0209  0.9956 0.0410
      -1.4     0.0279  0.9933 0.0543
      -1.3     0.0375  0.9844 0.0722
      -1.2     0.0519  0.9733 0.0985
      -1.1     0.0736  0.9467 0.1366
      -1.0     0.1047  0.9289 0.1881
      -0.9     0.1479  0.9178 0.2548
      -0.8     0.2011  0.8844 0.3277
      -0.7     0.2554  0.8467 0.3924
      -0.6     0.3129  0.8378 0.4556
      -0.5     0.3618  0.7822 0.4947
      -0.4     0.4449  0.7444 0.5569
      -0.3     0.5071  0.7111 0.5920
      -0.2     0.5634  0.6711 0.6126
      -0.1     0.6047  0.6356 0.6197
       0.0     0.6627  0.6156 0.6382
       0.1     0.6849  0.5844 0.6307
       0.2     0.7318  0.5578 0.6330
       0.3     0.7672  0.5200 0.6199
       0.4     0.8029  0.4889 0.6077
       0.5     0.8277  0.4378 0.5727
 

In [49]:
# ============================================================
# STEP 22 — BEST SVM THRESHOLD BY F1
# ============================================================

best_svm_threshold = svm_threshold_results.loc[
    svm_threshold_results["F1"].idxmax()
]

print("=" * 70)
print("BEST SVM THRESHOLD")
print("=" * 70)

print(
    f"Threshold : {best_svm_threshold['Threshold']:.2f}"
)

print(
    f"Precision : {best_svm_threshold['Precision']:.4f}"
)

print(
    f"Recall    : {best_svm_threshold['Recall']:.4f}"
)

print(
    f"F1        : {best_svm_threshold['F1']:.4f}"
)

BEST SVM THRESHOLD
Threshold : 0.00
Precision : 0.6627
Recall    : 0.6156
F1        : 0.6382


In [50]:
# ============================================================
# STEP 23 — SVM PRECISION TARGETS
# ============================================================

targets = [0.70, 0.80, 0.90, 0.95]

print("=" * 70)
print("SVM PRECISION TARGET ANALYSIS")
print("=" * 70)

for target in targets:

    candidates = svm_threshold_results[
        svm_threshold_results["Precision"] >= target
    ]

    if len(candidates) == 0:

        print(
            f"\nNo tested threshold achieved "
            f"{target:.0%} precision."
        )

    else:

        best = candidates.loc[
            candidates["Recall"].idxmax()
        ]

        print(
            f"\nTarget precision: {target:.0%}"
        )

        print(
            f"Threshold: {best['Threshold']:.2f}"
        )

        print(
            f"Precision: {best['Precision']:.4f}"
        )

        print(
            f"Recall:    {best['Recall']:.4f}"
        )

        print(
            f"F1:        {best['F1']:.4f}"
        )

SVM PRECISION TARGET ANALYSIS

Target precision: 70%
Threshold: 0.20
Precision: 0.7318
Recall:    0.5578
F1:        0.6330

Target precision: 80%
Threshold: 0.40
Precision: 0.8029
Recall:    0.4889
F1:        0.6077

Target precision: 90%
Threshold: 0.70
Precision: 0.9326
Recall:    0.3689
F1:        0.5287

Target precision: 95%
Threshold: 0.90
Precision: 0.9615
Recall:    0.2778
F1:        0.4310


In [52]:
# ============================================================
# STEP 24 — FINAL BASELINE COMPARISON
# ============================================================

comparison = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "Linear SVM"
    ],

    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_svm_pred)
    ],

    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_svm_pred)
    ],

    "F1": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_svm_pred)
    ],

    "ROC-AUC": [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test, y_svm_score)
    ],

    "PR-AUC": [
        average_precision_score(y_test, y_prob),
        average_precision_score(y_test, y_svm_score)
    ]
})

print(
    comparison.round(4)
    .to_string(index=False)
)

              Model  Precision  Recall     F1  ROC-AUC  PR-AUC
Logistic Regression     0.3231  0.8222 0.4639   0.9924  0.6181
         Linear SVM     0.6627  0.6156 0.6382   0.9899  0.6819


Error Analysis

In [53]:
# ============================================================
# STEP 25 — SVM ERROR ANALYSIS
# ============================================================

# Use the current SVM threshold of 0.0
svm_threshold = 0.0

y_error_pred = (
    y_svm_score >= svm_threshold
).astype(int)

error_analysis = test_df.copy()

error_analysis["actual"] = y_test.values
error_analysis["predicted"] = y_error_pred
error_analysis["score"] = y_svm_score

# ------------------------------------------------------------
# Categorize predictions
# ------------------------------------------------------------

error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual"] == 1) &
        (error_analysis["predicted"] == 1),

        (error_analysis["actual"] == 0) &
        (error_analysis["predicted"] == 0),

        (error_analysis["actual"] == 0) &
        (error_analysis["predicted"] == 1),

        (error_analysis["actual"] == 1) &
        (error_analysis["predicted"] == 0)
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)

print("=" * 70)
print("ERROR ANALYSIS SUMMARY")
print("=" * 70)

print(
    error_analysis["error_type"]
    .value_counts()
)

ERROR ANALYSIS SUMMARY
error_type
True Negative     88268
True Positive       277
False Negative      173
False Positive      141
Name: count, dtype: int64


In [54]:
# ============================================================
# STEP 26 — FALSE POSITIVES
# ============================================================

false_positives = error_analysis[
    error_analysis["error_type"] == "False Positive"
].copy()

false_positives = false_positives.sort_values(
    "score",
    ascending=False
)

print("=" * 70)
print("FALSE POSITIVES")
print("=" * 70)

print(f"Total false positives: {len(false_positives):,}")

for i, (_, row) in enumerate(
    false_positives.head(10).iterrows(),
    start=1
):

    print("\n" + "-" * 70)
    print(f"FALSE POSITIVE {i}")
    print("-" * 70)

    print("SVM SCORE:", round(row["score"], 4))

    print("\nSUBJECT:")
    print(row["Subject"])

    print("\nBODY:")
    print(str(row["Body"])[:1500])

FALSE POSITIVES
Total false positives: 141

----------------------------------------------------------------------
FALSE POSITIVE 1
----------------------------------------------------------------------
SVM SCORE: 1.0596

SUBJECT:
Hawkes Peers & Co.

BODY:
Please respond to Hello Jeff, I hope you are well. Below is a background on our firm. As you will see what characterizes our candidates is their current or recent experiences. Many have already rolled out into industry as well. Hawkes-Peers Co. is an executive search firm based in NYC, specializing in identifying opportunities for top-tier consultants from the top strategy firms-McKinsey, Bain, BCG, AT Kearney, and Booz Allen, who are looking to move into the corporate sector. These candidates also share the common qualification of an MBA from one of the elite business schools such as Harvard, Wharton, Sloan, Kellogg, etc. Our focus targets this network of individuals on an ongoing basis, keeping an active portfolio of these manageme

In [55]:
# ============================================================
# STEP 27 — FALSE NEGATIVES
# ============================================================

false_negatives = error_analysis[
    error_analysis["error_type"] == "False Negative"
].copy()

false_negatives = false_negatives.sort_values(
    "score",
    ascending=True
)

print("=" * 70)
print("FALSE NEGATIVES")
print("=" * 70)

print(f"Total false negatives: {len(false_negatives):,}")

for i, (_, row) in enumerate(
    false_negatives.head(10).iterrows(),
    start=1
):

    print("\n" + "-" * 70)
    print(f"FALSE NEGATIVE {i}")
    print("-" * 70)

    print("SVM SCORE:", round(row["score"], 4))

    print("\nSUBJECT:")
    print(row["Subject"])

    print("\nBODY:")
    print(str(row["Body"])[:1500])

FALSE NEGATIVES
Total false negatives: 173

----------------------------------------------------------------------
FALSE NEGATIVE 1
----------------------------------------------------------------------
SVM SCORE: -1.6034

SUBJECT:
RE: Golf Friday 7:48

BODY:
Great Your in. See Ya Friday To: jbyrd@byrdinterior.com 

----------------------------------------------------------------------
FALSE NEGATIVE 2
----------------------------------------------------------------------
SVM SCORE: -1.5292

SUBJECT:
ENE

BODY:
ENE WHERE'S THE BOTTOM???? I'm trying to figure out when to buy in. Brian Day Senior Marketing Representative AEC Marketing (USA) Inc. 950 17th Street, Suite 2600 Denver, CO 80202 (303)389-5006 (720)956-3572FAX 

----------------------------------------------------------------------
FALSE NEGATIVE 3
----------------------------------------------------------------------
SVM SCORE: -1.4041

SUBJECT:
Big Profits From Market Roller-Coaster

BODY:
Big Profits From Market Roller-Coast

In [56]:
# ============================================================
# STEP 28 — ERROR TYPE COUNTS
# ============================================================

error_summary = (
    error_analysis["error_type"]
    .value_counts()
    .rename_axis("Prediction_Type")
    .reset_index(name="Count")
)

print(
    error_summary.to_string(index=False)
)

Prediction_Type  Count
  True Negative  88268
  True Positive    277
 False Negative    173
 False Positive    141


ENSEMBLE

In [57]:
# ============================================================
# STEP 29 — LOGISTIC + LINEAR SVM ENSEMBLE
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# 1. NORMALIZE SVM DECISION SCORES
# ------------------------------------------------------------
# Logistic Regression already gives probabilities.
# SVM gives decision scores on a different scale.
#
# We convert both to 0-1 using min-max normalization
# on the TEST SET for this exploratory comparison.

svm_min = y_svm_score.min()
svm_max = y_svm_score.max()

svm_norm = (
    (y_svm_score - svm_min) /
    (svm_max - svm_min)
)

# Logistic regression probability
lr_prob = y_prob

# ------------------------------------------------------------
# 2. CREATE ENSEMBLE SCORES
# ------------------------------------------------------------

ensemble_50_50 = (
    0.50 * lr_prob +
    0.50 * svm_norm
)

ensemble_70_svm = (
    0.30 * lr_prob +
    0.70 * svm_norm
)

ensemble_70_lr = (
    0.70 * lr_prob +
    0.30 * svm_norm
)

# ------------------------------------------------------------
# 3. EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_ensemble(name, scores, threshold=0.50):

    predictions = (
        scores >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        scores
    )

    pr_auc = average_precision_score(
        y_test,
        scores
    )

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1        : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")
    print(f"PR-AUC    : {pr_auc:.4f}")

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_test,
            predictions
        )
    )

    return {
        "Model": name,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    }


# ------------------------------------------------------------
# 4. TEST ENSEMBLES
# ------------------------------------------------------------

results = []

results.append(
    evaluate_ensemble(
        "50% Logistic + 50% SVM",
        ensemble_50_50
    )
)

results.append(
    evaluate_ensemble(
        "30% Logistic + 70% SVM",
        ensemble_70_svm
    )
)

results.append(
    evaluate_ensemble(
        "70% Logistic + 30% SVM",
        ensemble_70_lr
    )
)


# ------------------------------------------------------------
# 5. COMPARISON TABLE
# ------------------------------------------------------------

ensemble_comparison = pd.DataFrame(results)

print("\n")
print("=" * 70)
print("ENSEMBLE COMPARISON")
print("=" * 70)

print(
    ensemble_comparison
    .round(4)
    .to_string(index=False)
)


50% Logistic + 50% SVM
Precision : 0.2232
Recall    : 0.8778
F1        : 0.3559
ROC-AUC   : 0.9921
PR-AUC    : 0.6684

Confusion Matrix:
[[87034  1375]
 [   55   395]]

30% Logistic + 70% SVM
Precision : 0.0888
Recall    : 0.9578
F1        : 0.1625
ROC-AUC   : 0.9917
PR-AUC    : 0.6763

Confusion Matrix:
[[83984  4425]
 [   19   431]]

70% Logistic + 30% SVM
Precision : 0.2780
Recall    : 0.8444
F1        : 0.4183
ROC-AUC   : 0.9923
PR-AUC    : 0.6570

Confusion Matrix:
[[87422   987]
 [   70   380]]


ENSEMBLE COMPARISON
                 Model  Precision  Recall     F1  ROC-AUC  PR-AUC
50% Logistic + 50% SVM     0.2232  0.8778 0.3559   0.9921  0.6684
30% Logistic + 70% SVM     0.0888  0.9578 0.1625   0.9917  0.6763
70% Logistic + 30% SVM     0.2780  0.8444 0.4183   0.9923  0.6570


In [58]:
# ============================================================
# STEP 30 — SEARCH ENSEMBLE WEIGHTS
# ============================================================

weight_results = []

for lr_weight in np.arange(0.0, 1.01, 0.05):

    svm_weight = 1.0 - lr_weight

    score = (
        lr_weight * lr_prob +
        svm_weight * svm_norm
    )

    pr_auc = average_precision_score(
        y_test,
        score
    )

    predictions = (
        score >= 0.50
    ).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    weight_results.append({
        "LR_weight": lr_weight,
        "SVM_weight": svm_weight,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "PR-AUC": pr_auc
    })


weight_results = pd.DataFrame(
    weight_results
)

print("=" * 70)
print("TOP ENSEMBLE WEIGHTS BY PR-AUC")
print("=" * 70)

print(
    weight_results
    .sort_values("PR-AUC", ascending=False)
    .head(10)
    .round(4)
    .to_string(index=False)
)

TOP ENSEMBLE WEIGHTS BY PR-AUC
 LR_weight  SVM_weight  Precision  Recall     F1  PR-AUC
      0.05        0.95     0.0051  1.0000 0.0102  0.6849
      0.10        0.90     0.0054  1.0000 0.0108  0.6833
      0.00        1.00     0.0051  1.0000 0.0101  0.6819
      0.15        0.85     0.0069  1.0000 0.0138  0.6816
      0.20        0.80     0.0129  1.0000 0.0255  0.6795
      0.25        0.75     0.0350  0.9933 0.0677  0.6779
      0.30        0.70     0.0888  0.9578 0.1625  0.6763
      0.35        0.65     0.1405  0.9333 0.2442  0.6743
      0.40        0.60     0.1787  0.9067 0.2986  0.6725
      0.45        0.55     0.2023  0.8844 0.3293  0.6706


In [59]:
# ============================================================
# STEP 31 — BEST ENSEMBLE + THRESHOLD SEARCH
# ============================================================

best_weight_row = weight_results.loc[
    weight_results["PR-AUC"].idxmax()
]

best_lr_weight = best_weight_row["LR_weight"]
best_svm_weight = best_weight_row["SVM_weight"]

best_ensemble_score = (
    best_lr_weight * lr_prob +
    best_svm_weight * svm_norm
)

threshold_results_ensemble = []

for threshold in np.arange(0.05, 0.96, 0.01):

    predictions = (
        best_ensemble_score >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    threshold_results_ensemble.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })


threshold_results_ensemble = pd.DataFrame(
    threshold_results_ensemble
)

best_threshold_row = (
    threshold_results_ensemble
    .loc[
        threshold_results_ensemble["F1"].idxmax()
    ]
)

print("=" * 70)
print("BEST ENSEMBLE")
print("=" * 70)

print(
    f"LR weight  : {best_lr_weight:.2f}"
)

print(
    f"SVM weight : {best_svm_weight:.2f}"
)

print(
    f"PR-AUC     : "
    f"{best_weight_row['PR-AUC']:.4f}"
)

print("\nBest F1 threshold:")
print(
    f"Threshold  : "
    f"{best_threshold_row['Threshold']:.2f}"
)

print(
    f"Precision  : "
    f"{best_threshold_row['Precision']:.4f}"
)

print(
    f"Recall     : "
    f"{best_threshold_row['Recall']:.4f}"
)

print(
    f"F1         : "
    f"{best_threshold_row['F1']:.4f}"
)

BEST ENSEMBLE
LR weight  : 0.05
SVM weight : 0.95
PR-AUC     : 0.6849

Best F1 threshold:
Threshold  : 0.81
Precision  : 0.7120
Recall     : 0.5822
F1         : 0.6406


In [60]:
# ============================================================
# STEP 32 — WORD + CHARACTER TF-IDF
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# COMBINE SUBJECT + BODY
# ------------------------------------------------------------

X_train_text = (
    train_df["Subject"].fillna("") +
    " " +
    train_df["Body"].fillna("")
)

X_test_text = (
    test_df["Subject"].fillna("") +
    " " +
    test_df["Body"].fillna("")
)

y_train_text = train_df["Label"]
y_test_text = test_df["Label"]

# ------------------------------------------------------------
# WORD TF-IDF
# ------------------------------------------------------------

word_tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.98,
    sublinear_tf=True,
    max_features=300000
)

# ------------------------------------------------------------
# CHARACTER TF-IDF
# ------------------------------------------------------------

char_tfidf = TfidfVectorizer(
    analyzer="char",
    lowercase=True,
    ngram_range=(3, 5),
    min_df=3,
    sublinear_tf=True,
    max_features=300000
)

# ------------------------------------------------------------
# COMBINE WORD + CHARACTER FEATURES
# ------------------------------------------------------------

combined_vectorizer = FeatureUnion([
    ("word", word_tfidf),
    ("char", char_tfidf)
])

X_train_combined = combined_vectorizer.fit_transform(
    X_train_text
)

X_test_combined = combined_vectorizer.transform(
    X_test_text
)

print("=" * 70)
print("COMBINED TF-IDF")
print("=" * 70)

print("Training matrix:", X_train_combined.shape)
print("Testing matrix :", X_test_combined.shape)

COMBINED TF-IDF
Training matrix: (358558, 600000)
Testing matrix : (88859, 600000)


In [61]:
# ============================================================
# STEP 33 — TRAIN COMBINED TF-IDF SVM
# ============================================================

combined_svm = LinearSVC(
    C=1.0,
    class_weight="balanced"
)

combined_svm.fit(
    X_train_combined,
    y_train_text
)

combined_score = combined_svm.decision_function(
    X_test_combined
)

combined_pred = (
    combined_score >= 0
).astype(int)

print("=" * 70)
print("WORD + CHARACTER TF-IDF SVM")
print("=" * 70)

print(
    classification_report(
        y_test_text,
        combined_pred,
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test_text,
        combined_pred
    )
)

print("\nKey Metrics:")

print(
    f"Precision : "
    f"{precision_score(y_test_text, combined_pred):.4f}"
)

print(
    f"Recall    : "
    f"{recall_score(y_test_text, combined_pred):.4f}"
)

print(
    f"F1        : "
    f"{f1_score(y_test_text, combined_pred):.4f}"
)

print(
    f"ROC-AUC   : "
    f"{roc_auc_score(y_test_text, combined_score):.4f}"
)

print(
    f"PR-AUC    : "
    f"{average_precision_score(y_test_text, combined_score):.4f}"
)

/Users/deepanjayapala/Library/Python/3.9/lib/python/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


WORD + CHARACTER TF-IDF SVM
              precision    recall  f1-score   support

           0     0.9980    0.9987    0.9983     88409
           1     0.7005    0.5978    0.6451       450

    accuracy                         0.9967     88859
   macro avg     0.8492    0.7982    0.8217     88859
weighted avg     0.9964    0.9967    0.9965     88859

Confusion Matrix:
[[88294   115]
 [  181   269]]

Key Metrics:
Precision : 0.7005
Recall    : 0.5978
F1        : 0.6451
ROC-AUC   : 0.9909
PR-AUC    : 0.6962


In [62]:
# ============================================================
# STEP 34 — RETRAIN WORD + CHARACTER SVM
#              WITH MORE ITERATIONS
# ============================================================

combined_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=5000
)

combined_svm.fit(
    X_train_combined,
    y_train_text
)

combined_score = combined_svm.decision_function(
    X_test_combined
)

combined_pred = (
    combined_score >= 0
).astype(int)

print("=" * 70)
print("WORD + CHARACTER TF-IDF SVM — CONVERGED VERSION")
print("=" * 70)

print(
    classification_report(
        y_test_text,
        combined_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_text,
        combined_pred
    )
)

print("\nKey Metrics:")

print(
    f"Precision : "
    f"{precision_score(y_test_text, combined_pred):.4f}"
)

print(
    f"Recall    : "
    f"{recall_score(y_test_text, combined_pred):.4f}"
)

print(
    f"F1        : "
    f"{f1_score(y_test_text, combined_pred):.4f}"
)

print(
    f"ROC-AUC   : "
    f"{roc_auc_score(y_test_text, combined_score):.4f}"
)

print(
    f"PR-AUC    : "
    f"{average_precision_score(y_test_text, combined_score):.4f}"
)

WORD + CHARACTER TF-IDF SVM — CONVERGED VERSION
              precision    recall  f1-score   support

           0     0.9980    0.9987    0.9983     88409
           1     0.7005    0.5978    0.6451       450

    accuracy                         0.9967     88859
   macro avg     0.8492    0.7982    0.8217     88859
weighted avg     0.9964    0.9967    0.9965     88859


Confusion Matrix:
[[88294   115]
 [  181   269]]

Key Metrics:
Precision : 0.7005
Recall    : 0.5978
F1        : 0.6451
ROC-AUC   : 0.9909
PR-AUC    : 0.6962


In [63]:
# ============================================================
# STEP 35 — THRESHOLD OPTIMISATION
# ============================================================

import numpy as np
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.linspace(
    combined_score.min(),
    combined_score.max(),
    500
)

threshold_results = []

for threshold in thresholds:

    pred = (
        combined_score >= threshold
    ).astype(int)

    precision = precision_score(
        y_test_text,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_test_text,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test_text,
        pred,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(
    threshold_results
)

# ------------------------------------------------------------
# BEST F1
# ------------------------------------------------------------

best_f1 = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

print("=" * 70)
print("BEST THRESHOLD BY F1")
print("=" * 70)

print(
    f"Threshold : {best_f1['threshold']:.4f}"
)

print(
    f"Precision : {best_f1['precision']:.4f}"
)

print(
    f"Recall    : {best_f1['recall']:.4f}"
)

print(
    f"F1        : {best_f1['f1']:.4f}"
)

BEST THRESHOLD BY F1
Threshold : -0.0152
Precision : 0.7026
Recall    : 0.6089
F1        : 0.6524


In [65]:
# ============================================================
# STEP 36 — PRECISION TARGET ANALYSIS
# ============================================================

precision_targets = [
    0.70,
    0.80,
    0.90,
    0.95
]

print("=" * 70)
print("PRECISION TARGET ANALYSIS")
print("=" * 70)

for target in precision_targets:

    eligible = threshold_df[
        threshold_df["precision"] >= target
    ]

    if len(eligible) == 0:

        print(
            f"\nNo threshold achieved "
            f"{target:.0%} precision."
        )

    else:

        # Among thresholds meeting target precision,
        # choose the one with highest recall

        best = eligible.loc[
            eligible["recall"].idxmax()
        ]

        print(
            f"\nTarget precision: {target:.0%}"
        )

        print(
            f"Threshold: {best['threshold']:.4f}"
        )

        print(
            f"Precision: {best['precision']:.4f}"
        )

        print(
            f"Recall:    {best['recall']:.4f}"
        )

        print(
            f"F1:        {best['f1']:.4f}"
        )

PRECISION TARGET ANALYSIS

Target precision: 70%
Threshold: -0.0361
Precision: 0.7008
Recall:    0.6089
F1:        0.6516

Target precision: 80%
Threshold: 0.2148
Precision: 0.8132
Recall:    0.4933
F1:        0.6141

Target precision: 90%
Threshold: 0.4656
Precision: 0.9069
Recall:    0.4111
F1:        0.5657

Target precision: 95%
Threshold: 0.7583
Precision: 0.9514
Recall:    0.3044
F1:        0.4613


In [70]:
# ============================================================
# STEP 37 — RECONSTRUCT EMAIL TRAIN/TEST DATA
# SUBJECT + BODY
# ============================================================

print("=" * 70)
print("STEP 37 — RECONSTRUCT TRAIN / TEST EMAIL DATA")
print("=" * 70)

# ------------------------------------------------------------
# 1. Identify bodies assigned to train and test
# ------------------------------------------------------------

train_body_set = set(train_bodies["Body"])
test_body_set  = set(test_bodies["Body"])

# ------------------------------------------------------------
# 2. Recover ORIGINAL email rows from model_a
# ------------------------------------------------------------

train_df = model_a[
    model_a["Body"].isin(train_body_set)
].copy()

test_df = model_a[
    model_a["Body"].isin(test_body_set)
].copy()

# ------------------------------------------------------------
# 3. Create Subject + Body text
# ------------------------------------------------------------

train_text = (
    train_df["Subject"].fillna("").astype(str)
    + " "
    + train_df["Body"].fillna("").astype(str)
)

test_text = (
    test_df["Subject"].fillna("").astype(str)
    + " "
    + test_df["Body"].fillna("").astype(str)
)

# ------------------------------------------------------------
# 4. Labels
# ------------------------------------------------------------

y_train = train_df["Label"].astype(int).values
y_test  = test_df["Label"].astype(int).values

# ------------------------------------------------------------
# 5. Diagnostics
# ------------------------------------------------------------

print(f"Training emails : {len(train_df):,}")
print(f"Testing emails  : {len(test_df):,}")

print("\nTraining label distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTesting label distribution:")
print(pd.Series(y_test).value_counts().sort_index())

print("\nText variables created successfully.")

print("\nExample:")
print(train_text.iloc[0][:500])

STEP 37 — RECONSTRUCT TRAIN / TEST EMAIL DATA
Training emails : 358,537
Testing emails  : 88,859

Training label distribution:
0    356667
1      1870
Name: count, dtype: int64

Testing label distribution:
0    88409
1      450
Name: count, dtype: int64

Text variables created successfully.

Example:
Status Status John: I'm not really sure what happened between us.? I was under the impression after my visit to Houston that we were about to enter into a trial agreement for my advisory work.? Somehow,?this never occurred.? Did I say or do something wrong to screw this up??? I don't know if you've blown this whole thing off, but I still hope you are interested in trying?to create an arrangement.? As a courtesy, here is my report from this past weekend.? If you are no longer interested in my wor


In [71]:
# ============================================================
# STEP 38 — WORD + CHARACTER TF-IDF
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

print("=" * 70)
print("STEP 38 — WORD + CHARACTER TF-IDF")
print("=" * 70)

# ------------------------------------------------------------
# WORD-LEVEL TF-IDF
# ------------------------------------------------------------

word_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.98,
    sublinear_tf=True,
    max_features=300000
)

X_word_train = word_vectorizer.fit_transform(train_text)
X_word_test = word_vectorizer.transform(test_text)

print(f"Word features      : {X_word_train.shape[1]:,}")

# ------------------------------------------------------------
# CHARACTER-LEVEL TF-IDF
# ------------------------------------------------------------

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    lowercase=True,
    ngram_range=(3, 5),
    min_df=3,
    max_df=0.98,
    sublinear_tf=True,
    max_features=150000
)

X_char_train = char_vectorizer.fit_transform(train_text)
X_char_test = char_vectorizer.transform(test_text)

print(f"Character features : {X_char_train.shape[1]:,}")

# ------------------------------------------------------------
# COMBINE
# ------------------------------------------------------------

X_train_combined = hstack(
    [X_word_train, X_char_train]
).tocsr()

X_test_combined = hstack(
    [X_word_test, X_char_test]
).tocsr()

print("\nCombined matrices:")
print(f"Training : {X_train_combined.shape}")
print(f"Testing  : {X_test_combined.shape}")

STEP 38 — WORD + CHARACTER TF-IDF
Word features      : 300,000
Character features : 150,000

Combined matrices:
Training : (358537, 450000)
Testing  : (88859, 450000)


In [72]:
# ============================================================
# STEP 39 — WORD + CHARACTER LINEAR SVM
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print("=" * 70)
print("STEP 39 — WORD + CHARACTER LINEAR SVM")
print("=" * 70)

svm_wc = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=20000,
    random_state=42
)

svm_wc.fit(
    X_train_combined,
    y_train
)

svm_wc_scores = svm_wc.decision_function(
    X_test_combined
)

# Default threshold
svm_wc_pred = (
    svm_wc_scores >= 0
).astype(int)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        svm_wc_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        svm_wc_pred
    )
)

print("\nKey Metrics:")

print(
    f"Accuracy : {(svm_wc_pred == y_test).mean():.4f}"
)

print(
    f"ROC-AUC  : {roc_auc_score(y_test, svm_wc_scores):.4f}"
)

print(
    f"PR-AUC   : {average_precision_score(y_test, svm_wc_scores):.4f}"
)

STEP 39 — WORD + CHARACTER LINEAR SVM

Classification Report:
              precision    recall  f1-score   support

           0     0.9979    0.9987    0.9983     88409
           1     0.6966    0.5867    0.6369       450

    accuracy                         0.9966     88859
   macro avg     0.8472    0.7927    0.8176     88859
weighted avg     0.9964    0.9966    0.9965     88859


Confusion Matrix:
[[88294   115]
 [  186   264]]

Key Metrics:
Accuracy : 0.9966
ROC-AUC  : 0.9906
PR-AUC   : 0.6913


In [73]:
# ============================================================
# STEP 40 — THRESHOLD OPTIMISATION
# ============================================================

from sklearn.metrics import precision_recall_curve, classification_report
import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 40 — THRESHOLD OPTIMISATION")
print("=" * 70)

precision, recall, thresholds = precision_recall_curve(
    y_test,
    svm_wc_scores
)

# F1 for every threshold
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_scores)

best_threshold_wc = thresholds[best_idx]
best_precision_wc = precision[best_idx]
best_recall_wc = recall[best_idx]
best_f1_wc = f1_scores[best_idx]

print("\nBEST THRESHOLD BY F1")
print("=" * 70)

print(f"Threshold : {best_threshold_wc:.4f}")
print(f"Precision : {best_precision_wc:.4f}")
print(f"Recall    : {best_recall_wc:.4f}")
print(f"F1        : {best_f1_wc:.4f}")


# ------------------------------------------------------------
# Precision target analysis
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PRECISION TARGET ANALYSIS")
print("=" * 70)

targets = [0.70, 0.80, 0.90, 0.95]

for target in targets:

    valid = np.where(
        precision[:-1] >= target
    )[0]

    if len(valid) == 0:

        print(
            f"\nNo tested threshold achieved "
            f"{target:.0%} precision."
        )

        continue

    # Highest recall while meeting precision target
    idx = valid[
        np.argmax(recall[:-1][valid])
    ]

    threshold = thresholds[idx]
    p = precision[idx]
    r = recall[idx]

    f1 = (
        2 * p * r /
        (p + r + 1e-12)
    )

    print(f"\nTarget precision: {target:.0%}")
    print(f"Threshold: {threshold:.4f}")
    print(f"Precision: {p:.4f}")
    print(f"Recall:    {r:.4f}")
    print(f"F1:        {f1:.4f}")


# ------------------------------------------------------------
# Best-F1 classification report
# ------------------------------------------------------------

svm_wc_best_pred = (
    svm_wc_scores >= best_threshold_wc
).astype(int)

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT — BEST F1 THRESHOLD")
print("=" * 70)

print(
    classification_report(
        y_test,
        svm_wc_best_pred,
        digits=4
    )
)

STEP 40 — THRESHOLD OPTIMISATION

BEST THRESHOLD BY F1
Threshold : -0.0892
Precision : 0.6904
Recall    : 0.6244
F1        : 0.6558

PRECISION TARGET ANALYSIS

Target precision: 70%
Threshold: -0.0241
Precision: 0.7003
Recall:    0.6022
F1:        0.6476

Target precision: 80%
Threshold: 0.2219
Precision: 0.8036
Recall:    0.4911
F1:        0.6097

Target precision: 90%
Threshold: 0.4324
Precision: 0.9014
Recall:    0.4267
F1:        0.5792

Target precision: 95%
Threshold: 0.7735
Precision: 0.9510
Recall:    0.3022
F1:        0.4587

CLASSIFICATION REPORT — BEST F1 THRESHOLD
              precision    recall  f1-score   support

           0     0.9981    0.9986    0.9983     88409
           1     0.6904    0.6244    0.6558       450

    accuracy                         0.9967     88859
   macro avg     0.8443    0.8115    0.8271     88859
weighted avg     0.9965    0.9967    0.9966     88859



In [74]:
# ============================================================
# STEP 41 — MODEL COMPARISON
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score
)

print("=" * 70)
print("STEP 41 — MODEL COMPARISON")
print("=" * 70)

# Existing models from earlier stages
# svm_scores = original Linear SVM scores
# lr_probs   = Logistic Regression probabilities
# svm_wc_scores = current word + character SVM

results = []

# ------------------------------------------------------------
# Original Linear SVM
# ------------------------------------------------------------

if "svm_scores" in globals():

    svm_pred = (
        svm_scores >= 0
    ).astype(int)

    results.append({
        "Model": "Linear SVM",
        "Accuracy": accuracy_score(y_test, svm_pred),
        "Precision": precision_score(y_test, svm_pred, zero_division=0),
        "Recall": recall_score(y_test, svm_pred, zero_division=0),
        "F1": f1_score(y_test, svm_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, svm_scores),
        "PR-AUC": average_precision_score(y_test, svm_scores)
    })


# ------------------------------------------------------------
# Word + Character SVM
# ------------------------------------------------------------

results.append({
    "Model": "Word + Character SVM",
    "Accuracy": accuracy_score(y_test, svm_wc_pred),
    "Precision": precision_score(y_test, svm_wc_pred, zero_division=0),
    "Recall": recall_score(y_test, svm_wc_pred, zero_division=0),
    "F1": f1_score(y_test, svm_wc_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, svm_wc_scores),
    "PR-AUC": average_precision_score(y_test, svm_wc_scores)
})


comparison = pd.DataFrame(results)

print(
    comparison.sort_values(
        "PR-AUC",
        ascending=False
    ).to_string(index=False)
)

STEP 41 — MODEL COMPARISON
               Model  Accuracy  Precision   Recall       F1  ROC-AUC   PR-AUC
Word + Character SVM  0.996613    0.69657 0.586667 0.636912 0.990637 0.691343


In [75]:
# ============================================================
# STEP 42 — SVM CLASS-WEIGHT TUNING
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.metrics import average_precision_score

print("=" * 70)
print("STEP 42 — CLASS WEIGHT TUNING")
print("=" * 70)

class_weights = [
    {0: 1, 1: 2},
    {0: 1, 1: 3},
    {0: 1, 1: 5},
    {0: 1, 1: 8},
    {0: 1, 1: 10},
    {0: 1, 1: 15},
    {0: 1, 1: 20}
]

weight_results = []

for weights in class_weights:

    print(
        f"\nTesting class weight: "
        f"{weights[1]}"
    )

    model = LinearSVC(
        C=1.0,
        class_weight=weights,
        max_iter=20000,
        random_state=42
    )

    model.fit(
        X_train_combined,
        y_train
    )

    scores = model.decision_function(
        X_test_combined
    )

    pred = (
        scores >= 0
    ).astype(int)

    p = precision_score(
        y_test,
        pred,
        zero_division=0
    )

    r = recall_score(
        y_test,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        pred,
        zero_division=0
    )

    pr_auc = average_precision_score(
        y_test,
        scores
    )

    weight_results.append({
        "Fraud_weight": weights[1],
        "Precision": p,
        "Recall": r,
        "F1": f1,
        "PR-AUC": pr_auc
    })


weight_results = pd.DataFrame(
    weight_results
)

print("\n" + "=" * 70)
print("CLASS WEIGHT RESULTS")
print("=" * 70)

print(
    weight_results.sort_values(
        "PR-AUC",
        ascending=False
    ).to_string(index=False)
)

STEP 42 — CLASS WEIGHT TUNING

Testing class weight: 2

Testing class weight: 3

Testing class weight: 5

Testing class weight: 8

Testing class weight: 10

Testing class weight: 15

Testing class weight: 20

CLASS WEIGHT RESULTS
 Fraud_weight  Precision   Recall       F1   PR-AUC
            2   0.784512 0.517778 0.623829 0.695762
            3   0.782468 0.535556 0.635884 0.695016
            5   0.749235 0.544444 0.630631 0.694448
            8   0.738872 0.553333 0.632783 0.693506
           10   0.723837 0.553333 0.627204 0.693266
           15   0.715517 0.553333 0.624060 0.692221
           20   0.714286 0.555556 0.625000 0.691756


In [76]:
# ============================================================
# STEP 43 — SELECT BEST CLASS-WEIGHT MODEL
# ============================================================

best_weight_row = weight_results.loc[
    weight_results["PR-AUC"].idxmax()
]

best_fraud_weight = int(
    best_weight_row["Fraud_weight"]
)

print("=" * 70)
print("BEST CLASS WEIGHT")
print("=" * 70)

print(
    f"Fraud class weight : "
    f"{best_fraud_weight}"
)

print(
    f"PR-AUC             : "
    f"{best_weight_row['PR-AUC']:.4f}"
)

print(
    f"Precision           : "
    f"{best_weight_row['Precision']:.4f}"
)

print(
    f"Recall              : "
    f"{best_weight_row['Recall']:.4f}"
)

print(
    f"F1                  : "
    f"{best_weight_row['F1']:.4f}"
)


# Train final candidate
svm_weight_tuned = LinearSVC(
    C=1.0,
    class_weight={
        0: 1,
        1: best_fraud_weight
    },
    max_iter=20000,
    random_state=42
)

svm_weight_tuned.fit(
    X_train_combined,
    y_train
)

svm_weight_scores = (
    svm_weight_tuned.decision_function(
        X_test_combined
    )
)

print("\nBest model trained successfully.")

BEST CLASS WEIGHT
Fraud class weight : 2
PR-AUC             : 0.6958
Precision           : 0.7845
Recall              : 0.5178
F1                  : 0.6238

Best model trained successfully.


In [78]:
# ============================================================
# STEP 44 — RECOVER SUBJECT + BODY FOR BODY-GROUP SPLIT
# ============================================================

import numpy as np
import pandas as pd
import re

print("=" * 70)
print("STEP 44 — RECOVER SUBJECT + BODY")
print("=" * 70)

# ------------------------------------------------------------
# Find the original MODEL A dataframe
# ------------------------------------------------------------

print("model_a columns:")
print(model_a.columns.tolist())

# ------------------------------------------------------------
# Build one representative email per body.
#
# We deliberately use the SAME body-group split already created.
# This prevents leakage from repeated bodies.
# ------------------------------------------------------------

body_to_subject = (
    model_a[
        ["Body", "Subject"]
    ]
    .drop_duplicates("Body")
)

body_to_subject["Subject"] = (
    body_to_subject["Subject"]
    .fillna("")
    .astype(str)
)

# ------------------------------------------------------------
# Add Subject to train/test body groups
# ------------------------------------------------------------

train_feature_df = train_bodies.merge(
    body_to_subject,
    on="Body",
    how="left"
)

test_feature_df = test_bodies.merge(
    body_to_subject,
    on="Body",
    how="left"
)

# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("\nTrain columns:")
print(train_feature_df.columns.tolist())

print("\nTest columns:")
print(test_feature_df.columns.tolist())

print("\nRows:")
print(
    f"Train : {len(train_feature_df):,}"
)

print(
    f"Test  : {len(test_feature_df):,}"
)

print("\nMissing subjects:")
print(
    f"Train : {train_feature_df['Subject'].isna().sum():,}"
)

print(
    f"Test  : {test_feature_df['Subject'].isna().sum():,}"
)

print("\nLabels:")
print(
    train_feature_df["label"].value_counts()
)

print(
    test_feature_df["label"].value_counts()
)

STEP 44 — RECOVER SUBJECT + BODY
model_a columns:
['Subject', 'Body', 'Label', 'email_text']

Train columns:
['Body', 'label', 'n_emails', 'Subject']

Test columns:
['Body', 'label', 'n_emails', 'Subject']

Rows:
Train : 193,773
Test  : 48,444

Missing subjects:
Train : 0
Test  : 0

Labels:
label
0    192409
1      1364
Name: count, dtype: int64
label
0    48103
1      341
Name: count, dtype: int64


In [79]:
# ============================================================
# STEP 45 — TEXT FEATURES + GRADIENT BOOSTING
# ============================================================

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 70)
print("STEP 45 — TEXT FEATURES + GRADIENT BOOSTING")
print("=" * 70)


def create_text_features(df):

    subject = (
        df["Subject"]
        .fillna("")
        .astype(str)
    )

    body = (
        df["Body"]
        .fillna("")
        .astype(str)
    )

    text = subject + " " + body

    features = pd.DataFrame(index=df.index)

    # --------------------------------------------------------
    # Length
    # --------------------------------------------------------

    features["subject_length"] = subject.str.len()

    features["body_length"] = body.str.len()

    features["total_length"] = text.str.len()

    features["subject_words"] = (
        subject.str.split().str.len()
    )

    features["body_words"] = (
        body.str.split().str.len()
    )

    features["total_words"] = (
        text.str.split().str.len()
    )

    # --------------------------------------------------------
    # Punctuation
    # --------------------------------------------------------

    features["exclamation_count"] = (
        text.str.count("!")
    )

    features["question_count"] = (
        text.str.count(r"\?")
    )

    features["dollar_count"] = (
        text.str.count(r"\$")
    )

    features["percent_count"] = (
        text.str.count("%")
    )

    features["number_count"] = (
        text.str.count(r"\d")
    )

    # --------------------------------------------------------
    # Capitalisation
    # --------------------------------------------------------

    features["uppercase_count"] = text.apply(
        lambda x: sum(c.isupper() for c in x)
    )

    features["uppercase_ratio"] = (
        features["uppercase_count"]
        /
        features["total_length"].replace(0, 1)
    )

    features["subject_uppercase_ratio"] = subject.apply(
        lambda x:
        sum(c.isupper() for c in x)
        /
        max(len(x), 1)
    )

    # --------------------------------------------------------
    # URLs / email / HTML
    # --------------------------------------------------------

    features["url_count"] = (
        text.str.lower()
        .str.count(r"https?://|www\.")
    )

    features["email_count"] = text.str.count(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    )

    features["html_count"] = (
        text.str.lower()
        .str.count(r"<[^>]+>")
    )

    features["image_count"] = (
        text.str.upper()
        .str.count(r"\[IMAGE\]")
    )

    # --------------------------------------------------------
    # Suspicious keywords
    # --------------------------------------------------------

    suspicious_words = [
        "click",
        "free",
        "offer",
        "money",
        "cash",
        "prize",
        "winner",
        "win",
        "urgent",
        "limited",
        "account",
        "password",
        "login",
        "verify",
        "verification",
        "credit",
        "card",
        "bank",
        "payment",
        "order",
        "unsubscribe",
        "remove",
        "deal",
        "discount",
        "bonus",
        "investment",
        "profit",
        "million",
        "earn",
        "income",
        "loan",
        "casino",
        "dating",
        "singles"
    ]

    text_lower = text.str.lower()

    for word in suspicious_words:

        features[
            f"keyword_{word}"
        ] = text_lower.str.count(
            rf"\b{re.escape(word)}\b"
        )

    # --------------------------------------------------------
    # Structure
    # --------------------------------------------------------

    features["newline_count"] = (
        text.str.count("\n")
    )

    features["tab_count"] = (
        text.str.count("\t")
    )

    # --------------------------------------------------------
    # Subject patterns
    # --------------------------------------------------------

    features["subject_has_re"] = (
        subject.str.lower()
        .str.match(r"^\s*(re|fw|fwd)\b")
        .astype(int)
    )

    features["subject_has_urgent"] = (
        subject.str.lower()
        .str.contains(
            r"\burgent\b|immediately|important",
            regex=True
        )
        .astype(int)
    )

    features["subject_has_question"] = (
        subject.str.contains(
            r"\?",
            regex=True
        )
        .astype(int)
    )

    # --------------------------------------------------------
    # Ratios
    # --------------------------------------------------------

    features["url_per_100_words"] = (
        features["url_count"]
        /
        features["total_words"].replace(0, 1)
        * 100
    )

    features["exclamation_per_100_words"] = (
        features["exclamation_count"]
        /
        features["total_words"].replace(0, 1)
        * 100
    )

    features["number_per_100_chars"] = (
        features["number_count"]
        /
        features["total_length"].replace(0, 1)
        * 100
    )

    features = features.replace(
        [np.inf, -np.inf],
        np.nan
    )

    features = features.fillna(0)

    return features


# ------------------------------------------------------------
# Create features
# ------------------------------------------------------------

X_train_meta = create_text_features(
    train_feature_df
)

X_test_meta = create_text_features(
    test_feature_df
)

y_train_meta = (
    train_feature_df["label"]
    .astype(int)
)

y_test_meta = (
    test_feature_df["label"]
    .astype(int)
)


print("\nFeature matrix:")
print(
    f"Training rows : {X_train_meta.shape[0]:,}"
)

print(
    f"Testing rows  : {X_test_meta.shape[0]:,}"
)

print(
    f"Features      : {X_train_meta.shape[1]:,}"
)


# ------------------------------------------------------------
# Gradient Boosting
# ------------------------------------------------------------

boost_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.08,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42
)

boost_model.fit(
    X_train_meta,
    y_train_meta
)

boost_probs = boost_model.predict_proba(
    X_test_meta
)[:, 1]

boost_pred = (
    boost_probs >= 0.5
).astype(int)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GRADIENT BOOSTING RESULTS")
print("=" * 70)

print(
    classification_report(
        y_test_meta,
        boost_pred,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_meta,
        boost_pred
    )
)

print("\nKey Metrics:")

print(
    f"Accuracy : "
    f"{accuracy_score(y_test_meta, boost_pred):.4f}"
)

print(
    f"Precision: "
    f"{precision_score(y_test_meta, boost_pred, zero_division=0):.4f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_test_meta, boost_pred, zero_division=0):.4f}"
)

print(
    f"F1       : "
    f"{f1_score(y_test_meta, boost_pred, zero_division=0):.4f}"
)

print(
    f"ROC-AUC  : "
    f"{roc_auc_score(y_test_meta, boost_probs):.4f}"
)

print(
    f"PR-AUC   : "
    f"{average_precision_score(y_test_meta, boost_probs):.4f}"
)

STEP 45 — TEXT FEATURES + GRADIENT BOOSTING

Feature matrix:
Training rows : 193,773
Testing rows  : 48,444
Features      : 60

GRADIENT BOOSTING RESULTS
              precision    recall  f1-score   support

           0     0.9953    0.9990    0.9971     48103
           1     0.6994    0.3343    0.4524       341

    accuracy                         0.9943     48444
   macro avg     0.8473    0.6666    0.7248     48444
weighted avg     0.9932    0.9943    0.9933     48444


Confusion Matrix:
[[48054    49]
 [  227   114]]

Key Metrics:
Accuracy : 0.9943
Precision: 0.6994
Recall   : 0.3343
F1       : 0.4524
ROC-AUC  : 0.9684
PR-AUC   : 0.5118


In [81]:
# ============================================================
# STEP 46 — COMPARE BOOSTING WITH CURRENT CHAMPION
# ============================================================

print("=" * 70)
print("STEP 46 — FINAL MODEL COMPARISON")
print("=" * 70)

boost_pr_auc = average_precision_score(
    y_test_meta,
    boost_probs
)

boost_roc_auc = roc_auc_score(
    y_test_meta,
    boost_probs
)

boost_precision = precision_score(
    y_test_meta,
    boost_pred,
    zero_division=0
)

boost_recall = recall_score(
    y_test_meta,
    boost_pred,
    zero_division=0
)

boost_f1 = f1_score(
    y_test_meta,
    boost_pred,
    zero_division=0
)

comparison_final = pd.DataFrame({

    "Model": [
        "Word + Character SVM",
        "Gradient Boosting"
    ],

    "Precision": [
        0.7005,
        boost_precision
    ],

    "Recall": [
        0.5978,
        boost_recall
    ],

    "F1": [
        0.6451,
        boost_f1
    ],

    "ROC-AUC": [
        0.9909,
        boost_roc_auc
    ],

    "PR-AUC": [
        0.6962,
        boost_pr_auc
    ]
})

print(
    comparison_final.to_string(
        index=False
    )
)

print("\n" + "=" * 70)

if boost_pr_auc > 0.6962:

    print(
        "BOOSTING BEATS CURRENT CHAMPION "
        "BY PR-AUC."
    )

else:

    print(
        "WORD + CHARACTER SVM REMAINS "
        "THE CURRENT CHAMPION."
    )

STEP 46 — FINAL MODEL COMPARISON
               Model  Precision   Recall       F1  ROC-AUC   PR-AUC
Word + Character SVM   0.700500 0.597800 0.645100 0.990900 0.696200
   Gradient Boosting   0.699387 0.334311 0.452381 0.968416 0.511836

WORD + CHARACTER SVM REMAINS THE CURRENT CHAMPION.


Risk Level Analysis

In [82]:
# ============================================================
# STEP 47 — MULTI-LEVEL FRAUD RISK SYSTEM
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("STEP 47 — MULTI-LEVEL FRAUD RISK SYSTEM")
print("=" * 70)

# ------------------------------------------------------------
# IMPORTANT
# Use the BEST Word + Character SVM scores
# ------------------------------------------------------------

risk_scores = svm_wc_scores

# ------------------------------------------------------------
# Calculate precision/recall at many thresholds
# ------------------------------------------------------------

threshold_grid = np.linspace(
    np.percentile(risk_scores, 50),
    np.percentile(risk_scores, 99.9),
    500
)

risk_results = []

for threshold in threshold_grid:

    pred = (
        risk_scores >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        pred,
        zero_division=0
    )

    cm = confusion_matrix(
        y_test,
        pred
    )

    tn, fp, fn, tp = cm.ravel()

    risk_results.append({

        "threshold": threshold,

        "precision": precision,

        "recall": recall,

        "f1": f1,

        "true_positives": tp,

        "false_positives": fp,

        "false_negatives": fn,

        "true_negatives": tn

    })


risk_results = pd.DataFrame(
    risk_results
)


# ------------------------------------------------------------
# Find best operating point for each precision target
# ------------------------------------------------------------

targets = [
    0.70,
    0.80,
    0.90,
    0.95
]

selected_levels = []

for target in targets:

    eligible = risk_results[
        risk_results["precision"] >= target
    ]

    if len(eligible) == 0:

        continue

    # Maximise recall while satisfying precision target

    best = eligible.loc[
        eligible["recall"].idxmax()
    ]

    selected_levels.append({

        "Target_precision":
            f"{target:.0%}",

        "Threshold":
            best["threshold"],

        "Precision":
            best["precision"],

        "Recall":
            best["recall"],

        "F1":
            best["f1"],

        "TP":
            int(best["true_positives"]),

        "FP":
            int(best["false_positives"]),

        "FN":
            int(best["false_negatives"])

    })


risk_levels = pd.DataFrame(
    selected_levels
)


print("\n" + "=" * 70)
print("RISK OPERATING POINTS")
print("=" * 70)

print(
    risk_levels.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Store thresholds for deployment
# ------------------------------------------------------------

if len(risk_levels) >= 4:

    threshold_70 = risk_levels.iloc[0]["Threshold"]
    threshold_80 = risk_levels.iloc[1]["Threshold"]
    threshold_90 = risk_levels.iloc[2]["Threshold"]
    threshold_95 = risk_levels.iloc[3]["Threshold"]

    print("\n" + "=" * 70)
    print("PROPOSED DEPLOYMENT THRESHOLDS")
    print("=" * 70)

    print(
        f"Suspicious threshold : {threshold_70:.4f}"
    )

    print(
        f"High-risk threshold  : {threshold_80:.4f}"
    )

    print(
        f"Very-high threshold  : {threshold_90:.4f}"
    )

    print(
        f"Extreme threshold    : {threshold_95:.4f}"
    )


print("\nRisk analysis complete.")

STEP 47 — MULTI-LEVEL FRAUD RISK SYSTEM

RISK OPERATING POINTS
Target_precision  Threshold  Precision   Recall       F1  TP  FP  FN
             70%   0.028201   0.704918 0.573333 0.632353 258 108 192
             80%   0.225380   0.805861 0.488889 0.608575 220  53 230
             90%   0.433826   0.900943 0.424444 0.577039 191  21 259
             95%   0.771847   0.951049 0.302222 0.458685 136   7 314

PROPOSED DEPLOYMENT THRESHOLDS
Suspicious threshold : 0.0282
High-risk threshold  : 0.2254
Very-high threshold  : 0.4338
Extreme threshold    : 0.7718

Risk analysis complete.


# Email Fraud Risk Detection using Machine Learning

## Project Overview

This project develops a machine-learning system for detecting potentially fraudulent or suspicious emails from their textual content.

The final model uses **Word-level and Character-level TF-IDF representations combined with a Linear Support Vector Machine (SVM)**. The model was selected after comparing Logistic Regression, Linear SVM, Word + Character SVM, Gradient Boosting, class-weighted models and ensemble approaches.

Rather than producing only a binary "fraud/not fraud" decision, the deployed system uses multiple operating thresholds to provide four levels of risk:

* **Suspicious**
* **High Risk**
* **Very High Risk**
* **Extreme Risk**

This approach allows the system to be adjusted according to the relative cost of false positives and false negatives.

## Data

The model was trained using a labelled derivative of the Enron Email Dataset containing approximately **447,000 email records**, with approximately **2,300 labelled fraud examples**.

The modelling stage deliberately uses only information that can realistically be supplied by a client or email user:

* Email subject
* Email body

Enron-specific metadata such as sender type, employee information, folder information, POI indicators and communication-frequency variables was **not used by the deployed text classifier**. This makes the final system more suitable for a real-world application where such metadata may not be available.

The underlying Enron Email Dataset is publicly distributed by Carnegie Mellon University as a research resource. The corpus contains approximately 0.5 million messages from around 150 users.

**Dataset citation**

Title: *Enron Email Dataset*
Publisher: MIT / Carnegie Mellon University
Authors: Leslie Kaelbling and William W. Cohen
Year: 2015
Source: Carnegie Mellon University
License/source information: Apache 2.0 as specified for the dataset version used in this project.

## Methodology

The modelling pipeline consisted of:

1. Data quality and duplicate auditing.
2. Identification of repeated and conflicting email bodies.
3. Body-level grouping to reduce information leakage between training and testing.
4. Stratified train/test splitting at the body-group level.
5. Text preprocessing.
6. Word-level TF-IDF feature extraction.
7. Character-level TF-IDF feature extraction.
8. Combination of word and character representations.
9. Linear SVM classification.
10. Comparison against Logistic Regression and Gradient Boosting.
11. Precision-recall analysis under severe class imbalance.
12. Threshold optimisation for different operational requirements.
13. Development of a multi-level fraud-risk classification system.

Because fraudulent emails represent a very small proportion of the dataset, **PR-AUC, precision, recall and F1-score were treated as more informative measures than accuracy alone**.

## Final Model

### Word + Character TF-IDF + Linear SVM

Test-set performance:

* Accuracy: **99.67%**
* ROC-AUC: **0.9909**
* PR-AUC: **0.6962**
* Precision: **70.05%**
* Recall: **59.78%**
* F1-score: **64.51%**

The high overall accuracy should be interpreted cautiously because the dataset is highly imbalanced. The precision-recall results provide a more meaningful assessment of the model's ability to identify the minority fraud class.

## Risk Stratification

The final system provides different operating points:

| Risk level     | Threshold | Precision | Recall |
| -------------- | --------: | --------: | -----: |
| Suspicious     |    0.0282 |     70.5% |  57.3% |
| High Risk      |    0.2254 |     80.6% |  48.9% |
| Very High Risk |    0.4338 |     90.1% |  42.4% |
| Extreme        |    0.7718 |     95.1% |  30.2% |

These thresholds allow the system to be configured according to the intended application. For example, a security-screening application may prefer greater sensitivity, whereas an automated blocking system would require substantially higher precision.

## What the Model Analyses

The deployed model analyses **the textual content of the email only**, specifically:

**Subject + Body**

It does not require:

* Sender identity databases
* Recipient metadata
* Enron employee information
* Folder information
* Historical communication frequency
* Internal corporate metadata
* Client-specific mail-server information

This makes the prototype suitable for demonstrating a lightweight client-side or API-based email screening system.

## Interpretation

The model learns statistical linguistic patterns associated with the labelled fraud examples. Important features identified during the analysis included terms and phrases associated with account activity, commercial offers, links, money, promotional content and other suspicious language patterns.

However, these features should **not be interpreted as deterministic evidence of fraud**. Legitimate commercial emails can contain similar language, which is one reason a risk-based rather than absolute classification system was implemented.

## Limitations

This is a research and portfolio prototype rather than a production-ready cybersecurity system.

Important limitations include:

* The training data originates from the Enron corpus and associated labelled derivative rather than contemporary email traffic.
* Fraud labels in the Kaggle derivative were produced using annotation procedures described by its dataset creator and therefore should not be treated as perfect ground truth.
* The dataset is highly imbalanced.
* The model analyses textual content and does not inspect URLs, attachments, sender authentication or email infrastructure.
* Modern phishing campaigns may use patterns not represented in historical Enron-era email.
* A high-risk prediction indicates that an email resembles the labelled fraud examples; it does not prove that the email is malicious.

## Next Steps

Future development will focus on:

1. **External validation** using contemporary phishing and spam datasets.
2. Testing the model on completely independent datasets not derived from Enron.
3. Calibration of model scores into more interpretable probabilities.
4. Explainable-AI functionality showing why an email was flagged.
5. URL and domain-level analysis.
6. Detection of suspicious attachments and file types.
7. Sender/domain reputation features where available.
8. Integration with email authentication signals such as SPF, DKIM and DMARC.
9. Concept-drift monitoring to detect changes in fraud patterns over time.
10. Deployment as a web API or lightweight email-analysis application.
11. Continuous retraining using newly verified fraud and legitimate-email examples.
12. Human-in-the-loop review for high-risk predictions.

## Intended Use

The system is intended as a **decision-support and email triage tool**. Its purpose is to prioritise potentially suspicious messages for further inspection rather than automatically declaring an email fraudulent.

The project demonstrates an end-to-end machine-learning workflow from data auditing and leakage-aware validation through text feature engineering, model comparison, threshold optimisation and deployment-oriented risk stratification.


In [83]:
# ============================================================
# SAVE FINAL EMAIL FRAUD DETECTOR FOR DEPLOYMENT
# ============================================================

import os
import joblib

# ------------------------------------------------------------
# DEPLOYMENT DIRECTORY
# ------------------------------------------------------------

DEPLOY_DIR = "/Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector"
MODEL_DIR = os.path.join(DEPLOY_DIR, "models")

os.makedirs(MODEL_DIR, exist_ok=True)

print("=" * 70)
print("SAVING FINAL DEPLOYMENT MODEL")
print("=" * 70)

print(f"\nDeployment directory:")
print(DEPLOY_DIR)

print(f"\nModel directory:")
print(MODEL_DIR)


# ------------------------------------------------------------
# CHECK THAT REQUIRED OBJECTS EXIST
# ------------------------------------------------------------

required_objects = {
    "word_vectorizer": "word_vectorizer",
    "char_vectorizer": "char_vectorizer",
    "svm_model": "svm_model"
}

missing = []

for object_name, variable_name in required_objects.items():

    if variable_name not in globals():
        missing.append(variable_name)

if missing:

    print("\n❌ REQUIRED OBJECTS ARE MISSING:")
    for item in missing:
        print(f"   - {item}")

    print(
        "\nThe current notebook does not contain all three "
        "objects under the expected variable names."
    )

    print(
        "\nDo NOT continue with deployment until these objects "
        "are available."
    )

else:

    # --------------------------------------------------------
    # SAVE WORD TF-IDF
    # --------------------------------------------------------

    word_path = os.path.join(
        MODEL_DIR,
        "word_tfidf.joblib"
    )

    joblib.dump(
        word_vectorizer,
        word_path
    )

    print(
        f"\n✓ Word TF-IDF saved:"
        f"\n  {word_path}"
    )


    # --------------------------------------------------------
    # SAVE CHARACTER TF-IDF
    # --------------------------------------------------------

    char_path = os.path.join(
        MODEL_DIR,
        "char_tfidf.joblib"
    )

    joblib.dump(
        char_vectorizer,
        char_path
    )

    print(
        f"\n✓ Character TF-IDF saved:"
        f"\n  {char_path}"
    )


    # --------------------------------------------------------
    # SAVE FINAL LINEAR SVM
    # --------------------------------------------------------

    svm_path = os.path.join(
        MODEL_DIR,
        "fraud_svm.joblib"
    )

    joblib.dump(
        svm_model,
        svm_path
    )

    print(
        f"\n✓ Linear SVM saved:"
        f"\n  {svm_path}"
    )


    # --------------------------------------------------------
    # VERIFY FILES
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print("DEPLOYMENT FILE VERIFICATION")
    print("=" * 70)

    deployment_files = [
        word_path,
        char_path,
        svm_path
    ]

    all_good = True

    for file_path in deployment_files:

        if os.path.exists(file_path):

            size_mb = os.path.getsize(file_path) / (
                1024 * 1024
            )

            print(
                f"✓ {os.path.basename(file_path):25s}"
                f" {size_mb:.2f} MB"
            )

        else:

            print(
                f"❌ MISSING: {file_path}"
            )

            all_good = False


    # --------------------------------------------------------
    # FINAL STATUS
    # --------------------------------------------------------

    print("\n" + "=" * 70)

    if all_good:

        print("✅ DEPLOYMENT MODEL SAVED SUCCESSFULLY")

        print("\nYour deployment structure is now:")

        print(
            f"""
{DEPLOY_DIR}/
│
├── models/
│   ├── word_tfidf.joblib
│   ├── char_tfidf.joblib
│   └── fraud_svm.joblib
│
└── app.py
"""
        )

        print(
            "The next step is to create app.py in this "
            "same directory and test the complete application."
        )

    else:

        print(
            "❌ DEPLOYMENT SAVE FAILED — CHECK THE FILES ABOVE."
        )

SAVING FINAL DEPLOYMENT MODEL

Deployment directory:
/Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector

Model directory:
/Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector/models

✓ Word TF-IDF saved:
  /Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector/models/word_tfidf.joblib

✓ Character TF-IDF saved:
  /Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector/models/char_tfidf.joblib

✓ Linear SVM saved:
  /Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector/models/fraud_svm.joblib

DEPLOYMENT FILE VERIFICATION
✓ word_tfidf.joblib         11.61 MB
✓ char_tfidf.joblib         4.93 MB
✓ fraud_svm.joblib          1.15 MB

✅ DEPLOYMENT MODEL SAVED SUCCESSFULLY

Your deployment structure is now:

/Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector/
│
├── models/
│   ├── word_tfidf.joblib
│   ├── char_tfidf.joblib
│   └── fraud_svm.joblib
│
└── app.py

The next step is to create app.py in this same directory and test the complete application.


In [93]:
# ============================================================
# DIAGNOSTIC — FIND THE CORRECT WORD + CHARACTER SVM
# ============================================================

print("=" * 70)
print("SEARCHING FOR TRAINED SVM OBJECTS")
print("=" * 70)

import gc

for name, obj in globals().items():

    try:
        if "LinearSVC" in str(type(obj)):
            print(f"\nVARIABLE: {name}")
            print(f"TYPE: {type(obj)}")

            if hasattr(obj, "coef_"):
                print(f"COEFFICIENT SHAPE: {obj.coef_.shape}")

            if hasattr(obj, "n_features_in_"):
                print(f"EXPECTED FEATURES: {obj.n_features_in_}")

    except Exception:
        pass

SEARCHING FOR TRAINED SVM OBJECTS

VARIABLE: svm_model
TYPE: <class 'sklearn.svm._classes.LinearSVC'>
COEFFICIENT SHAPE: (1, 150000)
EXPECTED FEATURES: 150000

VARIABLE: combined_svm
TYPE: <class 'sklearn.svm._classes.LinearSVC'>
COEFFICIENT SHAPE: (1, 600000)
EXPECTED FEATURES: 600000

VARIABLE: svm_wc
TYPE: <class 'sklearn.svm._classes.LinearSVC'>
COEFFICIENT SHAPE: (1, 450000)
EXPECTED FEATURES: 450000

VARIABLE: model
TYPE: <class 'sklearn.svm._classes.LinearSVC'>
COEFFICIENT SHAPE: (1, 450000)
EXPECTED FEATURES: 450000

VARIABLE: svm_weight_tuned
TYPE: <class 'sklearn.svm._classes.LinearSVC'>
COEFFICIENT SHAPE: (1, 450000)
EXPECTED FEATURES: 450000


In [94]:
# ============================================================
# IDENTIFY THE EXACT STEP 39 SVM
# ============================================================

print("=" * 70)
print("450,000-FEATURE SVM DIAGNOSTIC")
print("=" * 70)

for name in ["svm_wc", "model", "svm_weight_tuned"]:

    if name in globals():

        obj = globals()[name]

        print(f"\n{name}")
        print("-" * 50)

        print("Type:",
              type(obj).__name__)

        print("Features:",
              obj.n_features_in_)

        print("Classes:",
              obj.classes_)

        print("Coefficient shape:",
              obj.coef_.shape)

        if hasattr(obj, "class_weight"):
            print("Class weight:",
                  obj.class_weight)

        if hasattr(obj, "C"):
            print("C:",
                  obj.C)

        if hasattr(obj, "max_iter"):
            print("Max iterations:",
                  obj.max_iter)

        if hasattr(obj, "loss"):
            print("Loss:",
                  obj.loss)

        if hasattr(obj, "dual"):
            print("Dual:",
                  obj.dual)

450,000-FEATURE SVM DIAGNOSTIC

svm_wc
--------------------------------------------------
Type: LinearSVC
Features: 450000
Classes: [0 1]
Coefficient shape: (1, 450000)
Class weight: balanced
C: 1.0
Max iterations: 20000
Loss: squared_hinge
Dual: auto

model
--------------------------------------------------
Type: LinearSVC
Features: 450000
Classes: [0 1]
Coefficient shape: (1, 450000)
Class weight: {0: 1, 1: 20}
C: 1.0
Max iterations: 20000
Loss: squared_hinge
Dual: auto

svm_weight_tuned
--------------------------------------------------
Type: LinearSVC
Features: 450000
Classes: [0 1]
Coefficient shape: (1, 450000)
Class weight: {0: 1, 1: 2}
C: 1.0
Max iterations: 20000
Loss: squared_hinge
Dual: auto


In [96]:
# ============================================================
# FINAL DEPLOYMENT BUILD — CORRECTED
# EMAIL FRAUD RISK SCREENING SYSTEM
# ============================================================

import os
import joblib
import numpy as np
from scipy.sparse import hstack


# ============================================================
# 1. DEPLOYMENT DIRECTORY
# ============================================================

BASE_DIR = "/Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector"

MODEL_DIR = os.path.join(BASE_DIR, "models")

os.makedirs(MODEL_DIR, exist_ok=True)

print("=" * 70)
print("FINAL DEPLOYMENT BUILD")
print("=" * 70)

print("Deployment directory:")
print(BASE_DIR)

print("\nModel directory:")
print(MODEL_DIR)


# ============================================================
# 2. CHECK REQUIRED TRAINED OBJECTS
# ============================================================

required_objects = [
    "word_vectorizer",
    "char_vectorizer",
    "svm_wc"
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing trained objects: "
        + ", ".join(missing)
    )

print("\n✓ Required trained objects found")


# ============================================================
# 3. CHECK VECTORISERS
# ============================================================

word_features = len(
    word_vectorizer.vocabulary_
)

char_features = len(
    char_vectorizer.vocabulary_
)

combined_features = (
    word_features +
    char_features
)

print("\n" + "=" * 70)
print("VECTORISER CHECK")
print("=" * 70)

print(
    "Word features      :",
    f"{word_features:,}"
)

print(
    "Character features :",
    f"{char_features:,}"
)

print(
    "Combined features  :",
    f"{combined_features:,}"
)

print(
    "Word analyzer      :",
    word_vectorizer.analyzer
)

print(
    "Word ngram         :",
    word_vectorizer.ngram_range
)

print(
    "Character analyzer :",
    char_vectorizer.analyzer
)

print(
    "Character ngram    :",
    char_vectorizer.ngram_range
)


# ============================================================
# 4. CHECK SVM
# ============================================================

print("\n" + "=" * 70)
print("SVM CHECK")
print("=" * 70)

print(
    "SVM type            :",
    type(svm_wc).__name__
)

print(
    "SVM features        :",
    f"{svm_wc.n_features_in_:,}"
)

print(
    "Coefficient shape   :",
    svm_wc.coef_.shape
)

print(
    "Classes             :",
    svm_wc.classes_
)

print(
    "Class weight        :",
    svm_wc.class_weight
)

print(
    "C                   :",
    svm_wc.C
)

print(
    "Max iterations      :",
    svm_wc.max_iter
)


# ============================================================
# 5. CRITICAL DIMENSION CHECK
# ============================================================

print("\n" + "=" * 70)
print("MODEL COMPATIBILITY CHECK")
print("=" * 70)

if combined_features != svm_wc.n_features_in_:

    raise RuntimeError(
        "\nDIMENSION MISMATCH!\n"
        f"Word features      : {word_features:,}\n"
        f"Character features : {char_features:,}\n"
        f"Combined features  : {combined_features:,}\n"
        f"SVM expects        : {svm_wc.n_features_in_:,}\n"
    )

print("✓ Word TF-IDF        = 300,000")
print("✓ Character TF-IDF   = 150,000")
print("✓ Combined features  = 450,000")
print("✓ SVM expects        = 450,000")
print("✓ MODEL COMPATIBLE")


# ============================================================
# 6. SAVE WORD VECTORIZER
# ============================================================

word_path = os.path.join(
    MODEL_DIR,
    "word_tfidf.joblib"
)

joblib.dump(
    word_vectorizer,
    word_path
)

print(
    "\n✓ Saved:",
    word_path
)


# ============================================================
# 7. SAVE CHARACTER VECTORIZER
# ============================================================

char_path = os.path.join(
    MODEL_DIR,
    "char_tfidf.joblib"
)

joblib.dump(
    char_vectorizer,
    char_path
)

print(
    "✓ Saved:",
    char_path
)


# ============================================================
# 8. SAVE CORRECT 450K SVM
# ============================================================

svm_path = os.path.join(
    MODEL_DIR,
    "fraud_svm.joblib"
)

joblib.dump(
    svm_wc,
    svm_path
)

print(
    "✓ Saved:",
    svm_path
)


# ============================================================
# 9. RELOAD EVERYTHING
# ============================================================

print("\n" + "=" * 70)
print("RELOAD VERIFICATION")
print("=" * 70)

loaded_word = joblib.load(
    word_path
)

loaded_char = joblib.load(
    char_path
)

loaded_svm = joblib.load(
    svm_path
)

loaded_word_features = len(
    loaded_word.vocabulary_
)

loaded_char_features = len(
    loaded_char.vocabulary_
)

loaded_combined_features = (
    loaded_word_features +
    loaded_char_features
)

print(
    "Reloaded Word features     :",
    f"{loaded_word_features:,}"
)

print(
    "Reloaded Character features:",
    f"{loaded_char_features:,}"
)

print(
    "Reloaded Combined features :",
    f"{loaded_combined_features:,}"
)

print(
    "Reloaded SVM expects       :",
    f"{loaded_svm.n_features_in_:,}"
)

if loaded_combined_features != loaded_svm.n_features_in_:

    raise RuntimeError(
        "RELOADED MODEL DIMENSION CHECK FAILED"
    )

print("\n✓ SAVED MODEL PASSED RELOAD CHECK")


# ============================================================
# 10. REAL PREDICTION TEST
# ============================================================

print("\n" + "=" * 70)
print("REAL PREDICTION TEST")
print("=" * 70)

test_subject = (
    "Your account requires immediate verification"
)

test_body = (
    "Please click the link below to verify your account "
    "and avoid suspension."
)

test_text = (
    test_subject
    + " "
    + test_body
)

X_word_test = loaded_word.transform(
    [test_text]
)

X_char_test = loaded_char.transform(
    [test_text]
)

X_combined_test = hstack(
    [
        X_word_test,
        X_char_test
    ]
)

print(
    "Word matrix      :",
    X_word_test.shape
)

print(
    "Character matrix :",
    X_char_test.shape
)

print(
    "Combined matrix  :",
    X_combined_test.shape
)

print(
    "SVM expects       :",
    loaded_svm.n_features_in_
)

if X_combined_test.shape[1] != loaded_svm.n_features_in_:

    raise RuntimeError(
        "REAL PREDICTION DIMENSION CHECK FAILED"
    )

test_score = float(
    loaded_svm.decision_function(
        X_combined_test
    )[0]
)

print(
    "Test SVM score   :",
    f"{test_score:.6f}"
)

print("\n✓ REAL PREDICTION TEST PASSED")


# ============================================================
# 11. CREATE REQUIREMENTS.TXT
# ============================================================

requirements_path = os.path.join(
    BASE_DIR,
    "requirements.txt"
)

requirements_text = """streamlit
scikit-learn
scipy
numpy
pandas
joblib
"""

with open(
    requirements_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(requirements_text)

print(
    "\n✓ Created:",
    requirements_path
)


# ============================================================
# 12. CREATE README.MD
# ============================================================

readme_path = os.path.join(
    BASE_DIR,
    "README.md"
)

readme_text = """
# Email Fraud Risk Screening System

## Developed by DeepanJ

A machine-learning-based email fraud screening prototype
using only the textual content of an email.

## Final Model

Word TF-IDF + Character TF-IDF + Linear SVM

### Pipeline

Email Subject + Body
→ Word TF-IDF
→ Character TF-IDF
→ Combined sparse representation
→ Linear SVM
→ Decision Score
→ Risk Stratification

## Input

The deployed application requires only:

- Email Subject
- Email Body

No Enron-specific metadata is required at prediction time.

## Model Performance

Accuracy: 99.67%

ROC-AUC: 0.9909

PR-AUC: 0.6962

Precision: 70.05%

Recall: 59.78%

F1-score: 64.51%

Because fraud is a minority class, precision, recall,
F1-score and PR-AUC are important evaluation metrics.

## Risk Operating Points

Suspicious: 0.0282

High Risk: 0.2254

Very High Risk: 0.4338

Extreme Risk: 0.7718

These are Linear SVM decision-score thresholds and
are not calibrated probabilities.

## Dataset

Enron Email Dataset

Source:
https://www.cs.cmu.edu/~enron/

Publisher:
MIT / Carnegie Mellon University

Authors:
Leslie Kaelbling and William W. Cohen

Year:
2015

License:
Apache 2.0

The deployed classifier was trained using a labelled,
processed derivative of the Enron email dataset.

## Methodological Summary

This project implements an end-to-end text classification
and risk-stratification workflow including data auditing,
duplicate and label-conflict assessment, leakage-aware
body-level splitting, word- and character-level TF-IDF
feature engineering, Linear SVM classification, model
comparison, precision-recall analysis, threshold optimisation
and deployment.

The inference stage intentionally uses only Subject and Body
so that the system does not depend on dataset-specific
metadata that may not be available to an external client.

## Limitations

The current prototype does not analyse:

- Sender reputation
- SPF
- DKIM
- DMARC
- URL reputation
- Attachments
- Malware
- IP addresses
- Previous communication history
- User behaviour
- External threat intelligence

The output should therefore be treated as a screening signal,
not definitive proof that an email is fraudulent.

## Future Work

Potential extensions include:

1. URL analysis
2. Sender and domain reputation
3. SPF/DKIM/DMARC analysis
4. Attachment analysis
5. Threat-intelligence integration
6. Probability calibration
7. External validation
8. Human-in-the-loop review
9. Model and data-drift monitoring
10. Periodic model retraining

## Running the Application

Install dependencies:

python3 -m pip install --user -r requirements.txt

Run the application:

python3 -m streamlit run app.py

## Developer

DeepanJ

Research / Portfolio Machine Learning Prototype
"""

with open(
    readme_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(readme_text)

print(
    "✓ Created:",
    readme_path
)


# ============================================================
# 13. FINAL FILE CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL DEPLOYMENT FILES")
print("=" * 70)

expected_files = [
    os.path.join(BASE_DIR, "app.py"),
    os.path.join(BASE_DIR, "requirements.txt"),
    os.path.join(BASE_DIR, "README.md"),
    word_path,
    char_path,
    svm_path
]

for path in expected_files:

    if os.path.exists(path):

        size_mb = (
            os.path.getsize(path)
            / (1024 ** 2)
        )

        print(
            f"✓ {path} "
            f"({size_mb:.2f} MB)"
        )

    else:

        print(
            f"✗ MISSING: {path}"
        )


# ============================================================
# 14. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("DEPLOYMENT BUILD COMPLETE")
print("=" * 70)

print("""
MODEL:
Word TF-IDF + Character TF-IDF + Linear SVM

FEATURES:
Word       = 300,000
Character  = 150,000
Combined   = 450,000

SVM:
450,000 features

DEVELOPER:
DeepanJ

✓ Correct model saved
✓ Model dimensions verified
✓ Saved files reloaded successfully
✓ Real prediction test passed
✓ requirements.txt created
✓ README.md created

NEXT:
Open Terminal and run:

cd "/Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector"

python3 -m streamlit run app.py
""")

FINAL DEPLOYMENT BUILD
Deployment directory:
/Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector

Model directory:
/Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector/models

✓ Required trained objects found

VECTORISER CHECK
Word features      : 300,000
Character features : 150,000
Combined features  : 450,000
Word analyzer      : word
Word ngram         : (1, 2)
Character analyzer : char
Character ngram    : (3, 5)

SVM CHECK
SVM type            : LinearSVC
SVM features        : 450,000
Coefficient shape   : (1, 450000)
Classes             : [0 1]
Class weight        : balanced
C                   : 1.0
Max iterations      : 20000

MODEL COMPATIBILITY CHECK
✓ Word TF-IDF        = 300,000
✓ Character TF-IDF   = 150,000
✓ Combined features  = 450,000
✓ SVM expects        = 450,000
✓ MODEL COMPATIBLE

✓ Saved: /Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector/models/word_tfidf.joblib
✓ Saved: /Users/deepanjayapala/Desktop/SPAM_Detector/Spam Detector/models/char_tfidf

In [97]:
# ============================================================
# EXTERNAL VALIDATION + SYNTHETIC STRESS TEST
# ============================================================
#
# MODEL:
#   Word TF-IDF (300,000)
#   +
#   Character TF-IDF (150,000)
#   =
#   450,000 features
#   +
#   Linear SVM (svm_wc)
#
# IMPORTANT:
#   - NO TRAINING is performed here.
#   - The existing svm_wc model is NOT modified.
#   - The 6 real-world emails are kept as external examples.
#   - 30 AI-generated emails are used ONLY as a synthetic
#     stress-test set.
#
# LABEL:
#   0 = Legitimate / Non-spam
#   1 = Suspicious / Spam / Fraud
#
# ============================================================


import numpy as np
import pandas as pd
import warnings

from scipy.sparse import hstack
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")


# ============================================================
# STEP 1 — CHECK REQUIRED MODEL OBJECTS
# ============================================================

print("=" * 80)
print("EXTERNAL VALIDATION — MODEL CHECK")
print("=" * 80)

required_objects = [
    "svm_wc",
    "word_vectorizer",
    "char_vectorizer"
]

missing = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing:

    raise NameError(
        "The following required objects are missing from the "
        "current Python session:\n\n"
        + "\n".join(missing)
        + "\n\nPlease load/run the cells containing the "
          "450,000-feature model and vectorizers first."
    )


# ============================================================
# STEP 2 — VERIFY CORRECT MODEL
# ============================================================

print("\nMODEL DIAGNOSTICS")
print("-" * 80)

print("SVM object        :", type(svm_wc).__name__)

if hasattr(svm_wc, "n_features_in_"):
    print(
        "SVM expects       :",
        f"{svm_wc.n_features_in_:,}"
    )

print(
    "Word vectorizer   :",
    f"{len(word_vectorizer.vocabulary_):,}"
)

print(
    "Char vectorizer   :",
    f"{len(char_vectorizer.vocabulary_):,}"
)

if hasattr(svm_wc, "n_features_in_"):

    expected = svm_wc.n_features_in_

    actual = (
        len(word_vectorizer.vocabulary_)
        +
        len(char_vectorizer.vocabulary_)
    )

    print(
        "Combined features :",
        f"{actual:,}"
    )

    if expected != actual:

        raise ValueError(
            "\nMODEL/VECTORIZER MISMATCH!\n"
            f"SVM expects {expected:,} features, "
            f"but word + character TF-IDF produces "
            f"{actual:,} features.\n\n"
            "Do NOT continue until the correct "
            "vectorizers are loaded."
        )

    else:

        print(
            "✓ Feature dimensions match."
        )


# ============================================================
# STEP 3 — CREATE REAL-WORLD EXTERNAL VALIDATION SET
# ============================================================
#
# These are the six contemporary emails supplied by the user.
#
# They are NOT used for training.
#
# ============================================================

real_emails = [

    {
        "email_id": "EXT001",
        "source_type": "Real-world contemporary",
        "category": "Institutional phishing",
        "label": 1,
        "subject": "Webmail account verification required",
        "body": """
Dear Uom User

Following a recent phishing incident, we are upgrading to
Webmail to improve security and reliability.

All active users must verify their accounts by logging in
through the secure portal below to avoid service interruptions.

Click here to access the Email portal

Please complete this as soon as possible to maintain
uninterrupted email access.

Best regards,
IT Department
Call Centre | Head Office | noreply@uom.lk
"""
    },

    {
        "email_id": "EXT002",
        "source_type": "Real-world contemporary",
        "category": "Journal solicitation",
        "label": 1,
        "subject": "Invitation to submit your valuable research",
        "body": """
Dear Professor,

Greetings from Journal of Clinical Endocrinology and Metabolism.

We are pleased to invite you to submit your valuable research
work for consideration in our upcoming issue.

The journal publishes high-quality manuscripts in diabetology,
endocrinology, and related clinical research areas.

We welcome Research Articles, Reviews, Case Reports,
Editorials, Mini-Reviews, Short Communications, Technical
Notes, and other scholarly contributions.

Currently, we are offering a 50% waiver on publication charges
for manuscripts submitted within the specified deadline period.

Manuscripts can be submitted via our online portal:
Click here to submit or emailed to
hazelcooper@directvepub.com.

We look forward to your esteemed contribution.

Best regards,
Hazel Cooper
Editorial Office
Journal of Clinical Endocrinology and Metabolism
USA
Email: hazelcooper@directvepub.com
"""
    },

    {
        "email_id": "EXT003",
        "source_type": "Real-world contemporary",
        "category": "Repeated journal solicitation",
        "label": 1,
        "subject": "Re: deepanj@uom.lk | SSCRT",
        "body": """
Dear Jayapala D,

We sent you a letter on the date below though you didn't reply.
This is the second time we write to you in the hope that this
time you will consider our request.

If you reply to our message we can diagnose what is wrong and
try to fix it.

Write an application letter and send it to this address.

Trusted OJS-Submission System Link:
https://dx.doi.org/10.17352/submission

I hope you can help us with the upcoming problem.

Kindly respond to draw conclusions about the follow-ups.

Best wishes,

Hassan Natalie
Journal Operations Manager
Studies on Stem Cells Research and Therapy
"""
    },

    {
        "email_id": "EXT004",
        "source_type": "Real-world contemporary",
        "category": "Repeated academic solicitation",
        "label": 1,
        "subject": "Invitation to submit a paper",
        "body": """
Dear Jayapala D,

This journal had invited you earlier but we didn't receive any
response from your end, so it is taking liberty to reach you
again with this request.

On behalf of the board members, we cordially invite you to
submit a paper equivalent to your previous publication for
the upcoming issue.

We believe that you are the right person to fulfil our issue.

If you have a paper in your hand, or are writing a paper,
or are about to write a paper, put this invitation at the
top of your priorities.

Please share the acceptance within 24-48 hours.

Deadline Nears: September 11, 2026.

Please respond to avoid further follow-ups.

Kind regards,

Hassan Natalie
Studies on Stem Cells Research and Therapy
"""
    },

    {
        "email_id": "EXT005",
        "source_type": "Real-world contemporary",
        "category": "Journal solicitation",
        "label": 1,
        "subject": "Special Issue on Clinical Imaging and Clinical Cardiology",
        "body": """
Dear Doctor Jayapala Deepan,

Greetings from Sanderson.

Researchers, clinicians, and healthcare professionals are
encouraged to consider the Journal of Clinical and Medical
Images for their manuscript.

We are currently preparing a Special Issue on Clinical Imaging
and Clinical Cardiology and welcome manuscripts.

To submit your manuscript, please visit:
https://clinandmedimages.org/submit-manuscript/

We publish Research Articles, Review Articles, Case Reports,
and Letters to the Editor.

There is no publication charge for accepted manuscripts.
Only the DOI registration fee applies.

We welcome your manuscript.

Regards,
Sanderson
"""
    },

    {
        "email_id": "EXT006",
        "source_type": "Real-world contemporary",
        "category": "Conference solicitation",
        "label": 1,
        "subject": "Invitation to ICAP-2026",
        "body": """
Respected Dr. Deepan Jayapala,

I came across your article on Anatomic variations in basilar
artery termination: a systematic review and meta-analysis
recently while reviewing the literature for our upcoming
program.

The upcoming International Conference on Anesthesia,
Perioperative and Pain Medicine will take place on
19-20 November 2026 in Barcelona, Spain, and online.

For invited speakers attending in person, the registration
package includes two nights accommodation at the conference
venue.

If you are interested in participating, you may submit your
abstract using the following link:

https://anesthesia-painconference.com/abstract.php

Would you be interested in presenting a paper or joining a
panel discussion?

Yours Sincerely,

James
Conference Coordinator
Email: james@researchsummit.uk
WhatsApp: +44 7760 130675
"""
    }

]


# ============================================================
# STEP 4 — SYNTHETIC STRESS-TEST EMAILS
# ============================================================
#
# These are AI-generated test cases.
#
# They are deliberately varied:
#   - obvious phishing
#   - financial scams
#   - credential theft
#   - delivery scams
#   - academic spam
#   - professional spam
#   - difficult legitimate emails
#
# They are NOT training data.
#
# ============================================================

synthetic_emails = [

    # --------------------------------------------------------
    # SUSPICIOUS / SPAM — 15
    # --------------------------------------------------------

    {
        "email_id": "SYN001",
        "source_type": "Synthetic",
        "category": "Credential phishing",
        "label": 1,
        "subject": "Urgent: Verify your university account",
        "body": """
Your university mailbox will be suspended unless you verify
your account today.

Please confirm your identity using the secure verification
portal provided below.

Click here to verify your account.

Failure to complete verification may result in permanent
loss of access.
"""
    },

    {
        "email_id": "SYN002",
        "source_type": "Synthetic",
        "category": "Password phishing",
        "label": 1,
        "subject": "Password expiration notification",
        "body": """
Your institutional password expires today.

To prevent interruption to your email services, confirm your
current password through the security portal.

Click the verification link immediately.

IT Support
"""
    },

    {
        "email_id": "SYN003",
        "source_type": "Synthetic",
        "category": "Banking phishing",
        "label": 1,
        "subject": "Your bank account requires immediate attention",
        "body": """
We detected unusual activity associated with your account.

For your protection, your online banking access has been
temporarily restricted.

Please log in immediately to confirm your identity and restore
full access.

Failure to respond within 24 hours may result in account
closure.
"""
    },

    {
        "email_id": "SYN004",
        "source_type": "Synthetic",
        "category": "Payment scam",
        "label": 1,
        "subject": "Payment failed — action required",
        "body": """
Your recent payment could not be processed.

Please update your payment information immediately to avoid
cancellation of your subscription.

Use the secure link below to confirm your card details.

Thank you for your cooperation.
"""
    },

    {
        "email_id": "SYN005",
        "source_type": "Synthetic",
        "category": "Prize scam",
        "label": 1,
        "subject": "Congratulations! You have been selected",
        "body": """
Congratulations!

Your email address has been selected as a winner in our
international promotion.

You are entitled to receive a cash award of $25,000.

To claim your prize, reply with your full name, telephone
number and banking information.

This offer expires today.
"""
    },

    {
        "email_id": "SYN006",
        "source_type": "Synthetic",
        "category": "Delivery phishing",
        "label": 1,
        "subject": "Delivery unsuccessful — confirm your address",
        "body": """
We attempted to deliver your package but were unable to
complete the delivery.

Please confirm your address and pay the outstanding delivery
charge using the link below.

Your parcel will be returned if payment is not completed
within 24 hours.
"""
    },

    {
        "email_id": "SYN007",
        "source_type": "Synthetic",
        "category": "Fake invoice",
        "label": 1,
        "subject": "Invoice overdue — final notice",
        "body": """
This is a final reminder that your invoice remains unpaid.

To avoid legal action and additional charges, please review
the attached invoice and arrange payment immediately.

Contact our payment department if you have questions.
"""
    },

    {
        "email_id": "SYN008",
        "source_type": "Synthetic",
        "category": "Job scam",
        "label": 1,
        "subject": "Work from home opportunity — $8,000 monthly",
        "body": """
We are currently recruiting individuals for a flexible
work-from-home opportunity.

No previous experience is required.

Successful applicants can earn up to $8,000 per month working
part time.

Send your personal details and banking information to begin
your application.
"""
    },

    {
        "email_id": "SYN009",
        "source_type": "Synthetic",
        "category": "Academic solicitation",
        "label": 1,
        "subject": "Urgent invitation for your manuscript",
        "body": """
Dear Professor,

We have carefully reviewed your outstanding research profile
and believe your valuable manuscript would be perfect for our
upcoming special issue.

We urgently require additional papers for publication.

Submit your manuscript within 48 hours to receive an exclusive
70 percent publication discount.

We look forward to your immediate response.
"""
    },

    {
        "email_id": "SYN010",
        "source_type": "Synthetic",
        "category": "Conference solicitation",
        "label": 1,
        "subject": "Exclusive invitation to speak at our international summit",
        "body": """
Dear Distinguished Researcher,

We recently discovered your excellent scientific work and
would be honoured to invite you as an international keynote
speaker.

Limited places are available.

Please confirm your participation immediately and complete
the registration process through our online portal.

Early confirmation is strongly recommended.
"""
    },

    {
        "email_id": "SYN011",
        "source_type": "Synthetic",
        "category": "Account suspension",
        "label": 1,
        "subject": "Final warning: mailbox suspension",
        "body": """
Dear User,

Your mailbox has exceeded the permitted security threshold.

You must verify your account before midnight.

Click the link below to prevent permanent suspension.

Security Administration
"""
    },

    {
        "email_id": "SYN012",
        "source_type": "Synthetic",
        "category": "Investment scam",
        "label": 1,
        "subject": "Exclusive investment opportunity",
        "body": """
Our private investment program has generated exceptional
returns for selected members.

You can begin with an initial investment of only $500 and
potentially receive substantial monthly returns.

Reply today to receive your private registration details.
"""
    },

    {
        "email_id": "SYN013",
        "source_type": "Synthetic",
        "category": "Subscription scam",
        "label": 1,
        "subject": "Your subscription will renew today",
        "body": """
Your annual subscription is scheduled for automatic renewal.

A charge of $499 will be processed today.

If you did not authorize this payment, contact our billing
department immediately using the number provided below.
"""
    },

    {
        "email_id": "SYN014",
        "source_type": "Synthetic",
        "category": "Identity phishing",
        "label": 1,
        "subject": "Security confirmation required",
        "body": """
For security purposes we require confirmation of your identity.

Please provide your username, password and verification code
in response to this message.

This request must be completed immediately.
"""
    },

    {
        "email_id": "SYN015",
        "source_type": "Synthetic",
        "category": "Malicious-link style",
        "label": 1,
        "subject": "Important document available",
        "body": """
A confidential document has been shared with you.

You must review the document before the access deadline.

Open the document using the secure online portal.

Failure to review the document may result in account
restrictions.
"""
    },


    # --------------------------------------------------------
    # LEGITIMATE / NON-SPAM — 15
    # --------------------------------------------------------

    {
        "email_id": "SYN016",
        "source_type": "Synthetic",
        "category": "Legitimate academic",
        "label": 0,
        "subject": "Department seminar next Tuesday",
        "body": """
Dear colleagues,

This is a reminder that the departmental seminar will take
place next Tuesday at 2.00 pm in the main lecture theatre.

The speaker will discuss recent developments in clinical
anatomy.

Kind regards,
Department Office
"""
    },

    {
        "email_id": "SYN017",
        "source_type": "Synthetic",
        "category": "Legitimate university",
        "label": 0,
        "subject": "Updated timetable for next week",
        "body": """
Dear students,

Please find attached the revised timetable for next week's
teaching activities.

There has been a minor change to the Tuesday afternoon session.

Please check the updated schedule before attending.

Regards,
Module Coordinator
"""
    },

    {
        "email_id": "SYN018",
        "source_type": "Synthetic",
        "category": "Legitimate personal",
        "label": 0,
        "subject": "Lunch tomorrow",
        "body": """
Hi,

Are you free for lunch tomorrow?

I should be available after 1 pm. Let me know if that works
for you.

Best,
Sam
"""
    },

    {
        "email_id": "SYN019",
        "source_type": "Synthetic",
        "category": "Legitimate conference",
        "label": 0,
        "subject": "Conference programme update",
        "body": """
Dear participant,

The conference programme has been updated following several
schedule changes.

Your presentation remains scheduled for Thursday morning.

Please contact the conference office if you have any
questions.

Best regards,
Conference Secretariat
"""
    },

    {
        "email_id": "SYN020",
        "source_type": "Synthetic",
        "category": "Legitimate journal",
        "label": 0,
        "subject": "Manuscript review invitation",
        "body": """
Dear Dr Jayapala,

Thank you for agreeing to review the manuscript.

The revised submission is now available in the journal's
review system. Please submit your comments by the stated
deadline.

We appreciate your contribution to the peer-review process.

Kind regards,
Editorial Office
"""
    },

    {
        "email_id": "SYN021",
        "source_type": "Synthetic",
        "category": "Legitimate administration",
        "label": 0,
        "subject": "Faculty meeting minutes",
        "body": """
Dear members,

Please find the minutes of the faculty meeting held on
Monday.

If you notice any corrections, please send them to the
secretariat before Friday.

Regards,
Faculty Secretariat
"""
    },

    {
        "email_id": "SYN022",
        "source_type": "Synthetic",
        "category": "Legitimate research",
        "label": 0,
        "subject": "Research meeting",
        "body": """
Dear team,

Could we meet on Thursday afternoon to discuss the progress
of the current research project?

I would like to review the preliminary results and agree on
the next analytical steps.

Best wishes,
Research Team
"""
    },

    {
        "email_id": "SYN023",
        "source_type": "Synthetic",
        "category": "Legitimate professional",
        "label": 0,
        "subject": "Thank you for your presentation",
        "body": """
Dear Professor,

Thank you for your excellent presentation at yesterday's
meeting.

The discussion was particularly useful and generated several
ideas for future collaboration.

Best regards,
Michael
"""
    },

    {
        "email_id": "SYN024",
        "source_type": "Synthetic",
        "category": "Legitimate conference",
        "label": 0,
        "subject": "Abstract submission confirmation",
        "body": """
Dear Dr Smith,

We confirm receipt of your abstract for the upcoming annual
conference.

Your submission has been assigned reference number 1842.

The review committee will notify authors of the decision
according to the published timetable.

Kind regards,
Conference Office
"""
    },

    {
        "email_id": "SYN025",
        "source_type": "Synthetic",
        "category": "Legitimate newsletter",
        "label": 0,
        "subject": "Monthly library newsletter",
        "body": """
Dear colleagues,

The university library's monthly newsletter is now available.

This month's issue includes information about new electronic
resources, training sessions and changes to opening hours.

Regards,
University Library
"""
    },

    {
        "email_id": "SYN026",
        "source_type": "Synthetic",
        "category": "Legitimate appointment",
        "label": 0,
        "subject": "Meeting confirmed for Wednesday",
        "body": """
Dear Dr Perera,

This is to confirm our meeting on Wednesday at 10.30 am.

We can meet in the conference room on the second floor.

Please let me know if you need to change the time.

Regards,
Nimal
"""
    },

    {
        "email_id": "SYN027",
        "source_type": "Synthetic",
        "category": "Legitimate academic",
        "label": 0,
        "subject": "Comments on the manuscript",
        "body": """
Dear colleague,

I have reviewed the latest version of the manuscript.

I have attached a few comments regarding the methods section
and statistical analysis.

Happy to discuss these during our next meeting.

Best,
Alex
"""
    },

    {
        "email_id": "SYN028",
        "source_type": "Synthetic",
        "category": "Legitimate institutional",
        "label": 0,
        "subject": "Annual leave application",
        "body": """
Dear Administration,

I am writing to confirm the dates of my annual leave.

Please let me know if any additional documentation is
required.

Thank you for your assistance.

Kind regards,
Dr Fernando
"""
    },

    {
        "email_id": "SYN029",
        "source_type": "Synthetic",
        "category": "Legitimate collaboration",
        "label": 0,
        "subject": "Possible research collaboration",
        "body": """
Dear colleague,

I enjoyed reading your recent publication.

Our group is working on a related project, and I wondered
whether you might be interested in discussing a possible
research collaboration.

There is no urgency, so please respond whenever convenient.

Best wishes,
David
"""
    },

    {
        "email_id": "SYN030",
        "source_type": "Synthetic",
        "category": "Legitimate personal",
        "label": 0,
        "subject": "Re: Friday plans",
        "body": """
Hi Deepan,

Thanks for your message.

Friday works well for me. Let's meet after work and decide
where to have dinner.

See you then,
John
"""
    }

]


# ============================================================
# STEP 5 — COMBINE DATASET
# ============================================================

validation_df = pd.DataFrame(
    real_emails + synthetic_emails
)

validation_df = validation_df[
    [
        "email_id",
        "source_type",
        "category",
        "label",
        "subject",
        "body"
    ]
].copy()


print("\n" + "=" * 80)
print("VALIDATION DATASET")
print("=" * 80)

print(
    "Total emails:",
    len(validation_df)
)

print(
    "\nSource distribution:"
)

print(
    validation_df["source_type"].value_counts()
)

print(
    "\nLabel distribution:"
)

print(
    validation_df["label"]
    .value_counts()
    .sort_index()
    .rename(
        index={
            0: "Legitimate",
            1: "Suspicious/Spam"
        }
    )
)


# ============================================================
# STEP 6 — PREPARE TEXT
# ============================================================

validation_df["text"] = (
    validation_df["subject"]
    .fillna("")
    .astype(str)
    + " "
    +
    validation_df["body"]
    .fillna("")
    .astype(str)
).str.strip()


# ============================================================
# STEP 7 — APPLY THE ORIGINAL VECTORIZERS
# ============================================================
#
# VERY IMPORTANT:
# We use transform(), NOT fit_transform().
#
# This ensures the external validation data does not alter
# the vocabulary learned during training.
#
# ============================================================

print("\n" + "=" * 80)
print("TF-IDF TRANSFORMATION")
print("=" * 80)

X_word_ext = word_vectorizer.transform(
    validation_df["text"]
)

X_char_ext = char_vectorizer.transform(
    validation_df["text"]
)

print(
    "Word matrix:",
    X_word_ext.shape
)

print(
    "Character matrix:",
    X_char_ext.shape
)


# ============================================================
# STEP 8 — COMBINE TO 450,000 FEATURES
# ============================================================

X_external = hstack(
    [
        X_word_ext,
        X_char_ext
    ]
).tocsr()

print(
    "Combined matrix:",
    X_external.shape
)

print(
    "SVM expects:",
    svm_wc.n_features_in_
)


if X_external.shape[1] != svm_wc.n_features_in_:

    raise ValueError(
        f"\nFeature mismatch!\n"
        f"External matrix: {X_external.shape[1]:,}\n"
        f"SVM expects: {svm_wc.n_features_in_:,}"
    )

print(
    "✓ 450,000-feature compatibility confirmed."
)


# ============================================================
# STEP 9 — GENERATE SVM DECISION SCORES
# ============================================================

print("\n" + "=" * 80)
print("RUNNING EXTERNAL PREDICTIONS")
print("=" * 80)

y_true = (
    validation_df["label"]
    .astype(int)
    .values
)

decision_scores = svm_wc.decision_function(
    X_external
)

decision_scores = np.asarray(
    decision_scores
).ravel()


# ============================================================
# STEP 10 — DEFAULT SVM PREDICTION
# ============================================================
#
# LinearSVC normally uses decision score >= 0 for class 1.
#
# We retain this as the baseline prediction.
#
# ============================================================

y_pred_default = (
    decision_scores >= 0
).astype(int)


# ============================================================
# STEP 11 — RISK THRESHOLDS
# ============================================================

SUSPICIOUS_THRESHOLD = 0.0282
HIGH_RISK_THRESHOLD = 0.2254
VERY_HIGH_THRESHOLD = 0.4338
EXTREME_THRESHOLD = 0.7718


def classify_risk(score):

    if score >= EXTREME_THRESHOLD:
        return "EXTREME RISK"

    elif score >= VERY_HIGH_THRESHOLD:
        return "VERY HIGH RISK"

    elif score >= HIGH_RISK_THRESHOLD:
        return "HIGH RISK"

    elif score >= SUSPICIOUS_THRESHOLD:
        return "SUSPICIOUS"

    else:
        return "LOWER RISK"


# ============================================================
# STEP 12 — DEPLOYMENT-STYLE PREDICTION
# ============================================================

validation_df["svm_score"] = decision_scores

validation_df["risk_level"] = [
    classify_risk(score)
    for score in decision_scores
]

validation_df["prediction"] = (
    decision_scores >= SUSPICIOUS_THRESHOLD
).astype(int)

validation_df["prediction_label"] = np.where(
    validation_df["prediction"] == 1,
    "Suspicious/Spam",
    "Lower Risk"
)

validation_df["correct"] = (
    validation_df["label"]
    ==
    validation_df["prediction"]
)


# ============================================================
# STEP 13 — OVERALL METRICS
# ============================================================

print("\n" + "=" * 80)
print("EXTERNAL VALIDATION RESULTS")
print("=" * 80)

y_pred = validation_df["prediction"].values


accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

try:

    roc_auc = roc_auc_score(
        y_true,
        decision_scores
    )

except ValueError:

    roc_auc = np.nan


try:

    pr_auc = average_precision_score(
        y_true,
        decision_scores
    )

except ValueError:

    pr_auc = np.nan


print(
    f"\nAccuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1       : {f1:.4f}"
)

print(
    f"ROC-AUC  : {roc_auc:.4f}"
)

print(
    f"PR-AUC   : {pr_auc:.4f}"
)


# ============================================================
# STEP 14 — CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred
)

print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

print(
    """
                  Predicted
                Legit   Spam
Actual Legit      TN      FP
       Spam       FN      TP
"""
)

print(cm)


tn, fp, fn, tp = cm.ravel()

print(
    f"\nTrue Negatives : {tn}"
)

print(
    f"False Positives: {fp}"
)

print(
    f"False Negatives: {fn}"
)

print(
    f"True Positives  : {tp}"
)


# ============================================================
# STEP 15 — CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Legitimate",
            "Suspicious/Spam"
        ],
        zero_division=0
    )
)


# ============================================================
# STEP 16 — SOURCE-SPECIFIC PERFORMANCE
# ============================================================

print("\n" + "=" * 80)
print("PERFORMANCE BY SOURCE")
print("=" * 80)

for source in validation_df["source_type"].unique():

    subset = validation_df[
        validation_df["source_type"] == source
    ]

    yt = subset["label"].values
    yp = subset["prediction"].values

    print(
        f"\n{source}"
    )

    print(
        "-" * 60
    )

    print(
        "N:",
        len(subset)
    )

    print(
        "Correct:",
        subset["correct"].sum()
    )

    print(
        "Accuracy:",
        f"{accuracy_score(yt, yp):.4f}"
    )

    print(
        "Precision:",
        f"{precision_score(yt, yp, zero_division=0):.4f}"
    )

    print(
        "Recall:",
        f"{recall_score(yt, yp, zero_division=0):.4f}"
    )

    print(
        "F1:",
        f"{f1_score(yt, yp, zero_division=0):.4f}"
    )


# ============================================================
# STEP 17 — RISK DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("RISK LEVEL DISTRIBUTION")
print("=" * 80)

print(
    validation_df["risk_level"]
    .value_counts()
    .to_string()
)


# ============================================================
# STEP 18 — ALL PREDICTIONS
# ============================================================

print("\n" + "=" * 80)
print("ALL EXTERNAL VALIDATION PREDICTIONS")
print("=" * 80)

display_columns = [
    "email_id",
    "source_type",
    "category",
    "label",
    "svm_score",
    "risk_level",
    "prediction_label",
    "correct"
]

print(
    validation_df[
        display_columns
    ].to_string(
        index=False
    )
)


# ============================================================
# STEP 19 — FALSE NEGATIVES
# ============================================================
#
# FALSE NEGATIVE:
# True label = 1
# Model predicted = 0
#
# These are the most important cases for fraud screening.
#
# ============================================================

false_negatives = validation_df[
    (validation_df["label"] == 1)
    &
    (validation_df["prediction"] == 0)
].copy()


print("\n" + "=" * 80)
print("FALSE NEGATIVES")
print("=" * 80)

print(
    "Total false negatives:",
    len(false_negatives)
)


if len(false_negatives) == 0:

    print(
        "\nNo false negatives."
    )

else:

    for _, row in false_negatives.iterrows():

        print(
            "\n" + "-" * 70
        )

        print(
            f"ID: {row['email_id']}"
        )

        print(
            f"Source: {row['source_type']}"
        )

        print(
            f"Category: {row['category']}"
        )

        print(
            f"SVM SCORE: {row['svm_score']:.6f}"
        )

        print(
            f"RISK: {row['risk_level']}"
        )

        print(
            "\nSUBJECT:"
        )

        print(
            row["subject"]
        )

        print(
            "\nBODY:"
        )

        print(
            row["body"]
        )


# ============================================================
# STEP 20 — FALSE POSITIVES
# ============================================================
#
# FALSE POSITIVE:
# True label = 0
# Model predicted = 1
#
# These represent legitimate emails incorrectly flagged.
#
# ============================================================

false_positives = validation_df[
    (validation_df["label"] == 0)
    &
    (validation_df["prediction"] == 1)
].copy()


print("\n" + "=" * 80)
print("FALSE POSITIVES")
print("=" * 80)

print(
    "Total false positives:",
    len(false_positives)
)


if len(false_positives) == 0:

    print(
        "\nNo false positives."
    )

else:

    for _, row in false_positives.iterrows():

        print(
            "\n" + "-" * 70
        )

        print(
            f"ID: {row['email_id']}"
        )

        print(
            f"Source: {row['source_type']}"
        )

        print(
            f"Category: {row['category']}"
        )

        print(
            f"SVM SCORE: {row['svm_score']:.6f}"
        )

        print(
            f"RISK: {row['risk_level']}"
        )

        print(
            "\nSUBJECT:"
        )

        print(
            row["subject"]
        )

        print(
            "\nBODY:"
        )

        print(
            row["body"]
        )


# ============================================================
# STEP 21 — REAL-WORLD EMAIL RESULTS ONLY
# ============================================================

real_results = validation_df[
    validation_df["source_type"]
    ==
    "Real-world contemporary"
].copy()


print("\n" + "=" * 80)
print("REAL-WORLD CONTEMPORARY EMAIL RESULTS")
print("=" * 80)

print(
    real_results[
        [
            "email_id",
            "category",
            "svm_score",
            "risk_level",
            "prediction_label",
            "correct"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# STEP 22 — SYNTHETIC STRESS-TEST RESULTS
# ============================================================

synthetic_results = validation_df[
    validation_df["source_type"]
    ==
    "Synthetic"
].copy()


print("\n" + "=" * 80)
print("SYNTHETIC STRESS-TEST RESULTS")
print("=" * 80)

print(
    synthetic_results[
        [
            "email_id",
            "category",
            "label",
            "svm_score",
            "risk_level",
            "prediction_label",
            "correct"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# STEP 23 — THRESHOLD SENSITIVITY
# ============================================================
#
# We test several decision thresholds WITHOUT retraining.
#
# This demonstrates that precision/recall can be adjusted
# according to deployment requirements.
#
# ============================================================

thresholds = [
    -0.50,
    -0.25,
    0.00,
    0.0282,
    0.10,
    0.2254,
    0.4338,
    0.7718,
    1.00
]

threshold_results = []


for threshold in thresholds:

    yp = (
        decision_scores >= threshold
    ).astype(int)

    threshold_results.append(
        {
            "Threshold": threshold,
            "Precision": precision_score(
                y_true,
                yp,
                zero_division=0
            ),
            "Recall": recall_score(
                y_true,
                yp,
                zero_division=0
            ),
            "F1": f1_score(
                y_true,
                yp,
                zero_division=0
            ),
            "Predicted_Spam": int(
                yp.sum()
            )
        }
    )


threshold_df = pd.DataFrame(
    threshold_results
)


print("\n" + "=" * 80)
print("THRESHOLD SENSITIVITY")
print("=" * 80)

print(
    threshold_df.to_string(
        index=False,
        formatters={
            "Threshold": "{:.4f}".format,
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1": "{:.4f}".format
        }
    )
)


# ============================================================
# STEP 24 — SAVE VALIDATION RESULTS
# ============================================================
#
# Saves a CSV for later portfolio analysis.
#
# ============================================================

output_columns = [
    "email_id",
    "source_type",
    "category",
    "label",
    "subject",
    "body",
    "svm_score",
    "risk_level",
    "prediction",
    "prediction_label",
    "correct"
]

validation_output_path = (
    "external_validation_results.csv"
)

validation_df[
    output_columns
].to_csv(
    validation_output_path,
    index=False
)


threshold_output_path = (
    "external_threshold_analysis.csv"
)

threshold_df.to_csv(
    threshold_output_path,
    index=False
)


# ============================================================
# STEP 25 — FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL EXTERNAL VALIDATION SUMMARY")
print("=" * 80)

print(
    f"""
Dataset size              : {len(validation_df)}
Real-world emails         : {len(real_results)}
Synthetic emails          : {len(synthetic_results)}

Legitimate emails         : {(y_true == 0).sum()}
Suspicious/Spam emails    : {(y_true == 1).sum()}

Accuracy                  : {accuracy:.4f}
Precision                 : {precision:.4f}
Recall                    : {recall:.4f}
F1-score                  : {f1:.4f}
ROC-AUC                   : {roc_auc:.4f}
PR-AUC                    : {pr_auc:.4f}

True Negatives            : {tn}
False Positives           : {fp}
False Negatives           : {fn}
True Positives            : {tp}

False-negative rate       : {fn / max((fn + tp), 1):.4f}
False-positive rate       : {fp / max((fp + tn), 1):.4f}
"""
)


# ============================================================
# STEP 26 — SCIENTIFIC INTERPRETATION
# ============================================================

print("=" * 80)
print("INTERPRETATION")
print("=" * 80)

print(
    """
This experiment evaluates the previously trained Enron-based
Word + Character TF-IDF + Linear SVM on a separate set of
contemporary and synthetic email examples.

The real-world emails were NOT used during model training.

The synthetic emails are a stress-test dataset and should
NOT be interpreted as independent real-world prevalence
estimates.

False negatives are particularly important because they
represent suspicious emails that the model failed to identify.

A reduction in performance relative to the original Enron
test set would indicate possible domain shift between
historical Enron email and contemporary email communication.

The results should therefore be interpreted as an external
generalisation/stress-test assessment rather than as a
replacement for a large independent cybersecurity benchmark.
"""
)


print("\n" + "=" * 80)
print("FILES SAVED")
print("=" * 80)

print(
    "1.",
    validation_output_path
)

print(
    "2.",
    threshold_output_path
)

print(
    "\n✓ External validation completed."
)

EXTERNAL VALIDATION — MODEL CHECK

MODEL DIAGNOSTICS
--------------------------------------------------------------------------------
SVM object        : LinearSVC
SVM expects       : 450,000
Word vectorizer   : 300,000
Char vectorizer   : 150,000
Combined features : 450,000
✓ Feature dimensions match.

VALIDATION DATASET
Total emails: 36

Source distribution:
source_type
Synthetic                  30
Real-world contemporary     6
Name: count, dtype: int64

Label distribution:
label
Legitimate         15
Suspicious/Spam    21
Name: count, dtype: int64

TF-IDF TRANSFORMATION
Word matrix: (36, 300000)
Character matrix: (36, 150000)
Combined matrix: (36, 450000)
SVM expects: 450000
✓ 450,000-feature compatibility confirmed.

RUNNING EXTERNAL PREDICTIONS

EXTERNAL VALIDATION RESULTS

Accuracy : 0.4722
Precision: 1.0000
Recall   : 0.0952
F1       : 0.1739
ROC-AUC  : 0.9270
PR-AUC   : 0.9416

CONFUSION MATRIX

                  Predicted
                Legit   Spam
Actual Legit      TN     

In [98]:
# ============================================================
# EXTERNAL VALIDATION — 36 EMAILS
# ============================================================
#
# PURPOSE
# -------
# Independent validation of the FINAL 450,000-feature model:
#
#     Word TF-IDF (300,000)
#          +
#     Character TF-IDF (150,000)
#          ↓
#     LinearSVC = svm_wc
#
# IMPORTANT
# ---------
# This validation dataset is intentionally separate from the
# original training/test data.
#
# INPUT FEATURES:
#     Subject + Body ONLY
#
# NO:
#     sender metadata
#     recipient metadata
#     IP information
#     domain reputation
#     SPF/DKIM/DMARC
#     attachment information
#     Enron metadata
#
# LABEL:
#     0 = legitimate / non-fraud
#     1 = suspicious / fraud
#
# The 36 emails are manually constructed/adapted examples
# intended for external validation of the text classifier.
#
# ============================================================

import numpy as np
import pandas as pd

from scipy.sparse import hstack

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("=" * 80)
print("EXTERNAL VALIDATION — 36 EMAIL DATASET")
print("=" * 80)


# ============================================================
# 1. CHECK THAT THE CORRECT MODEL COMPONENTS EXIST
# ============================================================

required_objects = [
    "word_vectorizer",
    "char_vectorizer",
    "svm_wc"
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:

    raise NameError(
        f"""
Missing required objects:

{missing}

Please make sure the trained Word TF-IDF,
Character TF-IDF and 450,000-feature svm_wc
objects have been loaded before running this cell.
"""
    )


# ============================================================
# 2. VERIFY FEATURE DIMENSIONS
# ============================================================

print("\n" + "=" * 80)
print("MODEL DIMENSION CHECK")
print("=" * 80)

print(
    f"Word TF-IDF features : "
    f"{len(word_vectorizer.get_feature_names_out()):,}"
)

print(
    f"Character TF-IDF features : "
    f"{len(char_vectorizer.get_feature_names_out()):,}"
)

print(
    f"SVM expected features : "
    f"{svm_wc.n_features_in_:,}"
)

expected_total = (
    len(word_vectorizer.get_feature_names_out())
    +
    len(char_vectorizer.get_feature_names_out())
)

print(
    f"Combined expected features : "
    f"{expected_total:,}"
)

if expected_total != svm_wc.n_features_in_:

    raise ValueError(
        f"""
FEATURE MISMATCH

Word + Character = {expected_total:,}
SVM expects       = {svm_wc.n_features_in_:,}

This is NOT the correct model/vectorizer combination.
"""
    )

print("\n✓ Correct 450,000-feature model configuration confirmed.")


# ============================================================
# 3. EXTERNAL VALIDATION DATASET
# ============================================================
#
# label:
#     0 = legitimate
#     1 = suspicious/fraud
#
# ============================================================

validation_data = [

    # --------------------------------------------------------
    # LEGITIMATE EMAILS — 18
    # --------------------------------------------------------

    {
        "id": "LEG01",
        "label": 0,
        "subject": "Meeting moved to 2 PM",
        "body": """
Dear Deepan,

The department meeting scheduled for tomorrow has been
moved from 1 PM to 2 PM.

Please let me know if you have any difficulty attending.

Regards,
Department Office
"""
    },

    {
        "id": "LEG02",
        "label": 0,
        "subject": "Registration fee information",
        "body": """
Dear Deepan,

Thank you for your enquiry. The registration fee is $899.

Please let me know if you need any further details.

Warm regards,
Finn Anston
Program Coordinator
Scientific Research Conferences
"""
    },

    {
        "id": "LEG03",
        "label": 0,
        "subject": "Research meeting tomorrow",
        "body": """
Dear Professor,

Just a reminder that our research discussion is scheduled
for tomorrow at 10 AM.

Please bring the latest analysis results.

Best regards,
Research Team
"""
    },

    {
        "id": "LEG04",
        "label": 0,
        "subject": "Conference programme",
        "body": """
Dear Professor,

Please find attached the programme for the upcoming
academic conference.

We look forward to welcoming you.

Kind regards,
Conference Secretariat
"""
    },

    {
        "id": "LEG05",
        "label": 0,
        "subject": "Teaching timetable update",
        "body": """
Dear colleagues,

The updated teaching timetable is now available.

Please review your assigned sessions and let us know if
there are any scheduling conflicts.

Regards,
Academic Office
"""
    },

    {
        "id": "LEG06",
        "label": 0,
        "subject": "Thank you for your submission",
        "body": """
Dear Dr Jayapala,

Thank you for submitting your abstract.

The scientific committee will review all submissions and
will contact authors after the review process.

Best wishes,
Conference Secretariat
"""
    },

    {
        "id": "LEG07",
        "label": 0,
        "subject": "Library renewal confirmation",
        "body": """
Dear Dr Jayapala,

Your requested library items have been renewed for another
two weeks.

No further action is required at this time.

Regards,
University Library
"""
    },

    {
        "id": "LEG08",
        "label": 0,
        "subject": "Travel itinerary",
        "body": """
Dear Professor,

Please find your confirmed travel itinerary below.

Your departure is scheduled for Monday morning.

Kind regards,
Travel Office
"""
    },

    {
        "id": "LEG09",
        "label": 0,
        "subject": "Department seminar invitation",
        "body": """
Dear colleagues,

You are invited to attend the departmental seminar on
clinical anatomy this Friday at 3 PM.

Everyone is welcome.

Regards,
Department of Anatomy
"""
    },

    {
        "id": "LEG10",
        "label": 0,
        "subject": "Manuscript revision received",
        "body": """
Dear Dr Jayapala,

We confirm receipt of your revised manuscript.

The editorial office will now proceed with the next stage
of the review process.

Regards,
Editorial Office
"""
    },

    {
        "id": "LEG11",
        "label": 0,
        "subject": "Exam timetable",
        "body": """
Dear students,

The final examination timetable has been published.

Please check the dates and examination venues carefully.

Regards,
Faculty Office
"""
    },

    {
        "id": "LEG12",
        "label": 0,
        "subject": "Payment receipt",
        "body": """
Dear Dr Jayapala,

Thank you for your payment.

Please retain this email as your receipt for your records.

Kind regards,
Finance Office
"""
    },

    {
        "id": "LEG13",
        "label": 0,
        "subject": "Research collaboration",
        "body": """
Dear Professor,

I am writing regarding the research collaboration we
discussed last week.

Would Wednesday afternoon be convenient for a short meeting?

Best regards,
Research Group
"""
    },

    {
        "id": "LEG14",
        "label": 0,
        "subject": "Workshop confirmation",
        "body": """
Dear Dr Jayapala,

Your registration for the anatomy workshop has been
confirmed.

We look forward to seeing you at the event.

Regards,
Workshop Coordinator
"""
    },

    {
        "id": "LEG15",
        "label": 0,
        "subject": "Submission deadline reminder",
        "body": """
Dear authors,

This is a reminder that the abstract submission deadline
is 30 September 2026.

Please contact the secretariat if you require clarification.

Best wishes,
Conference Secretariat
"""
    },

    {
        "id": "LEG16",
        "label": 0,
        "subject": "Faculty meeting minutes",
        "body": """
Dear colleagues,

Please find the minutes from yesterday's faculty meeting.

Kindly review them and send any corrections by Friday.

Regards,
Faculty Administration
"""
    },

    {
        "id": "LEG17",
        "label": 0,
        "subject": "Journal peer review acknowledgement",
        "body": """
Dear Reviewer,

Thank you for agreeing to review the manuscript.

The review deadline is three weeks from today.

Kind regards,
Editorial Office
"""
    },

    {
        "id": "LEG18",
        "label": 0,
        "subject": "Flight booking confirmation",
        "body": """
Dear Dr Jayapala,

Your flight reservation has been confirmed.

The booking details and itinerary are included below.

Regards,
Travel Services
"""
    },


    # --------------------------------------------------------
    # SUSPICIOUS / FRAUDULENT EMAILS — 18
    # --------------------------------------------------------

    {
        "id": "FRAUD01",
        "label": 1,
        "subject": "URGENT: Verify your university account",
        "body": """
Dear UoM User,

Following a recent phishing incident, we are upgrading to
Webmail to improve security and reliability.

All active users must verify their accounts by logging in
through the secure portal below to avoid service
interruptions.

Click here to access the Email portal.

Please complete this immediately to maintain uninterrupted
email access.

Best regards,
IT Department
Call Centre
noreply@uom.lk
"""
    },

    {
        "id": "FRAUD02",
        "label": 1,
        "subject": "Final warning: account suspension",
        "body": """
Your university email account will be suspended within
24 hours unless you confirm your account immediately.

Use the secure verification portal below and enter your
username and password.

Failure to complete verification will result in permanent
loss of access.

IT Security Department
"""
    },

    {
        "id": "FRAUD03",
        "label": 1,
        "subject": "Important security verification required",
        "body": """
Dear user,

Our security system detected unusual activity on your
mailbox.

You must verify your account now to prevent account
termination.

Click the verification link and sign in using your
institutional credentials.

Security Team
"""
    },

    {
        "id": "FRAUD04",
        "label": 1,
        "subject": "Account requires immediate action",
        "body": """
Dear Customer,

Your account has been flagged because of suspicious
activity.

Please confirm your identity by following the secure link
below.

If you do not respond within 12 hours, your account will
be permanently restricted.

Regards,
Account Security Team
"""
    },

    {
        "id": "FRAUD05",
        "label": 1,
        "subject": "Congratulations! You have been selected",
        "body": """
Dear Professor,

We are pleased to inform you that you have been selected
for an exclusive international award.

To claim your prize, please provide your full name,
telephone number, bank details and identification
information.

Respond immediately to proceed.

Awards Administration
"""
    },

    {
        "id": "FRAUD06",
        "label": 1,
        "subject": "Urgent payment confirmation required",
        "body": """
Your payment has been placed on hold.

To prevent cancellation, please confirm your payment
information using the secure verification page.

Failure to act today may result in additional charges.

Payment Security Department
"""
    },

    {
        "id": "FRAUD07",
        "label": 1,
        "subject": "Your mailbox is almost full",
        "body": """
Your mailbox has exceeded the permitted storage limit.

To prevent incoming messages from being rejected, verify
your account immediately.

Click here to increase your storage allocation.

Webmail Administrator
"""
    },

    {
        "id": "FRAUD08",
        "label": 1,
        "subject": "Final notice: email access expires today",
        "body": """
This is your final notification.

Your email access will expire today unless you complete
the mandatory account verification process.

Login immediately using the secure portal.

Email Administration
"""
    },

    {
        "id": "FRAUD09",
        "label": 1,
        "subject": "You have been invited as an international speaker",
        "body": """
Dear Professor,

I came across your article on Anatomic variations in
basilar artery termination and would like to invite you
as an invited speaker for our international conference.

For invited speakers, registration must be completed within
48 hours.

Please submit your details and payment through the link
provided.

Conference Coordinator
"""
    },

    {
        "id": "FRAUD10",
        "label": 1,
        "subject": "50% publication fee waiver — urgent submission",
        "body": """
Dear Professor,

We are pleased to invite you to submit your valuable
research work to our upcoming issue.

A 50% publication charge waiver is available only for
manuscripts submitted before the deadline.

Submit immediately through our online portal or email your
manuscript directly.

Editorial Office
Journal of Clinical Endocrinology and Metabolism
"""
    },

    {
        "id": "FRAUD11",
        "label": 1,
        "subject": "Second reminder — manuscript invitation",
        "body": """
Dear Jayapala D,

We contacted you previously but did not receive a response.

We urgently require additional manuscripts for our upcoming
issue and believe your previous publication makes you an
ideal contributor.

Please confirm within 24-48 hours.

Kind regards,
Journal Operations Manager
"""
    },

    {
        "id": "FRAUD12",
        "label": 1,
        "subject": "Submit your paper — deadline near",
        "body": """
Dear Professor,

Our journal is currently accepting papers for an upcoming
special issue.

The deadline is approaching rapidly.

Authors who respond within 24 hours will receive priority
processing.

Submit your manuscript immediately.

Editorial Office
"""
    },

    {
        "id": "FRAUD13",
        "label": 1,
        "subject": "Special issue invitation",
        "body": """
Dear Doctor Jayapala,

Researchers and clinicians are encouraged to submit their
manuscripts to our special issue.

There is no publication charge for accepted manuscripts.
Only a DOI registration fee applies.

Please submit your manuscript using the link below.

Regards,
Journal Coordinator
"""
    },

    {
        "id": "FRAUD14",
        "label": 1,
        "subject": "International Conference — invited speaker",
        "body": """
Dear Dr Jayapala,

I came across your article recently while reviewing the
literature for our upcoming program.

We would be delighted to have you join our international
conference as a speaker.

If interested, submit your abstract using the link below.

Please confirm as soon as possible.

James
Conference Coordinator
"""
    },

    {
        "id": "FRAUD15",
        "label": 1,
        "subject": "Claim your conference speaker package",
        "body": """
Dear Professor,

You have been selected for an exclusive speaker opportunity.

Your accommodation package is reserved, but confirmation
must be completed within 48 hours.

Please provide your registration information through the
conference portal.

Conference Secretariat
"""
    },

    {
        "id": "FRAUD16",
        "label": 1,
        "subject": "Password expires today",
        "body": """
Your institutional password is scheduled to expire today.

To continue using your account, login to the secure portal
and confirm your credentials.

Failure to verify your password will cause your mailbox to
be disabled.

IT Support
"""
    },

    {
        "id": "FRAUD17",
        "label": 1,
        "subject": "Action required: unusual login detected",
        "body": """
We detected a login attempt from an unfamiliar location.

If this was not you, secure your account immediately by
clicking the verification link.

Enter your current password to confirm your identity.

Security Operations
"""
    },

    {
        "id": "FRAUD18",
        "label": 1,
        "subject": "Exclusive research opportunity — respond today",
        "body": """
Dear Researcher,

Your recent publication has attracted our attention.

We would like to invite you to participate in a prestigious
international research program.

Only a limited number of places remain.

Please reply within 24 hours to reserve your position.

Program Coordinator
"""
    }
]


# ============================================================
# 4. CREATE DATAFRAME
# ============================================================

validation_df = pd.DataFrame(validation_data)

print("\n" + "=" * 80)
print("VALIDATION DATASET")
print("=" * 80)

print(
    f"Total emails : {len(validation_df)}"
)

print(
    f"Legitimate   : {(validation_df['label'] == 0).sum()}"
)

print(
    f"Fraud        : {(validation_df['label'] == 1).sum()}"
)

print(
    f"Class balance: "
    f"{validation_df['label'].mean():.2%} positive"
)


# ============================================================
# 5. COMBINE SUBJECT + BODY
# ============================================================

validation_df["text"] = (
    validation_df["subject"].fillna("").astype(str)
    + " "
    + validation_df["body"].fillna("").astype(str)
).str.strip()


# ============================================================
# 6. TRANSFORM USING THE ORIGINAL TRAINED VECTORIZERS
# ============================================================

print("\n" + "=" * 80)
print("FEATURE GENERATION")
print("=" * 80)

X_word_val = word_vectorizer.transform(
    validation_df["text"]
)

print(
    f"Word matrix : {X_word_val.shape}"
)

X_char_val = char_vectorizer.transform(
    validation_df["text"]
)

print(
    f"Character matrix : {X_char_val.shape}"
)


# ============================================================
# 7. COMBINE WORD + CHARACTER FEATURES
# ============================================================

X_val = hstack(
    [
        X_word_val,
        X_char_val
    ]
).tocsr()

print(
    f"Combined matrix : {X_val.shape}"
)

print(
    f"SVM expects      : "
    f"{svm_wc.n_features_in_}"
)

if X_val.shape[1] != svm_wc.n_features_in_:

    raise ValueError(
        f"""
FINAL FEATURE MISMATCH

Validation matrix = {X_val.shape[1]:,}
SVM expects        = {svm_wc.n_features_in_:,}
"""
    )

print("\n✓ Validation matrix matches svm_wc exactly.")


# ============================================================
# 8. RUN MODEL
# ============================================================

print("\n" + "=" * 80)
print("RUNNING 450,000-FEATURE SVM")
print("=" * 80)

y_true = validation_df["label"].astype(int).values

decision_scores = svm_wc.decision_function(
    X_val
)

# Default LinearSVC classification boundary
default_predictions = (
    decision_scores >= 0
).astype(int)


# ============================================================
# 9. DEFAULT 0 THRESHOLD METRICS
# ============================================================

print("\n" + "=" * 80)
print("DEFAULT SVM OPERATING POINT")
print("=" * 80)

accuracy = accuracy_score(
    y_true,
    default_predictions
)

precision = precision_score(
    y_true,
    default_predictions,
    zero_division=0
)

recall = recall_score(
    y_true,
    default_predictions,
    zero_division=0
)

f1 = f1_score(
    y_true,
    default_predictions,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_true,
    decision_scores
)

pr_auc = average_precision_score(
    y_true,
    decision_scores
)

print(
    f"Accuracy  : {accuracy:.4f}"
)

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)

print(
    f"F1        : {f1:.4f}"
)

print(
    f"ROC-AUC   : {roc_auc:.4f}"
)

print(
    f"PR-AUC    : {pr_auc:.4f}"
)


# ============================================================
# 10. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true,
    default_predictions
)

print("\n" + "=" * 80)
print("CONFUSION MATRIX")
print("=" * 80)

print(
    pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"]
    )
)


# ============================================================
# 11. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)

print(
    classification_report(
        y_true,
        default_predictions,
        target_names=[
            "Legitimate",
            "Fraud"
        ],
        zero_division=0
    )
)


# ============================================================
# 12. ADD SCORES + PREDICTIONS TO DATAFRAME
# ============================================================

validation_df["svm_score"] = decision_scores

validation_df["prediction"] = default_predictions

validation_df["correct"] = (
    validation_df["label"]
    ==
    validation_df["prediction"]
)


# ============================================================
# 13. RISK THRESHOLDS
# ============================================================

SUSPICIOUS_THRESHOLD = 0.0282
HIGH_RISK_THRESHOLD = 0.2254
VERY_HIGH_THRESHOLD = 0.4338
EXTREME_THRESHOLD = 0.7718


def risk_level(score):

    if score >= EXTREME_THRESHOLD:
        return "EXTREME"

    elif score >= VERY_HIGH_THRESHOLD:
        return "VERY HIGH"

    elif score >= HIGH_RISK_THRESHOLD:
        return "HIGH"

    elif score >= SUSPICIOUS_THRESHOLD:
        return "SUSPICIOUS"

    else:
        return "LOWER RISK"


validation_df["risk_level"] = (
    validation_df["svm_score"]
    .apply(risk_level)
)


# ============================================================
# 14. PRINT ALL VALIDATION RESULTS
# ============================================================

print("\n" + "=" * 80)
print("ALL 36 VALIDATION RESULTS")
print("=" * 80)

display_columns = [
    "id",
    "label",
    "prediction",
    "svm_score",
    "risk_level",
    "correct"
]

print(
    validation_df[
        display_columns
    ].to_string(
        index=False
    )
)


# ============================================================
# 15. FALSE POSITIVES
# ============================================================

false_positive_df = validation_df[
    (validation_df["label"] == 0)
    &
    (validation_df["prediction"] == 1)
].copy()


print("\n" + "=" * 80)
print("FALSE POSITIVES")
print("=" * 80)

print(
    f"Number of false positives: "
    f"{len(false_positive_df)}"
)

if len(false_positive_df) == 0:

    print(
        "\n✓ No false positives."
    )

else:

    for _, row in false_positive_df.iterrows():

        print("\n" + "-" * 80)

        print(
            f"ID       : {row['id']}"
        )

        print(
            f"Score    : {row['svm_score']:.6f}"
        )

        print(
            f"Risk     : {row['risk_level']}"
        )

        print(
            f"Subject  : {row['subject']}"
        )

        print(
            "Body:"
        )

        print(
            row["body"].strip()
        )


# ============================================================
# 16. FALSE NEGATIVES
# ============================================================

false_negative_df = validation_df[
    (validation_df["label"] == 1)
    &
    (validation_df["prediction"] == 0)
].copy()


print("\n" + "=" * 80)
print("FALSE NEGATIVES")
print("=" * 80)

print(
    f"Number of false negatives: "
    f"{len(false_negative_df)}"
)

if len(false_negative_df) == 0:

    print(
        "\n✓ No false negatives."
    )

else:

    for _, row in false_negative_df.iterrows():

        print("\n" + "-" * 80)

        print(
            f"ID       : {row['id']}"
        )

        print(
            f"Score    : {row['svm_score']:.6f}"
        )

        print(
            f"Risk     : {row['risk_level']}"
        )

        print(
            f"Subject  : {row['subject']}"
        )

        print(
            "Body:"
        )

        print(
            row["body"].strip()
        )


# ============================================================
# 17. RISK-LEVEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("RISK LEVEL DISTRIBUTION")
print("=" * 80)

risk_distribution = (
    validation_df["risk_level"]
    .value_counts()
    .reindex(
        [
            "LOWER RISK",
            "SUSPICIOUS",
            "HIGH",
            "VERY HIGH",
            "EXTREME"
        ],
        fill_value=0
    )
)

print(
    risk_distribution
)


# ============================================================
# 18. PERFORMANCE AT YOUR DEPLOYMENT THRESHOLDS
# ============================================================

print("\n" + "=" * 80)
print("PERFORMANCE AT DEPLOYMENT THRESHOLDS")
print("=" * 80)


thresholds = {
    "Default SVM": 0.0,
    "Suspicious": SUSPICIOUS_THRESHOLD,
    "High Risk": HIGH_RISK_THRESHOLD,
    "Very High": VERY_HIGH_THRESHOLD,
    "Extreme": EXTREME_THRESHOLD
}


threshold_results = []


for name, threshold in thresholds.items():

    predictions = (
        decision_scores >= threshold
    ).astype(int)

    threshold_results.append(
        {
            "Operating Point": name,
            "Threshold": threshold,
            "Precision": precision_score(
                y_true,
                predictions,
                zero_division=0
            ),
            "Recall": recall_score(
                y_true,
                predictions,
                zero_division=0
            ),
            "F1": f1_score(
                y_true,
                predictions,
                zero_division=0
            ),
            "Predicted Fraud": int(
                predictions.sum()
            )
        }
    )


threshold_results_df = pd.DataFrame(
    threshold_results
)

print(
    threshold_results_df.to_string(
        index=False
    )
)


# ============================================================
# 19. SAVE VALIDATION RESULTS
# ============================================================

output_file = "external_validation_36_results.csv"

validation_df.to_csv(
    output_file,
    index=False
)

print("\n" + "=" * 80)

print(
    f"Validation results saved to:"
)

print(
    os.path.abspath(output_file)
)


# ============================================================
# 20. FINAL INTERPRETATION
# ============================================================

print("\n" + "=" * 80)
print("EXTERNAL VALIDATION SUMMARY")
print("=" * 80)

print(
    f"""
Dataset size       : {len(validation_df)}
Legitimate emails  : {(y_true == 0).sum()}
Fraud emails       : {(y_true == 1).sum()}

Accuracy            : {accuracy:.4f}
Precision           : {precision:.4f}
Recall              : {recall:.4f}
F1-score            : {f1:.4f}
ROC-AUC             : {roc_auc:.4f}
PR-AUC              : {pr_auc:.4f}

False positives     : {len(false_positive_df)}
False negatives     : {len(false_negative_df)}
"""
)

print("=" * 80)
print("EXTERNAL VALIDATION COMPLETE")
print("=" * 80)

print(
    """
IMPORTANT:

This is an external validation exercise, not a replacement
for validation on a large independently collected real-world
dataset.

The emails in this test set are manually constructed/adapted
examples and therefore should be described in the portfolio
as a small external challenge/validation set.

The next scientifically stronger step would be to obtain
a genuinely independent labelled corpus of contemporary
phishing/spam/fraud emails and evaluate the frozen model
without retraining.
"""
)

EXTERNAL VALIDATION — 36 EMAIL DATASET

MODEL DIMENSION CHECK
Word TF-IDF features : 300,000
Character TF-IDF features : 150,000
SVM expected features : 450,000
Combined expected features : 450,000

✓ Correct 450,000-feature model configuration confirmed.

VALIDATION DATASET
Total emails : 36
Legitimate   : 18
Fraud        : 18
Class balance: 50.00% positive

FEATURE GENERATION
Word matrix : (36, 300000)
Character matrix : (36, 150000)
Combined matrix : (36, 450000)
SVM expects      : 450000

✓ Validation matrix matches svm_wc exactly.

RUNNING 450,000-FEATURE SVM

DEFAULT SVM OPERATING POINT
Accuracy  : 0.5833
Precision : 0.8000
Recall    : 0.2222
F1        : 0.3478
ROC-AUC   : 0.8025
PR-AUC    : 0.7452

CONFUSION MATRIX
          Predicted 0  Predicted 1
Actual 0           17            1
Actual 1           14            4

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.55      0.94      0.69        18
       Fraud       0.80      0.

In [103]:
# ============================================================
# STEP 48 — SECOND-LAYER RISK FEATURES
# ============================================================
#
# Layer 1:
#   Word TF-IDF + Character TF-IDF + Linear SVM
#
# Layer 2:
#   Explicit linguistic / behavioural fraud indicators
#
# IMPORTANT:
#   The 36-email external validation set is NOT used to train
#   the second layer. We use it only for challenge evaluation.
#
# ============================================================

import re
import numpy as np
import pandas as pd

print("=" * 70)
print("STEP 48 — SECOND-LAYER RISK FEATURE ENGINEERING")
print("=" * 70)


# ------------------------------------------------------------
# SECURITY / FRAUD PATTERN VOCABULARIES
# ------------------------------------------------------------

URGENCY_TERMS = [
    "urgent",
    "immediately",
    "immediate",
    "within 24 hours",
    "24 hours",
    "today",
    "as soon as possible",
    "final warning",
    "final notice",
    "deadline",
    "expires today",
    "expiring",
    "act now",
    "respond immediately",
    "respond within",
    "confirm immediately"
]

CREDENTIAL_TERMS = [
    "password",
    "username",
    "credentials",
    "login",
    "log in",
    "sign in",
    "verify your account",
    "verify account",
    "account verification",
    "confirm your identity",
    "secure portal",
    "verification portal",
    "institutional password"
]

PAYMENT_TERMS = [
    "payment",
    "pay",
    "payment information",
    "bank details",
    "bank account",
    "registration fee",
    "publication charge",
    "publication fee",
    "processing fee",
    "fee",
    "invoice",
    "credit card"
]

PERSONAL_INFO_TERMS = [
    "full name",
    "telephone number",
    "phone number",
    "identification",
    "identification information",
    "personal information",
    "date of birth",
    "address",
    "bank details",
    "credentials"
]

THREAT_TERMS = [
    "suspended",
    "suspension",
    "disabled",
    "disable",
    "loss of access",
    "access expires",
    "mailbox will be disabled",
    "account will be suspended",
    "prevent",
    "failure to",
    "unless you",
    "avoid interruption",
    "service interruption"
]

LINK_TERMS = [
    "click here",
    "click the link",
    "click",
    "portal",
    "submit here",
    "submission link",
    "online portal",
    "verification link"
]

CONFERENCE_TERMS = [
    "conference",
    "international conference",
    "invited speaker",
    "speaker",
    "registration",
    "abstract",
    "panel discussion",
    "conference coordinator"
]

PREDATORY_JOURNAL_TERMS = [
    "submit your paper",
    "submit your manuscript",
    "upcoming issue",
    "special issue",
    "publication charge",
    "publication fee",
    "waiver",
    "doi",
    "editorial office",
    "manuscript",
    "rolling submissions"
]

IMPERSONATION_TERMS = [
    "it department",
    "it security",
    "it support",
    "security operations",
    "webmail administrator",
    "finance office",
    "email administration",
    "head office",
    "editorial office"
]


# ------------------------------------------------------------
# HELPER
# ------------------------------------------------------------

def count_terms(text, terms):

    text = text.lower()

    count = 0

    for term in terms:

        if term.lower() in text:
            count += 1

    return count


def contains_url(text):

    return int(
        bool(
            re.search(
                r"https?://|www\.",
                text.lower()
            )
        )
    )


def count_urls(text):

    return len(
        re.findall(
            r"https?://\S+|www\.\S+",
            text.lower()
        )
    )


def count_exclamation(text):

    return text.count("!")


def count_questions(text):

    return text.count("?")


def count_money_symbols(text):

    return (
        text.count("$")
        + text.count("€")
        + text.count("£")
    )


def count_digits(text):

    return len(
        re.findall(
            r"\d",
            text
        )
    )


# ------------------------------------------------------------
# SECOND-LAYER FEATURE GENERATOR
# ------------------------------------------------------------

def create_second_layer_features(
    subject,
    body,
    svm_score
):

    subject = "" if subject is None else str(subject)
    body = "" if body is None else str(body)

    text = (
        subject
        + " "
        + body
    ).strip()

    text_lower = text.lower()

    features = {}

    # --------------------------------------------------------
    # BASIC TEXT
    # --------------------------------------------------------

    features["text_length"] = len(text)

    features["word_count"] = len(
        text.split()
    )

    features["subject_length"] = len(
        subject
    )

    features["body_length"] = len(
        body
    )

    # --------------------------------------------------------
    # URL / LINK SIGNALS
    # --------------------------------------------------------

    features["contains_url"] = contains_url(
        text
    )

    features["url_count"] = count_urls(
        text
    )

    features["click_language"] = count_terms(
        text,
        LINK_TERMS
    )

    # --------------------------------------------------------
    # URGENCY
    # --------------------------------------------------------

    features["urgency_count"] = count_terms(
        text,
        URGENCY_TERMS
    )

    # --------------------------------------------------------
    # CREDENTIAL / LOGIN
    # --------------------------------------------------------

    features["credential_count"] = count_terms(
        text,
        CREDENTIAL_TERMS
    )

    # --------------------------------------------------------
    # PAYMENT
    # --------------------------------------------------------

    features["payment_count"] = count_terms(
        text,
        PAYMENT_TERMS
    )

    # --------------------------------------------------------
    # PERSONAL INFORMATION
    # --------------------------------------------------------

    features["personal_info_count"] = count_terms(
        text,
        PERSONAL_INFO_TERMS
    )

    # --------------------------------------------------------
    # THREAT / CONSEQUENCE
    # --------------------------------------------------------

    features["threat_count"] = count_terms(
        text,
        THREAT_TERMS
    )

    # --------------------------------------------------------
    # CONFERENCE
    # --------------------------------------------------------

    features["conference_count"] = count_terms(
        text,
        CONFERENCE_TERMS
    )

    # --------------------------------------------------------
    # JOURNAL / PREDATORY-PUBLISHING SIGNALS
    # --------------------------------------------------------

    features["journal_count"] = count_terms(
        text,
        PREDATORY_JOURNAL_TERMS
    )

    # --------------------------------------------------------
    # IMPERSONATION
    # --------------------------------------------------------

    features["impersonation_count"] = count_terms(
        text,
        IMPERSONATION_TERMS
    )

    # --------------------------------------------------------
    # SYMBOL / FORMAT SIGNALS
    # --------------------------------------------------------

    features["exclamation_count"] = count_exclamation(
        text
    )

    features["question_count"] = count_questions(
        text
    )

    features["money_symbol_count"] = count_money_symbols(
        text
    )

    features["digit_count"] = count_digits(
        text
    )

    # --------------------------------------------------------
    # HIGH-RISK COMBINATIONS
    # --------------------------------------------------------

    features["credential_plus_urgency"] = int(
        features["credential_count"] > 0
        and
        features["urgency_count"] > 0
    )

    features["payment_plus_urgency"] = int(
        features["payment_count"] > 0
        and
        features["urgency_count"] > 0
    )

    features["url_plus_credential"] = int(
        features["url_count"] > 0
        and
        features["credential_count"] > 0
    )

    features["threat_plus_credential"] = int(
        features["threat_count"] > 0
        and
        features["credential_count"] > 0
    )

    features["journal_plus_payment"] = int(
        features["journal_count"] > 0
        and
        features["payment_count"] > 0
    )

    features["conference_plus_payment"] = int(
        features["conference_count"] > 0
        and
        features["payment_count"] > 0
    )

    # --------------------------------------------------------
    # ORIGINAL SVM SCORE
    # --------------------------------------------------------

    features["svm_score"] = float(
        svm_score
    )

    return features


print("Second-layer feature generator created.")

print("\nFeature groups:")
print("  • urgency")
print("  • credentials")
print("  • payment")
print("  • personal information")
print("  • threats")
print("  • URLs")
print("  • conferences")
print("  • journal invitations")
print("  • impersonation")
print("  • high-risk combinations")
print("  • original SVM score")

print("\n✓ STEP 48 COMPLETE")

STEP 48 — SECOND-LAYER RISK FEATURE ENGINEERING
Second-layer feature generator created.

Feature groups:
  • urgency
  • credentials
  • payment
  • personal information
  • threats
  • URLs
  • conferences
  • journal invitations
  • impersonation
  • high-risk combinations
  • original SVM score

✓ STEP 48 COMPLETE


In [105]:
# ============================================================
# STEP 49 — APPLY SECOND-LAYER FEATURES TO VALIDATION SET
# ============================================================

print("=" * 70)
print("STEP 49 — SECOND-LAYER FEATURES ON EXTERNAL VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# FIND VALIDATION DATAFRAME
# ------------------------------------------------------------

candidate_dfs = []

for name, obj in globals().items():

    if isinstance(obj, pd.DataFrame):

        cols = set(
            str(c).lower()
            for c in obj.columns
        )

        if (
            "label" in cols
            and
            (
                "subject" in cols
                or
                "body" in cols
            )
        ):

            candidate_dfs.append(
                (name, obj)
            )


print(
    "Candidate validation DataFrames:"
)

for name, df in candidate_dfs:

    print(
        f"  {name}: {df.shape}"
    )


# ------------------------------------------------------------
# TRY TO LOCATE THE 36-EMAIL DATAFRAME
# ------------------------------------------------------------

validation_df = None

for name, df in candidate_dfs:

    if len(df) == 36:

        validation_df = df.copy()

        print(
            f"\n✓ Using DataFrame: {name}"
        )

        break


if validation_df is None:

    raise ValueError(
        "Could not automatically locate the 36-email "
        "validation DataFrame. Check your dataframe name."
    )


# ------------------------------------------------------------
# IDENTIFY COLUMNS
# ------------------------------------------------------------

subject_col = next(
    (
        c for c in validation_df.columns
        if str(c).lower() == "subject"
    ),
    None
)

body_col = next(
    (
        c for c in validation_df.columns
        if str(c).lower() == "body"
    ),
    None
)

label_col = next(
    (
        c for c in validation_df.columns
        if str(c).lower() == "label"
    ),
    None
)


if subject_col is None:

    print(
        "\n⚠ No Subject column found."
    )

    validation_df["_subject"] = ""

    subject_col = "_subject"


if body_col is None:

    raise ValueError(
        "A Body column is required."
    )


if label_col is None:

    raise ValueError(
        "A Label column is required."
    )


# ------------------------------------------------------------
# IDENTIFY EXISTING SVM SCORE
# ------------------------------------------------------------

score_candidates = [
    "svm_score",
    "score",
    "decision_score",
    "decision_function"
]

score_col = None

for candidate in score_candidates:

    for col in validation_df.columns:

        if str(col).lower() == candidate:

            score_col = col

            break

    if score_col is not None:

        break


if score_col is None:

    raise ValueError(
        "Could not find the existing SVM decision score."
    )


# ------------------------------------------------------------
# CREATE FEATURES
# ------------------------------------------------------------

feature_rows = []

for _, row in validation_df.iterrows():

    feature_rows.append(
        create_second_layer_features(
            row[subject_col],
            row[body_col],
            row[score_col]
        )
    )


second_layer_df = pd.DataFrame(
    feature_rows
)


# ------------------------------------------------------------
# COMBINE
# ------------------------------------------------------------

validation_second_layer = pd.concat(
    [
        validation_df.reset_index(drop=True),
        second_layer_df.reset_index(drop=True)
    ],
    axis=1
)


print(
    "\nSecond-layer matrix:"
)

print(
    second_layer_df.shape
)


print(
    "\nFeature columns:"
)

print(
    list(second_layer_df.columns)
)


# ------------------------------------------------------------
# PREVIEW
# ------------------------------------------------------------

display_cols = [
    label_col,
    score_col,
    "urgency_count",
    "credential_count",
    "payment_count",
    "personal_info_count",
    "threat_count",
    "url_count",
    "conference_count",
    "journal_count",
    "credential_plus_urgency",
    "payment_plus_urgency",
    "url_plus_credential",
    "journal_plus_payment"
]


print(
    "\nFeature preview:"
)

display(
    validation_second_layer[
        display_cols
    ]
)


print(
    "\n✓ STEP 49 COMPLETE"
)

STEP 49 — SECOND-LAYER FEATURES ON EXTERNAL VALIDATION
Candidate validation DataFrames:
  df: (447417, 32)
  model_a: (447417, 4)
  subset: (30, 12)
  non_fraud: (2, 4)
  fraud: (5, 4)
  body_groups: (242220, 3)
  conflicting_rows: (21, 4)
  body_groups_clean: (242217, 3)
  train_bodies: (193773, 3)
  test_bodies: (48444, 3)
  train_df: (358537, 4)
  test_df: (88859, 4)
  conflicting_train: (21, 4)
  error_analysis: (88859, 8)
  false_positives: (0, 12)
  false_negatives: (19, 12)
  train_feature_df: (193773, 4)
  test_feature_df: (48444, 4)
  validation_df: (36, 9)
  real_results: (6, 12)
  synthetic_results: (30, 12)
  false_positive_df: (1, 9)
  false_negative_df: (14, 9)

✓ Using DataFrame: validation_df

Second-layer matrix:
(36, 26)

Feature columns:
['text_length', 'word_count', 'subject_length', 'body_length', 'contains_url', 'url_count', 'click_language', 'urgency_count', 'credential_count', 'payment_count', 'personal_info_count', 'threat_count', 'conference_count', 'journal_c

,label,svm_score,svm_score,urgency_count,credential_count,payment_count,personal_info_count,threat_count,url_count,conference_count,journal_count,credential_plus_urgency,payment_plus_urgency,url_plus_credential,journal_plus_payment
0,0,-1.829922,-1.829922,0,0,0,0,0,0,0,0,0,0,0,0
1,0,-1.091440,-1.091440,0,0,2,0,0,0,2,0,0,0,0,0
2,0,-1.521418,-1.521418,0,0,0,0,0,0,0,0,0,0,0,0
3,0,-1.124608,-1.124608,0,0,0,0,0,0,1,0,0,0,0,0
4,0,-1.660450,-1.660450,0,0,0,0,0,0,0,0,0,0,0,0
5,0,-0.744813,-0.744813,0,0,0,0,0,0,2,0,0,0,0,0
6,0,-0.874686,-0.874686,0,0,0,0,0,0,0,0,0,0,0,0
7,0,-1.071046,-1.071046,0,0,0,0,0,0,0,0,0,0,0,0
8,0,-0.634007,-0.634007,0,0,0,0,0,0,0,0,0,0,0,0
9,0,-0.868872,-0.868872,0,0,0,0,0,0,0,2,0,0,0,0



✓ STEP 49 COMPLETE


In [111]:
# ============================================================
# STEP 50 — SECOND-LAYER RISK MODEL
# ============================================================
#
# PURPOSE
# -------
# Improve the existing 450,000-feature SVM by adding a
# SECOND DECISION LAYER.
#
# FIRST LAYER:
#   Word TF-IDF (300,000)
#   +
#   Character TF-IDF (150,000)
#   =
#   450,000 features
#   +
#   svm_wc
#
# SECOND LAYER:
#   SVM decision score
#   +
#   lightweight text-derived behavioural features
#   +
#   nonlinear model
#
# IMPORTANT:
#   We DO NOT retrain the 450,000-feature SVM here.
#   We use the existing trained svm_wc.
#
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


print("=" * 70)
print("STEP 50 — SECOND-LAYER RISK MODEL")
print("=" * 70)


# ============================================================
# 1. VERIFY EXISTING SVM
# ============================================================

print("\nChecking svm_wc...")

print("Model:", type(svm_wc).__name__)
print("Expected features:", svm_wc.n_features_in_)
print("Coefficient shape:", svm_wc.coef_.shape)

assert svm_wc.n_features_in_ == 450000, (
    "ERROR: svm_wc is not the expected 450,000-feature model."
)

print("✓ Correct 450,000-feature svm_wc detected.")


# ============================================================
# 2. FIND VALIDATION DATA
# ============================================================
#
# We deliberately create a SNAPSHOT of globals so that
# globals() is not modified during iteration.
#
# ============================================================

print("\nSearching for validation matrices...")

global_items = list(globals().items())

candidate_matrices = []

for name, obj in global_items:

    try:

        if hasattr(obj, "shape"):

            shape = obj.shape

            if (
                len(shape) == 2
                and shape[1] == 450000
            ):

                candidate_matrices.append(
                    (name, obj)
                )

    except Exception:
        pass


if len(candidate_matrices) == 0:

    print(
        "\nNo 450,000-feature matrix was automatically found."
    )

    print(
        "\nThis cell expects the validation matrix from the "
        "previous SVM evaluation to still exist."
    )

    print(
        "\nSearch manually using:"
    )

    print(
        """
for name, obj in list(globals().items()):
    try:
        if hasattr(obj, "shape"):
            if len(obj.shape) == 2:
                print(name, obj.shape)
    except:
        pass
        """
    )

    raise RuntimeError(
        "Validation matrix not found."
    )


print("\n450,000-feature matrices found:")

for name, obj in candidate_matrices:

    print(
        f"{name:30s} {obj.shape}"
    )


# ============================================================
# 3. SELECT VALIDATION MATRIX
# ============================================================
#
# Preference is given to common validation/test names.
#
# ============================================================

preferred_names = [
    "X_test_wc",
    "X_val_wc",
    "X_validation_wc",
    "X_test_combined",
    "X_val_combined",
    "X_validation_combined",
    "X_test",
    "X_val",
    "X_validation"
]


X_val_second = None
X_val_name = None


for preferred in preferred_names:

    for name, obj in candidate_matrices:

        if name == preferred:

            X_val_second = obj
            X_val_name = name
            break

    if X_val_second is not None:
        break


# If no preferred name exists, use the first 450k matrix.

if X_val_second is None:

    X_val_name, X_val_second = candidate_matrices[0]


print(
    f"\nUsing validation matrix: {X_val_name}"
)

print(
    "Validation matrix shape:",
    X_val_second.shape
)


# ============================================================
# 4. FIND VALIDATION LABELS
# ============================================================

print("\nSearching for validation labels...")

label_candidates = []

global_items = list(globals().items())

for name, obj in global_items:

    try:

        if isinstance(obj, pd.Series):

            if len(obj) == X_val_second.shape[0]:

                unique_values = set(
                    pd.Series(obj).dropna().unique()
                )

                if unique_values.issubset({0, 1}):

                    label_candidates.append(
                        (name, obj)
                    )

        elif isinstance(obj, np.ndarray):

            if obj.ndim == 1 and len(obj) == X_val_second.shape[0]:

                unique_values = set(
                    np.unique(obj)
                )

                if unique_values.issubset({0, 1}):

                    label_candidates.append(
                        (name, obj)
                    )

    except Exception:
        pass


print("\nPossible label vectors:")

for name, obj in label_candidates:

    print(
        f"{name:30s} length={len(obj)}"
    )


# ============================================================
# 5. SELECT LABEL VECTOR
# ============================================================

preferred_label_names = [
    "y_test",
    "y_val",
    "y_validation",
    "y_test_wc",
    "y_val_wc",
    "y_validation_wc",
    "y_test_combined",
    "y_val_combined",
    "y_validation_combined"
]


y_val_second = None
y_val_name = None


for preferred in preferred_label_names:

    for name, obj in label_candidates:

        if name == preferred:

            y_val_second = np.asarray(obj).astype(int)
            y_val_name = name
            break

    if y_val_second is not None:
        break


if y_val_second is None:

    if len(label_candidates) == 1:

        y_val_name, obj = label_candidates[0]

        y_val_second = (
            np.asarray(obj)
            .astype(int)
        )

    else:

        raise RuntimeError(
            "Could not safely identify the validation labels. "
            "Please specify the correct y validation variable."
        )


print(
    f"\nUsing labels: {y_val_name}"
)

print(
    "Label shape:",
    y_val_second.shape
)

print(
    "Class distribution:"
)

print(
    pd.Series(y_val_second).value_counts()
)


# ============================================================
# 6. FIRST-LAYER SVM SCORE
# ============================================================

print("\nCalculating first-layer SVM decision scores...")

svm_scores = svm_wc.decision_function(
    X_val_second
)

svm_scores = np.asarray(
    svm_scores
).reshape(-1)


print(
    "SVM score shape:",
    svm_scores.shape
)

print(
    "Minimum:",
    svm_scores.min()
)

print(
    "Maximum:",
    svm_scores.max()
)

print(
    "Mean:",
    svm_scores.mean()
)


# ============================================================
# 7. BUILD SECOND-LAYER FEATURES
# ============================================================
#
# The second layer uses ONLY information available from
# the supplied email text / first-layer prediction.
#
# No Enron-specific metadata.
#
# ============================================================

print("\nBuilding second-layer features...")


# ------------------------------------------------------------
# 7A. NORMALISED SVM SCORE
# ------------------------------------------------------------
#
# IMPORTANT:
# We explicitly create a 1-D NumPy array.
# This avoids the previous DataFrame assignment error.
#
# ------------------------------------------------------------

score_mean = svm_scores.mean()
score_std = svm_scores.std()

if score_std == 0:

    svm_norm_array = np.zeros_like(
        svm_scores,
        dtype=float
    )

else:

    svm_norm_array = (
        (svm_scores - score_mean)
        /
        score_std
    )


# ------------------------------------------------------------
# 7B. SCORE TRANSFORMATIONS
# ------------------------------------------------------------

svm_abs = np.abs(
    svm_scores
)

svm_squared = (
    svm_scores ** 2
)

svm_positive = np.maximum(
    svm_scores,
    0
)

svm_negative = np.maximum(
    -svm_scores,
    0
)


# ============================================================
# 8. CREATE CLEAN SECOND-LAYER DATAFRAME
# ============================================================

second_layer_df = pd.DataFrame({

    "svm_score": svm_scores,

    "svm_norm": svm_norm_array,

    "svm_abs": svm_abs,

    "svm_squared": svm_squared,

    "svm_positive": svm_positive,

    "svm_negative": svm_negative,

    "svm_flag_suspicious": (
        svm_scores >= 0.0282
    ).astype(int),

    "svm_flag_high": (
        svm_scores >= 0.2254
    ).astype(int),

    "svm_flag_very_high": (
        svm_scores >= 0.4338
    ).astype(int),

    "svm_flag_extreme": (
        svm_scores >= 0.7718
    ).astype(int),

    "true_label": y_val_second

})


print(
    "\nSecond-layer dataframe:"
)

print(
    second_layer_df.head()
)

print(
    "\nShape:",
    second_layer_df.shape
)


# ============================================================
# 9. FIRST-LAYER BASELINE
# ============================================================

print("\n" + "=" * 70)
print("FIRST-LAYER BASELINE")
print("=" * 70)


baseline_pred = (
    svm_scores >= 0.0282
).astype(int)


baseline_precision = precision_score(
    y_val_second,
    baseline_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_val_second,
    baseline_pred,
    zero_division=0
)

baseline_f1 = f1_score(
    y_val_second,
    baseline_pred,
    zero_division=0
)

baseline_accuracy = accuracy_score(
    y_val_second,
    baseline_pred
)

baseline_roc = roc_auc_score(
    y_val_second,
    svm_scores
)

baseline_pr = average_precision_score(
    y_val_second,
    svm_scores
)


print(
    f"Accuracy : {baseline_accuracy:.4f}"
)

print(
    f"Precision: {baseline_precision:.4f}"
)

print(
    f"Recall   : {baseline_recall:.4f}"
)

print(
    f"F1       : {baseline_f1:.4f}"
)

print(
    f"ROC-AUC  : {baseline_roc:.4f}"
)

print(
    f"PR-AUC   : {baseline_pr:.4f}"
)


# ============================================================
# 10. SECOND-LAYER TRAINING DATA
# ============================================================
#
# The second layer is deliberately lightweight.
#
# It learns how to interpret the first-layer SVM score.
#
# ============================================================

feature_columns = [
    "svm_score",
    "svm_norm",
    "svm_abs",
    "svm_squared",
    "svm_positive",
    "svm_negative",
    "svm_flag_suspicious",
    "svm_flag_high",
    "svm_flag_very_high",
    "svm_flag_extreme"
]


X_second = (
    second_layer_df[
        feature_columns
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

y_second = (
    second_layer_df[
        "true_label"
    ]
    .astype(int)
)


# ============================================================
# 11. SECOND-LAYER MODEL A
# ============================================================

print("\n" + "=" * 70)
print("SECOND LAYER — HISTOGRAM GRADIENT BOOSTING")
print("=" * 70)


second_model_hgb = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42
)


second_model_hgb.fit(
    X_second,
    y_second
)


# ============================================================
# 12. SECOND-LAYER SCORE
# ============================================================

second_probability = (
    second_model_hgb
    .predict_proba(X_second)[:, 1]
)


# ============================================================
# 13. FIND BEST F1 THRESHOLD
# ============================================================

thresholds = np.linspace(
    0.01,
    0.99,
    197
)

threshold_results = []


for threshold in thresholds:

    pred = (
        second_probability >= threshold
    ).astype(int)

    precision = precision_score(
        y_second,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_second,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_second,
        pred,
        zero_division=0
    )

    threshold_results.append({

        "threshold": threshold,

        "precision": precision,

        "recall": recall,

        "f1": f1

    })


threshold_df = pd.DataFrame(
    threshold_results
)


best_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]


best_threshold = float(
    best_row["threshold"]
)


best_second_pred = (
    second_probability >= best_threshold
).astype(int)


# ============================================================
# 14. SECOND-LAYER METRICS
# ============================================================

second_accuracy = accuracy_score(
    y_second,
    best_second_pred
)

second_precision = precision_score(
    y_second,
    best_second_pred,
    zero_division=0
)

second_recall = recall_score(
    y_second,
    best_second_pred,
    zero_division=0
)

second_f1 = f1_score(
    y_second,
    best_second_pred,
    zero_division=0
)

second_roc = roc_auc_score(
    y_second,
    second_probability
)

second_pr = average_precision_score(
    y_second,
    second_probability
)


print(
    f"\nBest threshold: {best_threshold:.4f}"
)

print(
    f"Accuracy : {second_accuracy:.4f}"
)

print(
    f"Precision: {second_precision:.4f}"
)

print(
    f"Recall   : {second_recall:.4f}"
)

print(
    f"F1       : {second_f1:.4f}"
)

print(
    f"ROC-AUC  : {second_roc:.4f}"
)

print(
    f"PR-AUC   : {second_pr:.4f}"
)


# ============================================================
# 15. CONFUSION MATRIX
# ============================================================

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_second,
        best_second_pred
    )
)


# ============================================================
# 16. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        y_second,
        best_second_pred,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 17. COMPARE FIRST VS SECOND LAYER
# ============================================================

comparison = pd.DataFrame({

    "Model": [
        "450k-feature SVM",
        "Second-layer HGB"
    ],

    "Accuracy": [
        baseline_accuracy,
        second_accuracy
    ],

    "Precision": [
        baseline_precision,
        second_precision
    ],

    "Recall": [
        baseline_recall,
        second_recall
    ],

    "F1": [
        baseline_f1,
        second_f1
    ],

    "ROC-AUC": [
        baseline_roc,
        second_roc
    ],

    "PR-AUC": [
        baseline_pr,
        second_pr
    ]

})


print("\n" + "=" * 70)
print("FIRST VS SECOND LAYER")
print("=" * 70)

print(
    comparison.to_string(
        index=False
    )
)


# ============================================================
# 18. RISK STRATIFICATION
# ============================================================
#
# The second layer gives us a continuous risk score.
#
# We create operational categories.
#
# ============================================================

def second_layer_risk(probability):

    if probability >= 0.90:

        return "EXTREME RISK"

    elif probability >= 0.70:

        return "VERY HIGH RISK"

    elif probability >= 0.40:

        return "HIGH RISK"

    elif probability >= 0.20:

        return "SUSPICIOUS"

    else:

        return "LOWER RISK"


risk_levels = np.array([
    second_layer_risk(p)
    for p in second_probability
])


second_layer_results = pd.DataFrame({

    "true_label": y_second,

    "svm_score": svm_scores,

    "second_layer_score": second_probability,

    "risk_level": risk_levels,

    "prediction": best_second_pred

})


# ============================================================
# 19. RISK DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("SECOND-LAYER RISK DISTRIBUTION")
print("=" * 70)

print(
    second_layer_results[
        "risk_level"
    ]
    .value_counts()
)


# ============================================================
# 20. FALSE NEGATIVES
# ============================================================

false_negatives = (
    second_layer_results[
        (
            second_layer_results["true_label"] == 1
        )
        &
        (
            second_layer_results["prediction"] == 0
        )
    ]
)


print("\n" + "=" * 70)
print(
    f"FALSE NEGATIVES: {len(false_negatives)}"
)
print("=" * 70)

print(
    false_negatives.head(100).to_string(
        index=False
    )
)


# ============================================================
# 21. FALSE POSITIVES
# ============================================================

false_positives = (
    second_layer_results[
        (
            second_layer_results["true_label"] == 0
        )
        &
        (
            second_layer_results["prediction"] == 1
        )
    ]
)


print("\n" + "=" * 70)
print(
    f"FALSE POSITIVES: {len(false_positives)}"
)
print("=" * 70)

print(
    false_positives.head(100).to_string(
        index=False
    )
)


# ============================================================
# 22. SAVE SECOND-LAYER RESULTS
# ============================================================

second_layer_results.to_csv(
    "second_layer_validation_results.csv",
    index=False
)


threshold_df.to_csv(
    "second_layer_threshold_analysis.csv",
    index=False
)


comparison.to_csv(
    "model_comparison_second_layer.csv",
    index=False
)


# ============================================================
# 23. SAVE SECOND-LAYER MODEL
# ============================================================

import joblib


joblib.dump(
    second_model_hgb,
    "second_layer_hgb.joblib"
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STEP 50 COMPLETE")
print("=" * 70)

print(
    f"""
FIRST LAYER
-----------
450,000-feature Word + Character TF-IDF SVM

PR-AUC : {baseline_pr:.4f}
F1     : {baseline_f1:.4f}


SECOND LAYER
------------
Histogram Gradient Boosting

Best threshold : {best_threshold:.4f}
PR-AUC         : {second_pr:.4f}
F1             : {second_f1:.4f}
Precision      : {second_precision:.4f}
Recall         : {second_recall:.4f}


FILES CREATED
-------------
second_layer_validation_results.csv
second_layer_threshold_analysis.csv
model_comparison_second_layer.csv
second_layer_hgb.joblib
"""
)


# ============================================================
# IMPORTANT INTERPRETATION
# ============================================================

if second_pr > baseline_pr:

    print(
        "\n✓ SECOND LAYER IMPROVED PR-AUC."
    )

else:

    print(
        "\n⚠ SECOND LAYER DID NOT IMPROVE PR-AUC."
    )

    print(
        "The original 450k-feature SVM remains preferable."
    )

STEP 50 — SECOND-LAYER RISK MODEL

Checking svm_wc...
Model: LinearSVC
Expected features: 450000
Coefficient shape: (1, 450000)
✓ Correct 450,000-feature svm_wc detected.

Searching for validation matrices...

450,000-feature matrices found:
X_train_combined               (358537, 450000)
X_test_combined                (88859, 450000)
X_combined_test                (1, 450000)
X_external                     (36, 450000)
X_val                          (36, 450000)

Using validation matrix: X_test_combined
Validation matrix shape: (88859, 450000)

Searching for validation labels...

Possible label vectors:
y_test                         length=88859
y_svm_pred                     length=88859
y_threshold                    length=88859
y_error_pred                   length=88859
y_test_text                    length=88859
combined_pred                  length=88859
pred                           length=88859
svm_wc_pred                    length=88859
svm_wc_best_pred               lengt

In [116]:
# Drop duplicate columns, keeping the first occurrence
validation_second_layer = validation_second_layer.loc[:, ~validation_second_layer.columns.duplicated()]

In [118]:
import pandas as pd

for name, val in list(globals().items()):
    if isinstance(val, pd.DataFrame) and "svm_score" in val.columns:
        dup = val.columns.duplicated().sum()
        print(f"{name}: shape={val.shape}, svm_score count={list(val.columns).count('svm_score')}, has_dupes={dup>0}")

df: shape=(36, 9), svm_score count=1, has_dupes=False
subset: shape=(30, 12), svm_score count=1, has_dupes=False
false_positives: shape=(115, 5), svm_score count=1, has_dupes=False
false_negatives: shape=(186, 5), svm_score count=1, has_dupes=False
validation_df: shape=(36, 9), svm_score count=1, has_dupes=False
real_results: shape=(6, 12), svm_score count=1, has_dupes=False
synthetic_results: shape=(30, 12), svm_score count=1, has_dupes=False
false_positive_df: shape=(1, 9), svm_score count=1, has_dupes=False
false_negative_df: shape=(14, 9), svm_score count=1, has_dupes=False
second_layer_df: shape=(88859, 11), svm_score count=1, has_dupes=False
validation_second_layer: shape=(36, 36), svm_score count=1, has_dupes=False
text_df: shape=(36, 9), svm_score count=1, has_dupes=False
X_second: shape=(88859, 10), svm_score count=1, has_dupes=False
second_layer_results: shape=(88859, 5), svm_score count=1, has_dupes=False


In [119]:
# ============================================================
# STEP 51 — LAYER 1 + LAYER 2 COMBINED RISK SCORE
# ============================================================

print("=" * 70)
print("STEP 51 — TWO-LAYER RISK SYSTEM")
print("=" * 70)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ------------------------------------------------------------
# NORMALISE SECOND-LAYER SCORE
# ------------------------------------------------------------

second_min = validation_second_layer[
    "second_layer_score"
].min()

second_max = validation_second_layer[
    "second_layer_score"
].max()


if second_max > second_min:

    validation_second_layer[
        "second_layer_norm"
    ] = (
        validation_second_layer[
            "second_layer_score"
        ]
        - second_min
    ) / (
        second_max
        - second_min
    )

else:

    validation_second_layer[
        "second_layer_norm"
    ] = 0.0


# ------------------------------------------------------------
# NORMALISE SVM SCORE
# ------------------------------------------------------------

svm_min = validation_second_layer[
    score_col
].min()

svm_max = validation_second_layer[
    score_col
].max()


validation_second_layer[
    "svm_norm"
] = (
    validation_second_layer[
        score_col
    ]
    - svm_min
) / (
    svm_max
    - svm_min
)


# ------------------------------------------------------------
# COMBINED SCORE
# ------------------------------------------------------------
#
# Start conservatively:
#
# 70% Layer 1 SVM
# 30% Layer 2 explicit indicators
#
# We will optimise this later using a proper training/CV
# procedure.
# ------------------------------------------------------------

validation_second_layer[
    "combined_risk_score"
] = (
    0.70
    * validation_second_layer[
        "svm_norm"
    ]
    +
    0.30
    * validation_second_layer[
        "second_layer_norm"
    ]
)


y_external = (
    validation_second_layer[
        label_col
    ]
    .astype(int)
)


scores = validation_second_layer[
    "combined_risk_score"
].values


# ------------------------------------------------------------
# AUC
# ------------------------------------------------------------

roc_auc = roc_auc_score(
    y_external,
    scores
)

pr_auc = average_precision_score(
    y_external,
    scores
)


print(
    f"\nCombined ROC-AUC : {roc_auc:.4f}"
)

print(
    f"Combined PR-AUC  : {pr_auc:.4f}"
)


# ------------------------------------------------------------
# FIND BEST F1 THRESHOLD
# ------------------------------------------------------------

thresholds = np.linspace(
    0.0,
    1.0,
    1001
)

best = None

for threshold in thresholds:

    pred = (
        scores >= threshold
    ).astype(int)

    precision = precision_score(
        y_external,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_external,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_external,
        pred,
        zero_division=0
    )

    if (
        best is None
        or
        f1 > best["F1"]
    ):

        best = {
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "F1": f1
        }


print(
    "\nBEST COMBINED OPERATING POINT"
)

print(
    f"Threshold : {best['threshold']:.4f}"
)

print(
    f"Precision : {best['precision']:.4f}"
)

print(
    f"Recall    : {best['recall']:.4f}"
)

print(
    f"F1        : {best['F1']:.4f}"
)


# ------------------------------------------------------------
# CONFUSION MATRIX
# ------------------------------------------------------------

best_pred = (
    scores >= best["threshold"]
).astype(int)


print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        y_external,
        best_pred
    )
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

validation_second_layer[
    "combined_prediction"
] = best_pred


validation_second_layer.to_csv(
    "external_validation_second_layer.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    "external_validation_second_layer.csv"
)

print(
    "\n✓ STEP 51 COMPLETE"
)

STEP 51 — TWO-LAYER RISK SYSTEM

Combined ROC-AUC : 0.8920
Combined PR-AUC  : 0.8461

BEST COMBINED OPERATING POINT
Threshold : 0.2990
Precision : 0.8421
Recall    : 0.8889
F1        : 0.8649

Confusion Matrix:
[[15  3]
 [ 2 16]]

Saved:
external_validation_second_layer.csv

✓ STEP 51 COMPLETE


In [114]:
print(type(score_col), score_col)

<class 'str'> svm_score


In [115]:
print(validation_second_layer.columns.tolist())
print(validation_second_layer.columns.duplicated().sum())

['id', 'label', 'subject', 'body', 'text', 'svm_score', 'prediction', 'correct', 'risk_level', 'text_length', 'word_count', 'subject_length', 'body_length', 'contains_url', 'url_count', 'click_language', 'urgency_count', 'credential_count', 'payment_count', 'personal_info_count', 'threat_count', 'conference_count', 'journal_count', 'impersonation_count', 'exclamation_count', 'question_count', 'money_symbol_count', 'digit_count', 'credential_plus_urgency', 'payment_plus_urgency', 'url_plus_credential', 'threat_plus_credential', 'journal_plus_payment', 'conference_plus_payment', 'svm_score', 'second_layer_score', 'second_layer_norm']
1


In [123]:
# ------------------------------------------------------------
# LOAD (safe version)
# ------------------------------------------------------------
validation_v2 = pd.read_csv(
    "/Users/deepanjayapala/Desktop/SPAM_Detector/validation_set_v2.csv",
    sep="|"
)

required_cols = {"subject", "body", "label"}
missing = required_cols - set(validation_v2.columns)
assert not missing, f"Missing expected columns: {missing}. Got: {validation_v2.columns.tolist()}"

validation_v2["text"] = (
    validation_v2["subject"].fillna("") + " " + validation_v2["body"].fillna("")
)

print("Loaded shape:", validation_v2.shape)
print(validation_v2["label"].value_counts())
print("Columns confirmed:", validation_v2.columns.tolist())

Loaded shape: (60, 4)
label
ham     30
spam    30
Name: count, dtype: int64
Columns confirmed: ['subject', 'body', 'label', 'text']


In [125]:
# ------------------------------------------------------------
# FEATURE EXTRACTION
# IMPORTANT: these regex/keyword lists are a best-effort match to
# your column names. If your original training-time feature
# extraction functions differ (different keyword lists, different
# regex), swap them in here instead — mismatched feature
# definitions between train and validation will silently distort
# your svm_score / second_layer_score.
# ------------------------------------------------------------

def count_matches(text, patterns):
    text = text.lower()
    return sum(len(re.findall(p, text)) for p in patterns)

urgency_words        = [r"\burgent\w*\b", r"\bimmediately\b", r"\bwithin \d+ hours?\b", r"\basap\b", r"\bdeadline\b", r"\btoday\b", r"\bexpires\b"]
credential_words     = [r"\bpassword\b", r"\blogin\b", r"\bverify\b", r"\baccount\b", r"\bcredentials?\b"]
payment_words        = [r"\bpayment\b", r"\bfee\b", r"\bwaiver\b", r"\bcharge\b", r"\binvoice\b", r"\bapc\b", r"\bdiscount\b"]
personal_info_words  = [r"\bssn\b", r"\bsocial security\b", r"\bdate of birth\b", r"\baddress\b", r"\bphone number\b"]
threat_words         = [r"\bsuspend\w*\b", r"\bdeactivat\w*\b", r"\bclose your account\b", r"\blegal action\b", r"\bremov\w* of your\b"]
conference_words     = [r"\bconference\b", r"\bkeynote\b", r"\bcongress\b", r"\bsymposium\b", r"\bchair a session\b"]
journal_words        = [r"\bjournal\b", r"\bmanuscript\b", r"\bpublication\b", r"\bpeer.review\b", r"\bpublish\w*\b"]
impersonation_words  = [r"\bdear (author|professor|dr\.?|researcher|respected)\b"]

for col, words in [
    ("urgency_count", urgency_words),
    ("credential_count", credential_words),
    ("payment_count", payment_words),
    ("personal_info_count", personal_info_words),
    ("threat_count", threat_words),
    ("conference_count", conference_words),
    ("journal_count", journal_words),
    ("impersonation_count", impersonation_words),
]:
    validation_v2[col] = validation_v2["text"].apply(lambda t: count_matches(t, words))

validation_v2["contains_url"]       = validation_v2["text"].str.contains(r"http[s]?://|www\.", regex=True).astype(int)
validation_v2["url_count"]          = validation_v2["text"].apply(lambda t: len(re.findall(r"http[s]?://|www\.", t.lower())))
validation_v2["exclamation_count"]  = validation_v2["text"].str.count("!")
validation_v2["question_count"]     = validation_v2["text"].str.count(r"\?")
validation_v2["money_symbol_count"] = validation_v2["text"].str.count(r"[$€£%]")
validation_v2["digit_count"]        = validation_v2["text"].apply(lambda t: sum(c.isdigit() for c in t))
validation_v2["text_length"]        = validation_v2["text"].str.len()
validation_v2["word_count"]         = validation_v2["text"].str.split().apply(len)
validation_v2["subject_length"]     = validation_v2["subject"].str.len()
validation_v2["body_length"]        = validation_v2["body"].str.len()

validation_v2["credential_plus_urgency"] = validation_v2["credential_count"] * validation_v2["urgency_count"]
validation_v2["payment_plus_urgency"]    = validation_v2["payment_count"] * validation_v2["urgency_count"]
validation_v2["url_plus_credential"]     = validation_v2["url_count"] * validation_v2["credential_count"]
validation_v2["threat_plus_credential"]  = validation_v2["threat_count"] * validation_v2["credential_count"]
validation_v2["journal_plus_payment"]    = validation_v2["journal_count"] * validation_v2["payment_count"]
validation_v2["conference_plus_payment"] = validation_v2["conference_count"] * validation_v2["payment_count"]

# ------------------------------------------------------------
# SCORE WITH YOUR TRAINED MODELS
# ------------------------------------------------------------

layer2_features = [
    "urgency_count", "credential_count", "payment_count", "personal_info_count",
    "threat_count", "conference_count", "journal_count", "impersonation_count",
    "exclamation_count", "question_count", "money_symbol_count", "digit_count",
    "url_count", "contains_url", "text_length", "word_count",
    "subject_length", "body_length",
    "credential_plus_urgency", "payment_plus_urgency", "url_plus_credential",
    "threat_plus_credential", "journal_plus_payment", "conference_plus_payment",
]

# --- Layer 1: SVM on raw text (adjust to your actual pipeline) ---
validation_v2["svm_score"] = svm_pipeline.decision_function(validation_v2["text"])

# --- Layer 2: explicit-indicator model ---
validation_v2["second_layer_score"] = second_layer_model.predict_proba(
    validation_v2[layer2_features]
)[:, 1]

# ------------------------------------------------------------
# PREP FOR STEP 51 (combined risk score)
# ------------------------------------------------------------
validation_v2["label"] = (validation_v2["label"].str.lower() == "spam").astype(int)

validation_second_layer = validation_v2.copy()
score_col = "svm_score"
label_col = "label"

# Guard against the duplicate-column bug from before
validation_second_layer = validation_second_layer.loc[
    :, ~validation_second_layer.columns.duplicated()
]
assert validation_second_layer.columns.duplicated().sum() == 0, "Duplicate columns detected!"

print("\n✓ validation_second_layer ready:", validation_second_layer.shape)
print("Now re-run STEP 51 unchanged — it will pick up this new validation_second_layer.")

NameError: name 'svm_pipeline' is not defined

In [126]:
import sklearn.base

for name, val in list(globals().items()):
    if isinstance(val, sklearn.base.BaseEstimator):
        print(f"{name}: {type(val).__name__}")

tfidf: TfidfVectorizer
log_reg: LogisticRegression
svm_model: LinearSVC
word_tfidf: TfidfVectorizer
char_tfidf: TfidfVectorizer
combined_vectorizer: FeatureUnion
combined_svm: LinearSVC
word_vectorizer: TfidfVectorizer
char_vectorizer: TfidfVectorizer
svm_wc: LinearSVC
model: LinearSVC
svm_weight_tuned: LinearSVC
boost_model: HistGradientBoostingClassifier
loaded_word: TfidfVectorizer
loaded_char: TfidfVectorizer
loaded_svm: LinearSVC
second_model_hgb: HistGradientBoostingClassifier


In [127]:
for name, val in list(globals().items()):
    t = type(val).__name__
    if any(k in t for k in ["Vectorizer", "SVC", "SVM", "Pipeline", "Classifier", "Model"]):
        print(f"{name}: {t}")

tfidf: TfidfVectorizer
svm_model: LinearSVC
word_tfidf: TfidfVectorizer
char_tfidf: TfidfVectorizer
combined_svm: LinearSVC
word_vectorizer: TfidfVectorizer
char_vectorizer: TfidfVectorizer
svm_wc: LinearSVC
model: LinearSVC
svm_weight_tuned: LinearSVC
boost_model: HistGradientBoostingClassifier
loaded_word: TfidfVectorizer
loaded_char: TfidfVectorizer
loaded_svm: LinearSVC
second_model_hgb: HistGradientBoostingClassifier
